# Download data

In [ ]:
!pip -q install huggingface_hub requests pyarrow
import os, io, re, csv, glob, time, zipfile, hashlib, requests
import numpy as np, pandas as pd
from urllib.parse import quote
from huggingface_hub import HfApi, login, hf_hub_download, snapshot_download
from kaggle_secrets import UserSecretsClient

api = HfApi()
REPO = "B4BUK4/MPVI_project"
TS = "Date and time"
api.create_repo(REPO, repo_type="dataset", private=True, exist_ok=True)

RECORDS = {"penmanshiel": 16807304, "kelmarsh": 16807551, "smarteole": 7342466}





snapshot_download(REPO, repo_type="dataset", allow_patterns=["parquet/*"], local_dir="/kaggle/working")

for d in sorted(glob.glob("/kaggle/working/parquet/*")) + sorted(glob.glob("/kaggle/working/parquet/status/*")):
    print(d, len(glob.glob(d + "/*.parquet")), "files")

In [ ]:
def scan(zip_path, member, tail=3):
    n = nbytes = longest = short = 0
    first, last = None, []
    with zipfile.ZipFile(zip_path) as z, z.open(member) as f:
        for line in f:
            n += 1
            L = len(line); nbytes += L; longest = max(longest, L)
            if n == 11: first = line                      # first data row
            if n > 11 and len(line.strip(b", \r\n")) < 20: short += 1
            last.append(line); last = last[-tail:]
    print(f"{n:,} lines | {nbytes/1e9:.2f} GB | avg {nbytes/n:,.0f} B/line | longest {longest:,} B | near-empty lines: {short:,}")
    print("first data row:", len(first), "B ->", first[:160])
    for l in last: print("tail:", len(l), "B ->", l[:160])

scan("/kaggle/working/hf/kelmarsh/Kelmarsh_SCADA_2023_5961.zip",
     "Turbine_Data_Kelmarsh_1_2023-01-01_-_2024-01-01_228.csv")

p = hf_hub_download(REPO, "penmanshiel/Penmanshiel_WT_static.csv", repo_type="dataset", local_dir="/kaggle/working/hf")
print(pd.read_csv(p).to_string())

z = zipfile.ZipFile("/kaggle/working/hf/penmanshiel/Penmanshiel_SCADA_2016_WT01-10_3107.zip")
st = next(n for n in z.namelist() if n.startswith("Status_"))
print(z.read(st)[:1500].decode("utf-8", "replace"))

In [ ]:
import re, collections
ts_re = re.compile(rb"^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2},")

def scan2(zip_path, member=None, peek_at=(60_000, 1_000_000, 2_000_000)):
    lens, samples_other, samples_at = collections.Counter(), [], {}
    seen, n_ts, n_other, dup, out_of_order, prev = set(), 0, 0, 0, 0, None
    with zipfile.ZipFile(zip_path) as z:
        member = member or next(n for n in z.namelist() if n.startswith("Turbine_Data"))
        print(member)
        with z.open(member) as f:
            for i, line in enumerate(f, 1):
                lens[len(line) // 500 * 500] += 1
                if i in peek_at:
                    samples_at[i] = (len(line), line[:160])
                if ts_re.match(line):
                    n_ts += 1
                    ts = line[:19]
                    if ts in seen: dup += 1
                    else: seen.add(ts)
                    if prev and ts < prev: out_of_order += 1
                    prev = ts
                else:
                    n_other += 1
                    if len(samples_other) < 5:
                        samples_other.append((i, len(line), line[:200]))
    print(f"timestamp rows: {n_ts:,} | unique timestamps: {len(seen):,} | repeated: {dup:,} | out of order: {out_of_order:,} | other lines: {n_other:,}")
    print("line-length buckets (bytes: count), top 8:", lens.most_common(8))
    for s in samples_other: print("other line:", s)
    for k, v in samples_at.items(): print("line", k, v)

scan2("/kaggle/working/hf/kelmarsh/Kelmarsh_SCADA_2023_5961.zip")
p22 = hf_hub_download(REPO, "kelmarsh/Kelmarsh_SCADA_2022_4457.zip", repo_type="dataset", local_dir="/kaggle/working/hf")
scan2(p22)

In [ ]:
import io, csv, re, os, time
TS = "Date and time"
WANT = [TS,
 "Wind speed (m/s)", "Wind speed, Standard deviation (m/s)", "Wind speed, Minimum (m/s)", "Wind speed, Maximum (m/s)",
 "Wind speed Sensor 1 (m/s)", "Wind speed Sensor 2 (m/s)", "Density adjusted wind speed (m/s)",
 "Wind direction (°)", "Wind direction, Standard deviation (°)", "Nacelle position (°)", "Vane position 1+2 (°)", "Yaw bearing angle (°)",
 "Nacelle ambient temperature (°C)",
 "Power (kW)", "Power, Standard deviation (kW)", "Power, Minimum (kW)", "Power, Maximum (kW)",
 "Turbine Power setpoint (kW)",
 "Blade angle (pitch position) A (°)", "Blade angle (pitch position) B (°)", "Blade angle (pitch position) C (°)",
 "Rotor speed (RPM)", "Generator RPM (RPM)",
 "Lost Production to Curtailment (Total) (kWh)", "Lost Production to Downtime (kWh)"]   # last two: filters only

def convert_member(z, member, out_path, chunk=200_000):
    with z.open(member) as raw:
        f = io.TextIOWrapper(raw, encoding="utf-8", newline="")
        line = f.readline()
        while line and not line.startswith("# Date and time"):
            line = f.readline()
        names = [c.strip() for c in next(csv.reader([line.lstrip("# ")]))]
        pos, cols = [], []
        for i, c in enumerate(names):
            if c in WANT and c not in cols:
                pos.append(i); cols.append(c)
        missing = [c for c in WANT if c not in cols]
        parts = []
        for ch in pd.read_csv(f, header=None, usecols=pos, names=cols, index_col=False,
                              chunksize=chunk, na_values=["NaN"]):
            parts.append(ch.dropna(subset=cols[1:], how="all"))
    df = pd.concat(parts, ignore_index=True)
    df[TS] = pd.to_datetime(df[TS])
    num = [c for c in cols if c != TS]
    df[num] = df[num].apply(pd.to_numeric, errors="coerce").astype("float32")
    df["_nn"] = df[num].notna().sum(axis=1)
    df = (df.sort_values([TS, "_nn"]).drop_duplicates(TS, keep="last")
            .drop(columns="_nn").reset_index(drop=True))
    df.to_parquet(out_path, index=False)
    return df, missing

pat = re.compile(r"Turbine_Data_([A-Za-z]+)_(\d+)_(\d{4}-\d{2}-\d{2})_-_(\d{4}-\d{2}-\d{2})")

def convert_zip(zip_path, limit=None):
    rows = []
    with zipfile.ZipFile(zip_path) as z:
        members = [n for n in z.namelist() if os.path.basename(n).startswith("Turbine_Data")]
        for m in members[:limit]:
            farm, t, s, e = pat.search(m).groups()
            out_dir = f"/kaggle/working/parquet/{farm.lower()}"
            os.makedirs(out_dir, exist_ok=True)
            out = f"{out_dir}/{farm.lower()}_{int(t):02d}_{s}_{e}.parquet"
            if os.path.exists(out):
                continue
            t0 = time.time()
            df, missing = convert_member(z, m, out)
            span = int((df[TS].iloc[-1] - df[TS].iloc[0]) / pd.Timedelta("10min")) + 1
            rows.append(dict(file=os.path.basename(out), rows=len(df), fill_pct=round(len(df) / span * 100, 1),
                             first=df[TS].iloc[0], last=df[TS].iloc[-1],
                             power_nan_pct=round(df["Power (kW)"].isna().mean() * 100, 1),
                             missing_cols=len(missing), secs=round(time.time() - t0)))
            if len(rows) == 1 and missing:
                print("missing from this file:", missing)
    return pd.DataFrame(rows)

# test: one clean file and one padded file
print(convert_zip("/kaggle/working/hf/kelmarsh/Kelmarsh_SCADA_2022_4457.zip", limit=1).to_string())
print(convert_zip("/kaggle/working/hf/kelmarsh/Kelmarsh_SCADA_2023_5961.zip", limit=1).to_string())

In [ ]:
import glob
pat_s = re.compile(r"Status_([A-Za-z]+)_(\d+)_(\d{4}-\d{2}-\d{2})_-_(\d{4}-\d{2}-\d{2})")

def convert_status(z, member, out_path):
    lines = z.read(member).decode("utf-8", "replace").splitlines()
    h = next(i for i, l in enumerate(lines) if l.startswith("Timestamp start"))
    df = pd.read_csv(io.StringIO("\n".join(lines[h:])))
    for c in ["Timestamp start", "Timestamp end"]:
        df[c] = pd.to_datetime(df[c], errors="coerce")
    txt = ["Duration", "Status", "Code", "Message", "Comment", "Service contract category", "IEC category"]
    df = df.astype({c: "string" for c in txt if c in df}).drop_duplicates().reset_index(drop=True)
    df.to_parquet(out_path, index=False)

files = sorted(e.path for e in api.list_repo_tree(REPO, repo_type="dataset", recursive=True)
               if hasattr(e, "size") and e.path.endswith(".zip") and "SCADA" in e.path)
failed = []
for path in files:
    t0, local = time.time(), f"/kaggle/working/hf/{path}"
    try:
        p = hf_hub_download(REPO, path, repo_type="dataset", local_dir="/kaggle/working/hf")
        with zipfile.ZipFile(p) as z:
            for m in z.namelist():
                b = os.path.basename(m)
                if b.startswith("Status_"):
                    farm, t, s, e = pat_s.search(b).groups()
                    d = f"/kaggle/working/parquet/status/{farm.lower()}"; os.makedirs(d, exist_ok=True)
                    out = f"{d}/{farm.lower()}_{int(t):02d}_{s}_{e}.parquet"
                    if not os.path.exists(out): convert_status(z, m, out)
        r = convert_zip(p)
        print(f"{path}: {len(r)} files converted, {time.time()-t0:.0f}s")
    except Exception as ex:
        failed.append((path, repr(ex))); print("FAILED", path, repr(ex))
    finally:
        if os.path.exists(local): os.remove(local)
print("failed:", failed)

# coverage summary, built from the parquet files themselves
recs = []
for p in sorted(glob.glob("/kaggle/working/parquet/*/*.parquet")):
    d = pd.read_parquet(p, columns=[TS, "Power (kW)"])
    farm, t = os.path.basename(p).split("_")[:2]
    span = int((d[TS].iloc[-1] - d[TS].iloc[0]) / pd.Timedelta("10min")) + 1
    recs.append(dict(farm=farm, turbine=int(t), start=d[TS].iloc[0], end=d[TS].iloc[-1],
                     rows=len(d), fill=len(d) / span, power_nan=d["Power (kW)"].isna().mean()))
S = pd.DataFrame(recs); S["year"] = S.start.dt.year
print(S.groupby(["farm", "year"]).agg(files=("turbine", "count"), turbines=("turbine", "nunique"),
      rows=("rows", "sum"), min_fill=("fill", "min"), power_nan=("power_nan", "mean")).round(3).to_string())
!du -sh /kaggle/working/parquet

In [ ]:
import numpy as np
zp = "/kaggle/working/hf/smarteole/SMARTEOLE-WFC-open-dataset.zip"
base = "SMARTEOLE-WFC-open-dataset/SMARTEOLE_WakeSteering_"
os.makedirs("/kaggle/working/parquet/smarteole", exist_ok=True)
with zipfile.ZipFile(zp) as z:
    scada = pd.read_csv(z.open(base + "SCADA_1minData.csv"), parse_dates=["time"])
    ctl = pd.read_csv(z.open(base + "ControlLog_1minData.csv"), parse_dates=["time"])
    wc = pd.read_csv(z.open(base + "WindCube_1minData.csv"), parse_dates=["time"])
scada = scada[["time"] + [c for c in scada.columns if c.endswith(("_avg", "_std")) or "derate" in c]].copy()
ctl = ctl[["time"] + [c for c in ctl.columns if c.endswith("_avg")]].copy()
for name, d in [("scada", scada), ("controllog", ctl), ("windcube", wc)]:
    num = d.columns.drop("time")
    d[num] = d[num].astype("float32")
    d.to_parquet(f"/kaggle/working/parquet/smarteole/smarteole_{name}.parquet", index=False)
    print(name, d.shape, d["time"].min(), d["time"].max())

# what did the yaw-offset experiment actually do?
a = ctl["control_log_offset_active_avg"] > 0.5
print(f"\noffset active: {a.sum():,} of {len(ctl):,} min ({a.mean():.1%}) | switches: {int((a.astype(int).diff().abs() == 1).sum())}")
print(ctl.loc[a, "control_log_offset_avg"].round().value_counts().head(8))
wd, ws = "control_log_wind_direction_filt_60s_deg_avg", "control_log_wind_speed_filt_60s_avg"
print(ctl.groupby(a)[[wd, ws]].describe(percentiles=[.1, .5, .9]).T.round(1))
dr = [c for c in scada.columns if "derate" in c]
print(dr, scada[dr].mean().round(3).to_dict())

# status files: any unparsed timestamps?
st = pd.concat(pd.read_parquet(p) for p in glob.glob("/kaggle/working/parquet/status/*/*.parquet"))
print("status rows:", len(st), "| NaT start:", st["Timestamp start"].isna().sum(), "| NaT end:", st["Timestamp end"].isna().sum())

api.upload_folder(folder_path="/kaggle/working/parquet", path_in_repo="parquet", repo_id=REPO, repo_type="dataset")

In [ ]:
def load(farm):
    out = []
    for f in sorted(glob.glob(f"/kaggle/working/parquet/{farm}/*.parquet")):
        d = pd.read_parquet(f)
        out.append(d[[c for c in [TS, "Wind speed (m/s)", "Power (kW)", "Turbine Power setpoint (kW)"] if c in d]])
    return pd.concat(out, ignore_index=True)

pen, kel = load("penmanshiel"), load("kelmarsh")
with zipfile.ZipFile(zp) as z:
    gpc = pd.read_csv(z.open(base + "GuaranteedPowerCurve_staticData.csv"))
print(gpc.head(3))

def curve(df):
    d = df[df["Wind speed (m/s)"].between(0, 25) & df["Power (kW)"].notna()]
    g = d.groupby(pd.cut(d["Wind speed (m/s)"], np.arange(-0.5, 25.5, 1)), observed=True)["Power (kW)"]
    t = pd.DataFrame({"n": g.size(), "p10": g.quantile(.1), "median": g.median(), "p90": g.quantile(.9)})
    t.index = [round(i.mid) for i in t.index]
    return t.round(0)

cp, ck = curve(pen), curve(kel)
tab = pd.DataFrame({"warranted_MM82": np.interp(cp.index, gpc.iloc[:, 0], gpc.iloc[:, 1]).round(0)}, index=cp.index)
tab = tab.join(cp.add_prefix("pen_")).join(ck.add_prefix("kel_"))
print(tab.loc[3:16].to_string())

for name, df in [("penmanshiel", pen), ("kelmarsh", kel)]:
    sp = df["Turbine Power setpoint (kW)"]
    print(f"\n{name} setpoint: {sp.notna().mean():.1%} non-NaN | top values:", sp.round().value_counts().head(5).to_dict())

In [ ]:
# (a) status files: why are end timestamps missing, and can Duration recover them?
st = pd.concat((pd.read_parquet(p).assign(src=p.split("/")[-2]) for p in glob.glob("/kaggle/working/parquet/status/*/*.parquet")), ignore_index=True)
st["end_nat"] = st["Timestamp end"].isna()
print(st.groupby(["src", "Status"])["end_nat"].agg(["size", "mean"]).round(3).to_string())
dur = st["Duration"].str.extract(r"^(\d+):(\d{2}):(\d{2})$").astype(float)
st["dur"] = pd.to_timedelta(dur[0] * 3600 + dur[1] * 60 + dur[2], unit="s")
ok = (~st.end_nat) & st["dur"].notna()
mism = ((st["Timestamp end"] - st["Timestamp start"] - st["dur"]).abs()[ok] > pd.Timedelta("2s")).mean()
print("duration parsed:", round(st["dur"].notna().mean(), 3), "| end != start+duration where both exist:", round(mism, 4))
print(st[st.end_nat].sample(5, random_state=0)[["src", "Timestamp start", "Timestamp end", "Duration", "Status", "Message"]].to_string())

# (b) setpoint: NaN share per year, and what power/wind the common values sit at
for name, df in [("penmanshiel", pen), ("kelmarsh", kel)]:
    d = df.copy(); d["year"] = d[TS].dt.year
    d["sp"] = d["Turbine Power setpoint (kW)"].round()
    d["over"] = d["Power (kW)"] > d["sp"] * 1.1 + 20
    print(f"\n{name}: setpoint NaN share by year:", d.groupby("year")["sp"].apply(lambda s: s.isna().mean()).round(2).to_dict())
    top = d["sp"].value_counts().head(6).index
    g = d[d["sp"].isin(top)].groupby("sp")
    print(pd.DataFrame({"n": g.size(), "ws_med": g["Wind speed (m/s)"].median(),
                        "pow_med": g["Power (kW)"].median(), "pow_p90": g["Power (kW)"].quantile(.9),
                        "share_pow_above_sp": g["over"].mean()}).round(2).to_string())

# (c) SMARTEOLE: where (wind direction) is the offset really applied?
off = ctl["control_log_offset_avg"]; wd = ctl["control_log_wind_direction_filt_60s_deg_avg"]
ws_ = ctl["control_log_wind_speed_filt_60s_avg"]; nz = off.abs() >= 0.5
print("\noffset sign:", {"pos": int((off >= 0.5).sum()), "neg": int((off <= -0.5).sum())}, "| min/max:", round(off.min(), 1), round(off.max(), 1))
sect = (wd // 30 * 30).astype("Int64")
print(pd.DataFrame({"all": sect.value_counts(), "flag_on": sect[ctl["control_log_offset_active_avg"] > 0.5].value_counts(),
                    "offset_nonzero": sect[nz].value_counts()}).fillna(0).astype(int).sort_index().to_string())
print("wind speed where offset nonzero:", ws_[nz].describe(percentiles=[.05, .5, .95]).round(1).to_dict())

In [ ]:
rated = 2050
LC = "Lost Production to Curtailment (Total) (kWh)"
cols = [TS, "Wind speed (m/s)", "Power (kW)", "Turbine Power setpoint (kW)", LC]
STAT = ["Stop", "Curtailment", "Communication"]

def build(farm):
    parts = []
    for f in sorted(glob.glob(f"/kaggle/working/parquet/{farm}/*.parquet")):
        x = pd.read_parquet(f, columns=cols)
        x["turbine"] = int(os.path.basename(f).split("_")[1])
        parts.append(x)
    d = pd.concat(parts, ignore_index=True).sort_values(["turbine", TS]).reset_index(drop=True)
    st = pd.concat((pd.read_parquet(p).assign(turbine=int(os.path.basename(p).split("_")[1]))
                    for p in glob.glob(f"/kaggle/working/parquet/status/{farm}/*.parquet")), ignore_index=True)
    st = st[st["Status"].isin(STAT) & st["Timestamp end"].notna()].drop_duplicates()
    for name in STAT:
        d["st_" + name.lower()] = False
    for t, idx in d.groupby("turbine").indices.items():
        ts = d[TS].values[idx]
        for name in STAT:
            s = st[(st.turbine == t) & (st.Status == name)]
            i0 = np.searchsorted(ts, (s["Timestamp start"] - pd.Timedelta("10min")).values, side="right")
            i1 = np.searchsorted(ts, s["Timestamp end"].values, side="left")
            ok = i1 > i0
            diff = np.zeros(len(ts) + 1, np.int32)
            np.add.at(diff, i0[ok], 1); np.add.at(diff, i1[ok], -1)
            d.loc[idx, "st_" + name.lower()] = np.cumsum(diff[:-1]) > 0
    d["sp_low_hiwind"] = (d["Turbine Power setpoint (kW)"] < 0.95 * rated) & (d["Wind speed (m/s)"] >= 5)
    d["loss_curt"] = d[LC] > 0.5
    d["core_nan"] = d["Wind speed (m/s)"].isna() | d["Power (kW)"].isna()
    return d

flags = ["st_stop", "st_curtailment", "st_communication", "sp_low_hiwind", "loss_curt", "core_nan"]
flagged = {}
for farm in ["penmanshiel", "kelmarsh"]:
    d = build(farm)
    hw = d["Wind speed (m/s)"] >= 8
    clean = ~d[flags].any(axis=1)
    print(f"\n=== {farm}: {len(d):,} intervals | clean {clean.mean():.1%} (ws>=8: {clean[hw].mean():.1%})")
    print(pd.DataFrame({"all": d[flags].mean(), "ws>=8": d.loc[hw, flags].mean()}).round(3).to_string())

    x = d[d["Wind speed (m/s)"].between(5.5, 15.5)].copy()
    x["bin"] = x["Wind speed (m/s)"].round().astype(int)
    x["clean"] = clean[x.index]
    a = x.groupby("bin")["Power (kW)"].quantile([.1, .5]).unstack().add_prefix("all_q")
    c = x[x.clean].groupby("bin")["Power (kW)"].quantile([.1, .5]).unstack().add_prefix("clean_q")
    print(a.join(c).round(0).to_string())

    tail = d[(d["Wind speed (m/s)"] >= 13) & (d["Power (kW)"] < 0.88 * rated)]
    print(f"ws>=13 & P<{0.88*rated:.0f}: {len(tail):,} of {int((d['Wind speed (m/s)'] >= 13).sum()):,} | flagged by:",
          tail[flags].mean().round(2).to_dict(), "| none:", round((~tail[flags].any(axis=1)).mean(), 2))
    print("setpoints there:", tail["Turbine Power setpoint (kW)"].round().value_counts(dropna=False).head(6).to_dict())
    flagged[farm] = d[[TS, "turbine"] + flags]
    del d, x

In [ ]:
for farm in ["penmanshiel", "kelmarsh"]:
    d = build(farm)
    hw = d[(d["Wind speed (m/s)"] >= 8) & d["Turbine Power setpoint (kW)"].notna()].copy()
    hw["sp"] = hw["Turbine Power setpoint (kW)"].round()
    hw["ratio"] = hw["Power (kW)"] / hw["sp"].where(hw["sp"] > 0)
    g = hw.groupby("sp").agg(n=("sp", "size"), ws_med=("Wind speed (m/s)", "median"),
                             p_med=("Power (kW)", "median"), ratio_med=("ratio", "median"),
                             stop=("st_stop", "mean"))
    g["share"] = g["n"] / len(hw)
    print(f"\n=== {farm}: setpoint values at ws>=8 ({len(hw):,} rows with a setpoint)")
    print(g[g["n"] >= 500].sort_values("n", ascending=False).head(15).round(2).to_string())
    low = hw[hw["sp"] < 0.95 * rated]
    print("sp < 95% rated:", len(low), "| by year:", low.groupby(low[TS].dt.year).size().to_dict())
    del d, hw

In [ ]:
for farm in ["penmanshiel", "kelmarsh"]:
    d = build(farm)
    d = d[d["Turbine Power setpoint (kW)"].notna() & d["Wind speed (m/s)"].notna()].copy()
    sp, ws = d["Turbine Power setpoint (kW)"], d["Wind speed (m/s)"]
    stat = d[["st_stop", "st_curtailment", "st_communication"]].any(axis=1)
    ref = d[(sp >= 2030) & ~stat & (ws >= 3)]
    curve = ref.groupby((ref["Wind speed (m/s)"] * 2).round() / 2)["Power (kW)"].median()
    d["p_ref"] = np.interp(ws, curve.index.values, curve.values)

    c = d[(sp < 0.95 * rated) & (sp > 0.25 * rated) & (ws >= 5)].copy()
    c["could_bind"] = c["p_ref"] > 1.05 * c["Turbine Power setpoint (kW)"]
    c["binding"] = c["could_bind"] & (c["Power (kW)"] >= 0.9 * c["Turbine Power setpoint (kW)"])
    c["bucket"] = pd.cut(c["Turbine Power setpoint (kW)"], [500, 1000, 1500, 1800, 1947])
    c["pw_vs_ref"] = c["Power (kW)"] / c["p_ref"]
    g = c.groupby("bucket", observed=True).agg(n=("binding", "size"), could_bind=("could_bind", "mean"),
                                               binding=("binding", "mean"), pw_vs_ref=("pw_vs_ref", "median"),
                                               stop=("st_stop", "mean"))
    print(f"\n=== {farm}: candidates {len(c):,} | could bind {c.could_bind.mean():.1%} | binding {c.binding.mean():.1%}")
    print(g.round(3).to_string())
    print("by year, binding rows:", c[c.binding].groupby(c[TS].dt.year).size().to_dict())
    del d, c

In [ ]:
flags2 = ["st_stop", "st_curtailment", "st_communication", "sp_idle", "sp_binding", "loss_curt", "core_nan"]
os.makedirs("/kaggle/working/parquet/flags", exist_ok=True)

for farm in ["penmanshiel", "kelmarsh"]:
    d = build(farm)
    sp, ws, P = d["Turbine Power setpoint (kW)"], d["Wind speed (m/s)"], d["Power (kW)"]
    stat = d[["st_stop", "st_curtailment", "st_communication"]].any(axis=1)
    ref = d[(sp >= 2030) & ~stat & (ws >= 3)]
    curve = ref.groupby((ref["Wind speed (m/s)"] * 2).round() / 2)["Power (kW)"].median()
    p_ref = np.interp(ws, curve.index.values, curve.values)

    d["sp_idle"] = (sp <= 0.25 * rated) & (ws >= 5)
    d["sp_binding"] = ((sp < 0.95 * rated) & (sp > 0.25 * rated) & (ws >= 5)
                       & (p_ref > 1.05 * sp) & (P >= 0.9 * sp))
    bad = d[flags2].any(axis=1)
    d["mask_m1"] = bad                      # True = exclude from model-1 training
    hw = ws >= 8

    print(f"\n=== {farm}: {len(d):,} intervals | clean {(~bad).mean():.1%} (ws>=8: {(~bad)[hw].mean():.1%})")
    print(pd.DataFrame({"all": d[flags2].mean(), "ws>=8": d.loc[hw, flags2].mean()}).round(3).to_string())

    x = d[ws.between(5.5, 15.4)].copy()
    x["bin"] = x["Wind speed (m/s)"].round().astype(int)
    a = x.groupby("bin")["Power (kW)"].quantile([.1, .5]).unstack().add_prefix("all_q")
    c = x[~x.mask_m1].groupby("bin")["Power (kW)"].quantile([.1, .5]).unstack().add_prefix("clean_q")
    print(a.join(c).round(0).to_string())

    tail = d[(ws >= 13) & (P < 0.88 * rated)]
    print(f"tail ws>=13 & P<{0.88*rated:.0f}: {len(tail):,} | flagged:",
          tail[flags2].mean().round(2).to_dict(), "| none:", round((~tail.mask_m1).mean(), 2))
    left = tail[~tail.mask_m1]
    print("unflagged tail: by year", left.groupby(left[TS].dt.year).size().to_dict(),
          "| by turbine", left.groupby("turbine").size().to_dict())
    print("unflagged tail: median ws", round(left["Wind speed (m/s)"].median(), 1), "| median P", round(left["Power (kW)"].median()))

    d[[TS, "turbine"] + flags2 + ["mask_m1"]].to_parquet(f"/kaggle/working/parquet/flags/{farm}_flags.parquet", index=False)
    del d, x

In [ ]:
for farm in ["penmanshiel", "kelmarsh"]:
    fl = pd.read_parquet(f"/kaggle/working/parquet/flags/{farm}_flags.parquet", columns=[TS, "turbine", "mask_m1"])
    parts = []
    for f in sorted(glob.glob(f"/kaggle/working/parquet/{farm}/*.parquet")):
        x = pd.read_parquet(f, columns=[TS, "Wind speed (m/s)", "Power (kW)"])
        x["turbine"] = int(os.path.basename(f).split("_")[1]); parts.append(x)
    d = pd.concat(parts).drop_duplicates(["turbine", TS])
    d = d.merge(fl.drop_duplicates(["turbine", TS]), on=["turbine", TS])
    ws, P = d["Wind speed (m/s)"], d["Power (kW)"]
    d["year"], d["month"] = d[TS].dt.year, d[TS].dt.month
    hi = ws >= 13
    d["utail"] = hi & (P < 0.88 * rated) & ~d.mask_m1

    print(f"\n=== {farm}")
    g = d[hi].groupby("year").agg(n_hi=("utail", "size"), unflagged_tail=("utail", "sum"), rate=("utail", "mean"))
    print(g.round(4).to_string())
    y16 = d[d.year == 2016]
    print("2016 unflagged tail by month:", y16[y16.utail].groupby("month").size().to_dict())
    print("2016 rows by month:", y16.groupby("month").size().to_dict())
    m = d[(~d.mask_m1) & ws.between(10, 12)]
    print(m.groupby("year")["Power (kW)"].quantile([.1, .5]).unstack().round(0).to_string())
    del d

In [ ]:
def load_merged(farm):
    fl = pd.read_parquet(f"/kaggle/working/parquet/flags/{farm}_flags.parquet", columns=[TS, "turbine", "mask_m1"])
    parts = []
    for f in sorted(glob.glob(f"/kaggle/working/parquet/{farm}/*.parquet")):
        x = pd.read_parquet(f, columns=[TS, "Wind speed (m/s)", "Power (kW)"])
        x["turbine"] = int(os.path.basename(f).split("_")[1]); parts.append(x)
    d = pd.concat(parts).drop_duplicates(["turbine", TS])
    return d.merge(fl.drop_duplicates(["turbine", TS]), on=["turbine", TS])

# Kelmarsh: monthly clean median at 10-12 m/s, per turbine
k = load_merged("kelmarsh")
k = k[(k[TS] < "2017-07-01") & (~k.mask_m1) & k["Wind speed (m/s)"].between(10, 12)]
k["ym"] = k[TS].dt.to_period("M")
print("Kelmarsh clean median power at 10-12 m/s, by month x turbine")
print(k.pivot_table(index="ym", columns="turbine", values="Power (kW)", aggfunc="median").round(0).to_string())
print("rows per month:", k.groupby("ym").size().to_dict())

# Penmanshiel: November 2016 unflagged tail by day and turbine
p = load_merged("penmanshiel")
n = p[(p[TS] >= "2016-11-01") & (p[TS] < "2016-12-01")].copy()
n["utail"] = (n["Wind speed (m/s)"] >= 13) & (n["Power (kW)"] < 0.88 * rated) & ~n.mask_m1
print("\nPenmanshiel Nov 2016 unflagged tail by day:", n[n.utail].groupby(n[TS].dt.day).size().to_dict())
print("by turbine:", n[n.utail].groupby("turbine").size().to_dict())

In [ ]:
# 1) Kelmarsh: exact date of the step change
k = load_merged("kelmarsh")
ws = k["Wind speed (m/s)"]
kc = k[(~k.mask_m1) & ws.between(10, 12)]
w = kc[(kc[TS] >= "2016-11-20") & (kc[TS] < "2016-12-20")]
print("Kelmarsh daily clean median at 10-12 m/s (farm-wide), Nov 20 - Dec 19 2016")
print(w.groupby(w[TS].dt.date)["Power (kW)"].agg(["median", "size"]).round(0).T.to_string())

# 2) Kelmarsh: calendar month x year, to see seasonality vs regime
kc2 = kc[kc[TS] >= "2016-12-01"]
print("\nKelmarsh clean median at 10-12 m/s, month x year (from Dec 2016)")
print(kc2.pivot_table(index=kc2[TS].dt.month, columns=kc2[TS].dt.year,
                      values="Power (kW)", aggfunc="median").round(0).to_string())

# 3) Penmanshiel: hourly view of 2016-11-04/05
p = load_merged("penmanshiel")
n = p[(p[TS] >= "2016-11-04 18:00") & (p[TS] < "2016-11-06 06:00")].copy()
n["tail"] = (n["Wind speed (m/s)"] >= 13) & (n["Power (kW)"] < 0.88 * rated)
n["hour"] = n[TS].dt.floor("h")
g = n.groupby("hour").agg(turbines=("turbine", "nunique"), n_tail=("tail", "sum"),
                          ws_med=("Wind speed (m/s)", "median"), ws_max=("Wind speed (m/s)", "max"),
                          p_med=("Power (kW)", "median"), flagged=("mask_m1", "mean"))
print("\nPenmanshiel 2016-11-04 18:00 to 11-06 06:00, hourly")
print(g[g.n_tail > 0].round(2).to_string())

In [ ]:
# 1) Kelmarsh: step date from high-wind rows (many more rows than 10-12 m/s)
k = load_merged("kelmarsh")
ws, P = k["Wind speed (m/s)"], k["Power (kW)"]
h = k[(~k.mask_m1) & (ws >= 14) & (k[TS] >= "2016-11-25") & (k[TS] < "2017-01-15")]
print("Kelmarsh clean ws>=14: daily median power / rows")
print(h.groupby(h[TS].dt.date)["Power (kW)"].agg(["median", "size"]).round(0).to_string())

# 2) cap vs curve shift: same season, 2016 vs 2017
def curve(a, b):
    x = k[(~k.mask_m1) & (k[TS] >= a) & (k[TS] < b) & (k["Wind speed (m/s)"] >= 3.5)].copy()
    x["bin"] = np.floor(x["Wind speed (m/s)"] + 0.5).astype(int)
    return x.groupby("bin")["Power (kW)"].agg(p50="median", p90=lambda s: s.quantile(.9), n="size")
c16, c17 = curve("2016-02-01", "2016-04-01"), curve("2017-02-01", "2017-04-01")
print("\nFeb-Mar 2016 vs Feb-Mar 2017, clean rows")
print(c16.join(c17, lsuffix="_2016", rsuffix="_2017").round(0).to_string())
pre = k[(~k.mask_m1) & (k[TS] >= "2016-02-01") & (k[TS] < "2016-12-01") & (k["Wind speed (m/s)"] >= 14)]
print("2016 ws>=14, most common power values (25 kW bins):",
      ((pre["Power (kW)"] / 25).round() * 25).value_counts().head(6).to_dict())

# 3) generic farm-cap detector: many turbines, same power, below rated, above rated wind
def farm_cap(farm, min_t):
    d = load_merged(farm)
    d = d[d["Wind speed (m/s)"] >= 13]
    g = d.groupby(TS)["Power (kW)"]
    s = pd.DataFrame({"n": g.size(), "med": g.median(), "iqr": g.quantile(.75) - g.quantile(.25)})
    return s[(s.n >= min_t) & (s.iqr < 20) & s.med.between(0.3 * rated, 0.95 * rated)]
for farm, mt in [("penmanshiel", 5), ("kelmarsh", 4)]:
    s = farm_cap(farm, mt)
    ym = s.groupby([s.index.year, s.index.month]).size()
    print(f"\n=== {farm}: {len(s)} cap-like timestamps")
    print("top year-months:", ym.sort_values(ascending=False).head(12).to_dict())
    print("plateau levels (25 kW bins):", ((s["med"] / 25).round() * 25).value_counts().head(8).to_dict())

In [ ]:
def load_full(farm):
    fl = pd.read_parquet(f"/kaggle/working/parquet/flags/{farm}_flags.parquet").drop_duplicates(["turbine", TS])
    fl = fl.drop(columns=["mask_m1"], errors="ignore")
    parts = []
    for f in sorted(glob.glob(f"/kaggle/working/parquet/{farm}/*.parquet")):
        x = pd.read_parquet(f, columns=[TS, "Wind speed (m/s)", "Power (kW)"])
        x["turbine"] = int(os.path.basename(f).split("_")[1]); parts.append(x)
    d = pd.concat(parts).drop_duplicates(["turbine", TS])
    return d.merge(fl, on=["turbine", TS])

base = ["st_stop", "st_curtailment", "st_communication", "sp_idle", "sp_binding", "loss_curt", "core_nan"]
new = ["cap_kel_2016", "farm_cap", "commissioning"]
cfg = {"penmanshiel": dict(min_t=5, comm="2016-09-01"), "kelmarsh": dict(min_t=4, comm="2016-02-01")}

for farm, c in cfg.items():
    d = load_full(farm)
    ws, P = d["Wind speed (m/s)"], d["Power (kW)"]
    hi = d[ws >= 13]
    g = hi.groupby(TS)["Power (kW)"]
    s = pd.DataFrame({"n": g.size(), "med": g.median(), "iqr": g.quantile(.75) - g.quantile(.25)})
    s = s[(s.n >= c["min_t"]) & (s.iqr < 20) & s.med.between(0.3 * rated, 0.95 * rated)]
    d["farm_cap"] = d[TS].isin(s.index)
    d["cap_kel_2016"] = (farm == "kelmarsh") & (d[TS] < "2016-12-23") & (ws >= 9)
    d["commissioning"] = d[TS] < c["comm"]
    d["mask_m1"] = d[base + new].any(axis=1)

    hw = ws >= 8
    print(f"\n=== {farm}: clean {(~d.mask_m1).mean():.1%} (ws>=8: {(~d.mask_m1)[hw].mean():.1%})")
    print(pd.DataFrame({"all": d[new].mean(), "ws>=8": d.loc[hw, new].mean()}).round(4).to_string())

    d["year"] = d[TS].dt.year
    m = d[(~d.mask_m1) & ws.between(10, 12)].groupby("year")["Power (kW)"].quantile([.1, .5]).unstack().round(0)
    hi = d[ws >= 13]
    ut = ((hi["Power (kW)"] < 0.88 * rated) & ~hi.mask_m1).groupby(hi.year).mean().round(4)
    print(m.join(ut.rename("unflagged_tail_rate")).to_string())

    d[[TS, "turbine"] + base + new + ["mask_m1"]].to_parquet(
        f"/kaggle/working/parquet/flags/{farm}_flags.parquet", index=False)
    del d

In [ ]:
spec = """mask_m1 definition (frozen 2026-11-06). True = exclude from model-1 training; keep rows for analysis and model 3.
st_stop / st_curtailment / st_communication: interval overlaps Greenbyte status event with valid end (window start-10min .. end)
sp_idle: setpoint <= 0.25*2050 and ws >= 5
sp_binding: 0.25*2050 < setpoint < 0.95*2050, ws >= 5, reference power > 1.05*setpoint, measured power >= 0.9*setpoint
loss_curt: Lost Production to Curtailment > 0.5 kWh
core_nan: wind speed or power missing
farm_cap: timestamp where >= 5 (Penmanshiel) / 4 (Kelmarsh) turbines at ws>=13 have power IQR < 20 kW and median in 30-95% of rated
cap_kel_2016: Kelmarsh, before 2016-12-23, ws >= 9 (unlogged ~1,675 kW cap)
commissioning: Kelmarsh < 2016-02-01, Penmanshiel < 2016-09-01
Not covered: Penmanshiel Nov 2016 residual (~3% tail rate), caps binding below 13 m/s, 2016-2017 setpoint-based rules (setpoint NaN).
"""
open("/kaggle/working/parquet/flags/MASK_SPEC.txt", "w").write(spec)
api.upload_folder(folder_path="/kaggle/working/parquet/flags", path_in_repo="parquet/flags",
                  repo_id=REPO, repo_type="dataset", commit_message="Add frozen mask flags")
print(sorted(f for f in api.list_repo_files(REPO, repo_type="dataset") if f.startswith("parquet/flags")))

In [ ]:
FEATS = ["Wind speed (m/s)", "Wind speed, Standard deviation (m/s)", "Wind speed, Minimum (m/s)", "Wind speed, Maximum (m/s)",
         "Wind speed Sensor 1 (m/s)", "Wind speed Sensor 2 (m/s)", "Density adjusted wind speed (m/s)",
         "Wind direction (°)", "Wind direction, Standard deviation (°)", "Nacelle position (°)", "Vane position 1+2 (°)",
         "Yaw bearing angle (°)", "Nacelle ambient temperature (°C)",
         "Power (kW)", "Power, Standard deviation (kW)", "Power, Minimum (kW)", "Power, Maximum (kW)",
         "Turbine Power setpoint (kW)", "Blade angle (pitch position) A (°)", "Blade angle (pitch position) B (°)",
         "Blade angle (pitch position) C (°)", "Rotor speed (RPM)", "Generator RPM (RPM)"]

def load_feat(farm):
    parts = []
    for f in sorted(glob.glob(f"/kaggle/working/parquet/{farm}/*.parquet")):
        x = pd.read_parquet(f)
        x["turbine"] = int(os.path.basename(f).split("_")[1]); parts.append(x)
    d = (pd.concat(parts).sort_values(["turbine", TS])
           .drop_duplicates(["turbine", TS], keep="last").reset_index(drop=True))
    fl = pd.read_parquet(f"/kaggle/working/parquet/flags/{farm}_flags.parquet",
                         columns=[TS, "turbine", "mask_m1"]).drop_duplicates(["turbine", TS])
    d = d.merge(fl, on=[TS, "turbine"], how="left")
    d["year"] = d[TS].dt.year
    return d

def frozen(d, c, n=6):
    x = d[c]
    new = x.ne(x.shift()) | d["turbine"].ne(d["turbine"].shift()) | d[TS].diff().ne(pd.Timedelta("10min"))
    rid = new.cumsum()
    return (rid.groupby(rid).transform("size") >= n) & x.notna()

for farm in ["penmanshiel", "kelmarsh"]:
    d = load_feat(farm)
    num = [c for c in FEATS if c in d.columns]
    print(f"\n=== {farm}: {len(d):,} rows")

    print("\n-- NaN % by column x year")
    print((d[num].isna().groupby(d.year).mean().T * 100).round(1).to_string())

    g = lambda c: d[c] if c in d.columns else pd.Series(np.nan, index=d.index)
    ws, P, wd = g("Wind speed (m/s)"), g("Power (kW)"), g("Wind direction (°)")
    rules = {
        "ws < 0": ws < 0,
        "ws > 35": ws > 35,
        "P > 1.1*rated": P > 1.1 * rated,
        "P < -50": P < -50,
        "wind dir outside 0-360": (wd < 0) | (wd > 360),
        "temp < -25 or > 45": (g("Nacelle ambient temperature (°C)") < -25) | (g("Nacelle ambient temperature (°C)") > 45),
        "pitch A < -5 or > 95": (g("Blade angle (pitch position) A (°)") < -5) | (g("Blade angle (pitch position) A (°)") > 95),
        "rotor rpm < 0 or > 20": (g("Rotor speed (RPM)") < 0) | (g("Rotor speed (RPM)") > 20),
        "ws min > mean or max < mean": (g("Wind speed, Minimum (m/s)") > ws + 0.01) | (g("Wind speed, Maximum (m/s)") < ws - 0.01),
        "P min > mean or max < mean": (g("Power, Minimum (kW)") > P + 0.5) | (g("Power, Maximum (kW)") < P - 0.5),
    }
    r = pd.DataFrame({k: dict(n=int(v.sum()), pct=round(v.mean() * 100, 3),
                              in_mask_pct=round(d.loc[v, "mask_m1"].mean() * 100, 1) if v.any() else np.nan)
                      for k, v in rules.items()}).T
    print("\n-- impossible / inconsistent values")
    print(r.to_string())

    fcols = [c for c in ["Wind speed (m/s)", "Wind speed Sensor 1 (m/s)", "Wind speed Sensor 2 (m/s)",
                         "Wind direction (°)", "Vane position 1+2 (°)", "Nacelle ambient temperature (°C)"] if c in d.columns]
    fz = pd.DataFrame({c: frozen(d, c) for c in fcols})
    print("\n-- frozen >= 1h, % of rows by year")
    print((fz.groupby(d.year).mean() * 100).round(2).T.to_string())
    for c in fcols[:1] + fcols[3:4]:
        t = fz[c].groupby(d.turbine).mean().sort_values(ascending=False).head(3) * 100
        print(f"worst turbines for frozen '{c}': {t.round(2).to_dict()}")
    print("frozen wind speed while P > 200 kW:", int((fz[fcols[0]] & (P > 200)).sum()),
          "| not in mask:", int((fz[fcols[0]] & ~d.mask_m1.astype(bool)).sum()))
    del d, fz

In [ ]:
def circ(a): return (a + 180) % 360 - 180

for farm in ["penmanshiel", "kelmarsh"]:
    d = load_feat(farm)
    d["ok"] = ~d["mask_m1"].fillna(True).astype(bool)
    num = [c for c in FEATS if c in d.columns]
    print(f"\n=== {farm}")

    # 1) availability onset: first month with <50% NaN, farm-wide
    m = d[TS].dt.to_period("M")
    na = d[num].isna().groupby(m).mean()
    onset = (na < 0.5).idxmax().astype(str)
    never = (na >= 0.5).all()
    onset[never] = "never"
    print("-- first month with >=50% data:", {k: v for k, v in onset.items() if v != str(m.min())})

    c = d[d.ok & d["Wind speed (m/s)"].between(3, 25)].copy()
    ws = c["Wind speed (m/s)"]

    # 2) channel ratios by year
    print("\n-- ratio to 'Wind speed (m/s)' on clean rows: median [p5, p95], by year")
    for col in ["Wind speed Sensor 1 (m/s)", "Wind speed Sensor 2 (m/s)", "Density adjusted wind speed (m/s)"]:
        r = (c[col] / ws).groupby(c.year).quantile([.05, .5, .95]).unstack().round(3)
        print(col); print(r.T.to_string())
    s12 = (c["Wind speed Sensor 1 (m/s)"] / c["Wind speed Sensor 2 (m/s)"])
    print("\n-- Sensor1 / Sensor2 median, turbine x year")
    print(s12.groupby([c.turbine, c.year]).median().unstack().round(3).to_string())

    # 3) turbine anemometer bias vs farm median at same timestamp
    b = c[c["Wind speed (m/s)"].between(4, 12)]
    gm = b.groupby(TS)["Wind speed (m/s)"].transform("median")
    n = b.groupby(TS)["Wind speed (m/s)"].transform("size")
    rel = (b["Wind speed (m/s)"] / gm)[n >= 4]
    print("\n-- turbine ws / farm median ws (4-12 m/s), turbine x year")
    print(rel.groupby([b.turbine[n >= 4], b.year[n >= 4]]).median().unstack().round(3).to_string())

    # 4) direction convention
    cd = c.dropna(subset=["Wind direction (°)", "Nacelle position (°)", "Vane position 1+2 (°)"])
    wd, nac, vn = cd["Wind direction (°)"], cd["Nacelle position (°)"], cd["Vane position 1+2 (°)"]
    print("\n-- direction convention: circular median / std of difference (deg)")
    for name, diff in [("wd - nac", circ(wd - nac)), ("wd - nac - vane", circ(wd - nac - vn)),
                       ("wd - nac + vane", circ(wd - nac + vn))]:
        print(f"{name:18s} median {diff.median():7.2f}  p5 {diff.quantile(.05):7.2f}  p95 {diff.quantile(.95):7.2f}")
    print("vane position range:", vn.quantile([0, .01, .5, .99, 1]).round(1).to_dict())

    # 5) density-adjusted ratio vs temperature
    t = c["Nacelle ambient temperature (°C)"]
    ratio = c["Density adjusted wind speed (m/s)"] / ws
    tb = pd.cut(t, [-20, -5, 0, 5, 10, 15, 20, 25, 40])
    g = ratio.groupby(tb, observed=True).agg(["median", "size"]).round(3)
    g["expected_cuberoot"] = [round((288.15 / (np.interp(x.mid, [-20, 40], [-20, 40]) + 273.15)) ** (1 / 3), 3) for x in g.index]
    print("\n-- density-adj / ws by temperature bin (expected = ideal gas at 1013 hPa, relative to 15C)")
    print(g.to_string())
    del d, c, b

In [ ]:
for farm in ["penmanshiel", "kelmarsh"]:
    d = load_feat(farm)
    d["ok"] = ~d["mask_m1"].fillna(True).astype(bool)
    c = d[d.ok].copy()
    ws = c["Wind speed (m/s)"]
    print(f"\n=== {farm}: {len(c):,} clean rows")

    print("\n-- wind speed quantiles by year (clean)")
    print(ws.groupby(c.year).quantile([.05, .25, .5, .75, .95]).unstack().round(2).to_string())

    wd = (c["Wind direction (°)"] % 360)
    sec = (np.floor(wd / 30) * 30).astype("Int64")
    print("\n-- wind direction sector share (%), by year")
    print((pd.crosstab(sec, c.year, normalize="columns") * 100).round(1).to_string())

    b = c[c["Wind speed (m/s)"].between(4, 12) & c["Wind speed, Standard deviation (m/s)"].notna()]
    ti = b["Wind speed, Standard deviation (m/s)"] / b["Wind speed (m/s)"]
    print("\n-- turbulence intensity (std/mean) quantiles at 4-12 m/s, by year")
    print(ti.groupby(b.year).quantile([.1, .5, .9]).unstack().round(3).to_string())

    print("\n-- temperature quantiles by year")
    print(c["Nacelle ambient temperature (°C)"].groupby(c.year).quantile([.05, .5, .95]).unstack().round(1).to_string())

    m = c[c["Wind speed (m/s)"].between(6, 10)]
    pm = m.groupby(["turbine", "year"])["Power (kW)"].median().unstack()
    rel = pm.div(pm.median(axis=1), axis=0)
    print("\n-- clean median power at 6-10 m/s, relative to turbine's own all-year median")
    print(rel.round(3).to_string())
    del d, c, m

In [ ]:
for farm in ["penmanshiel", "kelmarsh"]:
    d = load_feat(farm)
    c = d[~d["mask_m1"].fillna(True).astype(bool) & d["Wind speed (m/s)"].between(5, 9)].copy()
    c["bin"] = np.floor(c["Wind speed (m/s)"] * 2) / 2
    c["res"] = c["Power (kW)"] / c.groupby("bin")["Power (kW)"].transform("median")
    g = c.groupby(["turbine", "year"]).agg(res=("res", "median"), n=("res", "size")).reset_index()
    g.loc[g.n < 500, "res"] = np.nan
    t = g.pivot(index="turbine", columns="year", values="res")
    rel = t.div(t.median(axis=0), axis=0 if False else 1) if False else t.div(t.median(axis=0), axis=1)
    print(f"\n=== {farm}: power / farm reference curve (5-9 m/s, 0.5 m/s bins), turbine x year")
    print(t.round(3).to_string())
    print("\nrelative to farm-year median")
    print(rel.round(3).to_string())
    s = rel.stack()
    print("turbine-years off by >5%:", {k: round(v, 3) for k, v in s[(s - 1).abs() > 0.05].items()})
    del d, c

In [ ]:
login(UserSecretsClient().get_secret("HF_TOKEN"))

drift = {"penmanshiel": [(8, "2021-01-01", "2023-01-01"), (2, "2021-01-01", "2022-01-01")], "kelmarsh": []}
for farm, rules in drift.items():
    p = f"/kaggle/working/parquet/flags/{farm}_flags.parquet"
    f = pd.read_parquet(p)
    f["sensor_drift"] = False
    for t, a, b in rules:
        f.loc[(f.turbine == t) & (f[TS] >= a) & (f[TS] < b), "sensor_drift"] = True
    f["mask_m1"] = f[base + new + ["sensor_drift"]].any(axis=1)
    f.to_parquet(p, index=False)
    print(farm, "sensor_drift share:", round(f.sensor_drift.mean(), 4), "| clean:", round((~f.mask_m1).mean(), 4))

with open("/kaggle/working/parquet/flags/MASK_SPEC.txt", "a") as fh:
    fh.write("sensor_drift (added 2026-11-06): Penmanshiel T8 2021-2022, T2 2021 (power vs farm reference curve off by >5%, anemometer drift suspected). "
             "Decision: Kelmarsh model-1 training/eval from 2018-01 (farm-wide step 2017->2018, missing columns before).\n")
api.upload_folder(folder_path="/kaggle/working/parquet/flags", path_in_repo="parquet/flags",
                  repo_id=REPO, repo_type="dataset", commit_message="Add sensor_drift flag")
print("uploaded")

In [ ]:
cand = ["ws", "sin_wd", "cos_wd", "vane", "abs_vane", "temp", "TI"]
ctrl = ["P", "rpm", "pitchA", "setpoint"]

for farm in ["penmanshiel", "kelmarsh"]:
    d = load_feat(farm)
    c = d[~d["mask_m1"].fillna(True).astype(bool) & (d[TS] >= "2018-04-01")].copy()
    wd = np.deg2rad(c["Wind direction (°)"])
    X = pd.DataFrame({
        "ws": c["Wind speed (m/s)"],
        "sin_wd": np.sin(wd), "cos_wd": np.cos(wd),
        "vane": c["Vane position 1+2 (°)"].clip(-45, 45),
        "temp": c["Nacelle ambient temperature (°C)"],
        "TI": c["Wind speed, Standard deviation (m/s)"] / c["Wind speed (m/s)"].where(c["Wind speed (m/s)"] >= 1),
        "P": c["Power (kW)"], "rpm": c["Rotor speed (RPM)"],
        "pitchA": c["Blade angle (pitch position) A (°)"], "setpoint": c["Turbine Power setpoint (kW)"],
    })
    X["abs_vane"] = X["vane"].abs()
    X = X.dropna()
    X = X.sample(min(400_000, len(X)), random_state=0)
    print(f"\n=== {farm}: {len(X):,} sampled clean rows (2018-04+)")

    print("\n-- Spearman (candidates + control-side signals)")
    print(X[cand + ctrl].corr(method="spearman").round(2).to_string())

    Z = X[cand]; Z = (Z - Z.mean()) / Z.std()
    vif = pd.Series(np.diag(np.linalg.inv(np.corrcoef(Z.values.T))), index=cand)
    print("\n-- VIF (Pearson, candidates only)")
    print(vif.round(2).to_dict())

    X["bin"] = np.floor(X["ws"]).astype(int)
    rows = {}
    for b, g in X[X.bin.between(4, 11)].groupby("bin"):
        rows[b] = g[[c_ for c_ in cand if c_ != "ws"] + ["P"]].corr(method="spearman")["P"].drop("P")
    w = pd.DataFrame(rows).T
    print("\n-- within-bin Spearman with power, by ws bin (4-12 m/s)")
    print(w.round(2).to_string())
    print("mean:", w.mean().round(3).to_dict())

    q = X.groupby("bin")[["rpm", "pitchA", "setpoint"]].median().loc[3:20]
    print("\n-- median rotor speed / pitch / setpoint by ws bin")
    print(q.round(1).to_string())
    del d, c, X

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor as HGB
from sklearn.metrics import r2_score
from sklearn.inspection import permutation_importance

def feats(c):
    wd = np.deg2rad(c["Wind direction (°)"])
    X = pd.DataFrame({
        "ws": c["Wind speed (m/s)"],
        "sin_wd": np.sin(wd), "cos_wd": np.cos(wd),
        "vane": c["Vane position 1+2 (°)"].clip(-45, 45),
        "temp": c["Nacelle ambient temperature (°C)"],
        "TI": c["Wind speed, Standard deviation (m/s)"] / c["Wind speed (m/s)"].where(c["Wind speed (m/s)"] >= 1),
        "turbine": c["turbine"].astype(float)}, index=c.index)
    X["abs_vane"] = X["vane"].abs()
    return X

SETS = {"A_ws": ["ws"],
        "B_feats": ["ws", "sin_wd", "cos_wd", "vane", "abs_vane", "temp", "TI"],
        "C_feats+turb": ["ws", "sin_wd", "cos_wd", "vane", "abs_vane", "temp", "TI", "turbine"]}

for farm in ["penmanshiel", "kelmarsh"]:
    d = load_feat(farm)
    c = d[~d["mask_m1"].fillna(True).astype(bool) & (d[TS] >= "2018-04-01")
          & d["Wind speed (m/s)"].between(3, 20)].copy()
    X = feats(c); X["P"] = c["Power (kW)"]; X["year"] = c.year
    X = X.dropna()
    tr = X[X.year <= 2021].sample(400_000, random_state=0) if (X.year <= 2021).sum() > 400_000 else X[X.year <= 2021]
    te = X[X.year >= 2022]
    te = te.sample(min(200_000, len(te)), random_state=1)
    print(f"\n=== {farm}: train {len(tr):,} (<=2021) | test {len(te):,} (>=2022)")

    b = lambda w: np.floor(w * 4) / 4 + 0.125
    cv = tr.groupby(b(tr.ws))["P"].median()
    curve = lambda w: np.interp(w, cv.index.values, cv.values)
    tr = tr.assign(r=tr.P - curve(tr.ws)); te = te.assign(r=te.P - curve(te.ws))
    tr["ar"], te["ar"] = tr.r.abs(), te.r.abs()

    g = te.assign(bin=np.floor(te.ws).astype(int)).groupby("bin").agg(
        n=("r", "size"), mean_r=("r", "mean"), std_r=("r", "std"), mean_abs=("ar", "mean"))
    print("\n-- test residual by ws bin (kW)"); print(g.round(1).to_string())

    res, preds = {}, {}
    for name, cols in SETS.items():
        row = {}
        for tgt in ["r", "ar"]:
            m = HGB(max_iter=200, learning_rate=0.1, random_state=0).fit(tr[cols], tr[tgt])
            p = m.predict(te[cols]); row[f"R2_{tgt}"] = r2_score(te[tgt], p)
            if tgt == "ar": preds[name] = p; res[name + "_model"] = (m, cols)
        res[name] = row
    print("\n-- held-out R2 for residual (r) and spread (|r|)")
    print(pd.DataFrame({k: v for k, v in res.items() if not k.endswith("_model")}).T.round(3).to_string())

    q = {}
    for name in ["A_ws", "B_feats"]:
        qq = pd.qcut(pd.Series(preds[name], index=te.index), 5, labels=False, duplicates="drop")
        q[name] = te["ar"].groupby(qq).mean()
    print("\n-- actual mean |r| (kW) by quintile of predicted |r|")
    print(pd.DataFrame(q).round(1).to_string())

    m, cols = res["B_feats_model"]
    s = te.sample(min(50_000, len(te)), random_state=2)
    pi = permutation_importance(m, s[cols], s["ar"], n_repeats=2, random_state=0, scoring="r2")
    print("\n-- permutation importance (|r| model B, drop in R2):",
          dict(zip(cols, pi.importances_mean.round(4))))
    del d, c, X, tr, te

In [ ]:
import io
for farm, fn in [("penmanshiel", "Penmanshiel_WT_static.csv"), ("kelmarsh", "Kelmarsh_WT_static.csv")]:
    p = hf_hub_download(REPO, f"{farm}/{fn}", repo_type="dataset", local_dir="/kaggle/working/hf")
    s = pd.read_csv(p)
    print(f"\n=== {farm}: {s.shape}")
    print(s.columns.tolist())
    print(s.head(20).to_string())

In [ ]:
def circ(a): return (a + 180) % 360 - 180

for farm, fn in [("penmanshiel", "Penmanshiel_WT_static.csv"), ("kelmarsh", "Kelmarsh_WT_static.csv")]:
    s = pd.read_csv(f"/kaggle/working/hf/{farm}/{fn}").dropna(subset=["Latitude"])
    s["turbine"] = s["Alternative Title"].astype(str).str.findall(r"\d+").str[-1].astype(int)
    s = s.set_index("turbine")
    lat0, lon0 = s.Latitude.mean(), s.Longitude.mean()
    s["x"] = (s.Longitude - lon0) * 111320 * np.cos(np.deg2rad(lat0))
    s["y"] = (s.Latitude - lat0) * 110540
    D = s["Rotor Diameter (m)"].iloc[0]
    T = s.index.tolist()

    # layout: nearest neighbour in D
    nn = {}
    for i in T:
        dist = {j: np.hypot(s.x[i]-s.x[j], s.y[i]-s.y[j]) / D for j in T if j != i}
        j = min(dist, key=dist.get); nn[i] = (j, round(dist[j], 1), round(s["Elevation (m)"][i]))
    print(f"\n=== {farm}: D={D:.0f} m | turbine: (nearest, dist in D, elevation m)")
    print(nn)

    # time-aligned panels, clean rows only
    d = load_feat(farm)
    d.loc[d["mask_m1"].fillna(True).astype(bool), ["Power (kW)", "Wind speed (m/s)", "Wind direction (°)"]] = np.nan
    pv = lambda c: d.pivot(index=TS, columns="turbine", values=c)
    Pw, Ws, Wd = pv("Power (kW)"), pv("Wind speed (m/s)"), pv("Wind direction (°)")
    wsm = Ws.median(axis=1).where(Ws.notna().sum(axis=1) >= 4)
    wdir = np.rad2deg(np.arctan2(np.sin(np.deg2rad(Wd)).median(axis=1),
                                 np.cos(np.deg2rad(Wd)).median(axis=1))) % 360
    base = wsm.between(6, 9)

    rows = []
    for i in T:
        for j in T:
            if i == j: continue
            dx, dy = s.x[j]-s.x[i], s.y[j]-s.y[i]
            dist = np.hypot(dx, dy) / D
            if dist > 8: continue
            b = np.degrees(np.arctan2(dx, dy)) % 360          # bearing i -> j
            a = circ(wdir - (b + 180) % 360).abs()             # 0 = wind blows from i toward j
            a_rev = circ(wdir - b).abs()                       # 0 = wind blows from j toward i
            ok = base & Pw[i].notna() & Pw[j].notna() & (Pw[i] > 100)
            ratio = Pw[j] / Pw[i]
            al = ok & (a <= 15)
            off = ok & (a > 30) & (a_rev > 30)
            if al.sum() < 300 or off.sum() < 1000: continue
            rows.append(dict(up=i, down=j, dist_D=round(dist, 1), bearing=round(b),
                             n_aligned=int(al.sum()), ratio_aligned=ratio[al].median(),
                             ratio_off=ratio[off].median(), d_elev=round(s["Elevation (m)"][j]-s["Elevation (m)"][i])))
    r = pd.DataFrame(rows)
    r["deficit"] = r.ratio_aligned / r.ratio_off - 1
    print(f"\n-- pairs <=8 D, ws 6-9 (farm median), deficit = aligned/off-axis - 1 ({len(r)} pairs)")
    print(r.sort_values("deficit").head(15).round(3).to_string(index=False))
    print("pairs with deficit < -10%:", int((r.deficit < -0.10).sum()), "| < -5%:", int((r.deficit < -0.05).sum()),
          "| median deficit by dist_D bucket:",
          r.groupby(pd.cut(r.dist_D, [0, 3, 5, 8]), observed=True).deficit.median().round(3).to_dict())
    del d, Pw, Ws, Wd

In [ ]:
pd.set_option("display.width", 250)
for farm in ["penmanshiel", "kelmarsh"]:
    d = load_feat(farm)
    d.loc[d["mask_m1"].fillna(True).astype(bool), ["Power (kW)", "Wind speed (m/s)", "Wind direction (°)"]] = np.nan
    pv = lambda c: d.pivot(index=TS, columns="turbine", values=c)
    Pw, Ws, Wd = pv("Power (kW)"), pv("Wind speed (m/s)"), pv("Wind direction (°)")
    n = Pw.shape[1]
    wsm = Ws.median(axis=1).where(Ws.notna().sum(axis=1) >= int(0.8 * n))
    wdir = np.rad2deg(np.arctan2(np.sin(np.deg2rad(Wd)).median(axis=1),
                                 np.cos(np.deg2rad(Wd)).median(axis=1))) % 360
    ok = wsm.between(6, 9) & wdir.notna() & (Pw.notna().sum(axis=1) >= int(0.8 * n))
    rel = Pw.div(Pw.mean(axis=1), axis=0)[ok]
    sec = (wdir[ok] // 20 * 20).astype(int)
    tab = rel.groupby(sec).median().T
    share = (sec.value_counts(normalize=True).sort_index() * 100).round(1)
    print(f"\n=== {farm}: median P_turbine / farm-mean P, by 20-degree wind sector (ws 6-9, {int(ok.sum()):,} timestamps)")
    print(tab.round(2).to_string())
    print("share of timestamps per sector (%):", share.to_dict())
    del d, Pw, Ws, Wd

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor as HGB
from sklearn.metrics import r2_score

def circ(a): return (a + 180) % 360 - 180
STATIC = {"penmanshiel": "Penmanshiel_WT_static.csv", "kelmarsh": "Kelmarsh_WT_static.csv"}

def layout(farm):
    s = pd.read_csv(f"/kaggle/working/hf/{farm}/{STATIC[farm]}").dropna(subset=["Latitude"])
    s["turbine"] = s["Alternative Title"].astype(str).str.findall(r"\d+").str[-1].astype(int)
    s = s.set_index("turbine")
    lat0, lon0 = s.Latitude.mean(), s.Longitude.mean()
    s["x"] = (s.Longitude - lon0) * 111320 * np.cos(np.deg2rad(lat0))
    s["y"] = (s.Latitude - lat0) * 110540
    s["elev"] = s["Elevation (m)"]
    return s, float(s["Rotor Diameter (m)"].iloc[0])

def wake_feats(turb, wdir, s, D):
    ang, dist, n_up = np.full(len(turb), 180.0), np.full(len(turb), 10.0), np.zeros(len(turb))
    for i in s.index:
        m = turb == i
        nb = [j for j in s.index if j != i and np.hypot(s.x[j] - s.x[i], s.y[j] - s.y[i]) / D <= 8]
        if not m.any() or not nb: continue
        b = np.array([np.degrees(np.arctan2(s.x[j] - s.x[i], s.y[j] - s.y[i])) % 360 for j in nb])
        dd = np.array([np.hypot(s.x[j] - s.x[i], s.y[j] - s.y[i]) / D for j in nb])
        off = np.abs(circ(wdir[m][:, None] - b[None, :]))      # 0 = a neighbour sits straight upwind
        k = off.argmin(axis=1)
        ang[m], dist[m], n_up[m] = off[np.arange(len(k)), k], dd[k], (off <= 20).sum(axis=1)
    return ang, dist, n_up

B = ["ws", "sin_wd", "cos_wd", "vane", "abs_vane", "temp", "TI"]
F = ["ws_farm", "sin_wdf", "cos_wdf", "temp"]
POS, WAKE = ["x", "y", "elev"], ["up_ang", "up_dist", "n_up"]
SETTINGS = {"own ws": ("ws", {"B": B, "B+pos": B + POS, "B+pos+wake": B + POS + WAKE}),
            "farm ws": ("ws_farm", {"F": F, "F+pos": F + POS, "F+pos+wake": F + POS + WAKE})}

for farm in ["penmanshiel", "kelmarsh"]:
    s, D = layout(farm)
    d = load_feat(farm)
    g = d[~d["mask_m1"].fillna(True).astype(bool) & (d[TS] >= "2018-04-01")]
    pv = g.pivot(index=TS, columns="turbine", values="Wind speed (m/s)")
    pdr = g.pivot(index=TS, columns="turbine", values="Wind direction (°)")
    wdir = pd.Series(np.rad2deg(np.arctan2(np.sin(np.deg2rad(pdr)).median(axis=1),
                                           np.cos(np.deg2rad(pdr)).median(axis=1))) % 360, name="wdir")
    wdir.index.name = TS
    loo = {}
    for t in pv.columns:                      # free-stream proxy: median of the OTHER turbines
        o = pv.drop(columns=t)
        loo[t] = o.median(axis=1).where(o.notna().sum(axis=1) >= 4)
    loo = pd.DataFrame(loo); loo.index.name = TS
    st = loo.reset_index().melt(id_vars=TS, var_name="turbine", value_name="ws_farm")
    st["turbine"] = st["turbine"].astype(int)

    c = g[g["Wind speed (m/s)"].between(3, 20)]
    a, b2 = c[c.year <= 2021], c[c.year >= 2022]
    c = pd.concat([a.sample(min(400_000, len(a)), random_state=0), b2.sample(min(200_000, len(b2)), random_state=1)])
    c = (c.merge(st, on=[TS, "turbine"], how="left").merge(wdir.reset_index(), on=TS, how="left")
           .dropna(subset=["ws_farm", "wdir"]).reset_index(drop=True))

    X = feats(c)
    X["P"], X["year"], X["ws_farm"] = c["Power (kW)"].values, c["year"].values, c["ws_farm"].values
    X["sin_wdf"], X["cos_wdf"] = np.sin(np.deg2rad(c["wdir"].values)), np.cos(np.deg2rad(c["wdir"].values))
    for k in ["x", "y", "elev"]:
        X[k] = c["turbine"].map(s[k]).values
    X["up_ang"], X["up_dist"], X["n_up"] = wake_feats(c["turbine"].values, c["wdir"].values, s, D)
    X = X.dropna()
    tr, te = X[X.year <= 2021], X[X.year >= 2022]
    print(f"\n=== {farm}: train {len(tr):,} | test {len(te):,}")

    r2, diag = {}, {}
    mk = te["ws_farm"].between(5, 10) & (te["up_dist"] <= 6)
    bk = pd.cut(te.loc[mk, "up_ang"], [0, 10, 20, 40, 180], include_lowest=True)
    for setting, (wc, sets) in SETTINGS.items():
        cv = tr.groupby(np.floor(tr[wc] * 4) / 4 + 0.125)["P"].median()
        r_tr = tr["P"] - np.interp(tr[wc], cv.index, cv.values)
        r_te = te["P"] - np.interp(te[wc], cv.index, cv.values)
        for name, cols in sets.items():
            m = HGB(max_iter=150, learning_rate=0.1, random_state=0).fit(tr[cols], r_tr)
            r2[(setting, name)] = r2_score(r_te, m.predict(te[cols]))
        diag[setting] = r_te[mk].groupby(bk, observed=True).mean()
    diag = pd.DataFrame(diag); diag["n"] = bk.value_counts(sort=False)
    print("\n-- held-out R2 of residual r = P - binned curve(ws)")
    print(pd.Series(r2).round(3).to_string())
    print("\n-- mean residual (kW) by angle to nearest upwind neighbour (<=6 D), farm ws 5-10 m/s")
    print(diag.round(1).to_string())
    del d, g, X, tr, te, c

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor as HGB
from sklearn.metrics import r2_score

def circ(a): return (a + 180) % 360 - 180
STATIC = {"penmanshiel": "Penmanshiel_WT_static.csv", "kelmarsh": "Kelmarsh_WT_static.csv"}

def layout(farm):
    s = pd.read_csv(f"/kaggle/working/hf/{farm}/{STATIC[farm]}").dropna(subset=["Latitude"])
    s["turbine"] = s["Alternative Title"].astype(str).str.findall(r"\d+").str[-1].astype(int)
    s = s.set_index("turbine")
    lat0, lon0 = s.Latitude.mean(), s.Longitude.mean()
    s["x"] = (s.Longitude - lon0) * 111320 * np.cos(np.deg2rad(lat0))
    s["y"] = (s.Latitude - lat0) * 110540
    s["elev"] = s["Elevation (m)"]
    return s, float(s["Rotor Diameter (m)"].iloc[0])

def wake_feats(turb, wdir, s, D):
    ang, dist, n_up = np.full(len(turb), 180.0), np.full(len(turb), 10.0), np.zeros(len(turb))
    for i in s.index:
        m = turb == i
        nb = [j for j in s.index if j != i and np.hypot(s.x[j] - s.x[i], s.y[j] - s.y[i]) / D <= 8]
        if not m.any() or not nb: continue
        b = np.array([np.degrees(np.arctan2(s.x[j] - s.x[i], s.y[j] - s.y[i])) % 360 for j in nb])
        dd = np.array([np.hypot(s.x[j] - s.x[i], s.y[j] - s.y[i]) / D for j in nb])
        off = np.abs(circ(wdir[m][:, None] - b[None, :]))      # 0 = a neighbour sits straight upwind
        k = off.argmin(axis=1)
        ang[m], dist[m], n_up[m] = off[np.arange(len(k)), k], dd[k], (off <= 20).sum(axis=1)
    return ang, dist, n_up

B = ["ws", "sin_wd", "cos_wd", "vane", "abs_vane", "temp", "TI"]
F = ["ws_farm", "sin_wdf", "cos_wdf", "temp"]
POS, WAKE = ["x", "y", "elev"], ["up_ang", "up_dist", "n_up"]
SETTINGS = {"own ws": ("ws", {"B": B, "B+pos": B + POS, "B+pos+wake": B + POS + WAKE}),
            "farm ws": ("ws_farm", {"F": F, "F+pos": F + POS, "F+pos+wake": F + POS + WAKE})}

for farm in ["penmanshiel", "kelmarsh"]:
    s, D = layout(farm)
    d = load_feat(farm)
    g = d[~d["mask_m1"].fillna(True).astype(bool) & (d[TS] >= "2018-04-01")]
    pv = g.pivot(index=TS, columns="turbine", values="Wind speed (m/s)")
    pdr = g.pivot(index=TS, columns="turbine", values="Wind direction (°)")
    wdir = pd.Series(np.rad2deg(np.arctan2(np.sin(np.deg2rad(pdr)).median(axis=1),
                                           np.cos(np.deg2rad(pdr)).median(axis=1))) % 360, name="wdir")
    wdir.index.name = TS
    loo = {}
    for t in pv.columns:                      # free-stream proxy: median of the OTHER turbines
        o = pv.drop(columns=t)
        loo[t] = o.median(axis=1).where(o.notna().sum(axis=1) >= 4)
    loo = pd.DataFrame(loo); loo.index.name = TS
    st = loo.reset_index().melt(id_vars=TS, var_name="turbine", value_name="ws_farm")
    st["turbine"] = st["turbine"].astype(int)

    c = g[g["Wind speed (m/s)"].between(3, 20)]
    a, b2 = c[c.year <= 2021], c[c.year >= 2022]
    c = pd.concat([a.sample(min(400_000, len(a)), random_state=0), b2.sample(min(200_000, len(b2)), random_state=1)])
    c = (c.merge(st, on=[TS, "turbine"], how="left").merge(wdir.reset_index(), on=TS, how="left")
           .dropna(subset=["ws_farm", "wdir"]).reset_index(drop=True))

    X = feats(c)
    X["P"], X["year"], X["ws_farm"] = c["Power (kW)"].values, c["year"].values, c["ws_farm"].values
    X["sin_wdf"], X["cos_wdf"] = np.sin(np.deg2rad(c["wdir"].values)), np.cos(np.deg2rad(c["wdir"].values))
    for k in ["x", "y", "elev"]:
        X[k] = c["turbine"].map(s[k]).values
    X["up_ang"], X["up_dist"], X["n_up"] = wake_feats(c["turbine"].values, c["wdir"].values, s, D)
    X = X.dropna()
    tr, te = X[X.year <= 2021], X[X.year >= 2022]
    print(f"\n=== {farm}: train {len(tr):,} | test {len(te):,}")

    r2, diag = {}, {}
    mk = te["ws_farm"].between(5, 10) & (te["up_dist"] <= 6)
    bk = pd.cut(te.loc[mk, "up_ang"], [0, 10, 20, 40, 180], include_lowest=True)
    for setting, (wc, sets) in SETTINGS.items():
        cv = tr.groupby(np.floor(tr[wc] * 4) / 4 + 0.125)["P"].median()
        r_tr = tr["P"] - np.interp(tr[wc], cv.index, cv.values)
        r_te = te["P"] - np.interp(te[wc], cv.index, cv.values)
        for name, cols in sets.items():
            m = HGB(max_iter=150, learning_rate=0.1, random_state=0).fit(tr[cols], r_tr)
            r2[(setting, name)] = r2_score(r_te, m.predict(te[cols]))
        diag[setting] = r_te[mk].groupby(bk, observed=True).mean()
    diag = pd.DataFrame(diag); diag["n"] = bk.value_counts(sort=False)
    print("\n-- held-out R2 of residual r = P - binned curve(ws)")
    print(pd.Series(r2).round(3).to_string())
    print("\n-- mean residual (kW) by angle to nearest upwind neighbour (<=6 D), farm ws 5-10 m/s")
    print(diag.round(1).to_string())
    del d, g, X, tr, te, c

In [ ]:
import matplotlib.pyplot as plt
os.makedirs("/kaggle/working/figs", exist_ok=True)

for farm in ["penmanshiel", "kelmarsh"]:
    d = load_feat(farm)
    d["flag"] = d["mask_m1"].fillna(True).astype(bool)
    v = d[d["Wind speed (m/s)"].notna() & d["Power (kW)"].notna()]
    v = v.sample(min(600_000, len(v)), random_state=0)

    fig, ax = plt.subplots(1, 3, figsize=(18, 5), sharex=True, sharey=True)
    for a, (title, sub) in zip(ax, [("all rows", v), ("clean (kept)", v[~v.flag]), ("flagged (removed)", v[v.flag])]):
        hb = a.hexbin(sub["Wind speed (m/s)"], sub["Power (kW)"], gridsize=90, bins="log",
                      extent=(0, 25, -100, 2200), mincnt=1, cmap="viridis")
        a.set_title(f"{farm}: {title} (n={len(sub):,})"); a.set_xlabel("wind speed (m/s)")
    ax[0].set_ylabel("power (kW)")
    plt.tight_layout(); plt.savefig(f"/kaggle/working/figs/{farm}_v1_powercurve.png", dpi=110); plt.show()

    m = d.groupby(["turbine", d[TS].dt.to_period("M")])["flag"].mean().unstack(0)
    fig, a = plt.subplots(figsize=(14, 4.5))
    im = a.imshow(m.T.values, aspect="auto", cmap="magma", vmin=0, vmax=1)
    a.set_yticks(range(m.shape[1])); a.set_yticklabels(m.columns)
    ticks = list(range(0, len(m), 6)); a.set_xticks(ticks); a.set_xticklabels([str(m.index[i]) for i in ticks], rotation=60, fontsize=7)
    a.set_title(f"{farm}: share of intervals excluded by mask_m1, turbine x month"); a.set_ylabel("turbine")
    plt.colorbar(im, ax=a, label="flagged share"); plt.tight_layout()
    plt.savefig(f"/kaggle/working/figs/{farm}_v1_maskheatmap.png", dpi=110); plt.show()
    del d, v

In [ ]:
flag_cols = ["st_stop","st_curtailment","st_communication","sp_idle","sp_binding",
             "loss_curt","core_nan","farm_cap","cap_kel_2016","commissioning"]
drift = {"penmanshiel": [(8,"2021-01-01","2023-01-01"), (2,"2021-01-01","2022-01-01")], "kelmarsh": []}
FP = lambda farm: f"/kaggle/working/parquet/flags/{farm}_flags.parquet"

for farm, rules in drift.items():
    f = pd.read_parquet(FP(farm))
    print(farm, "columns:", f.columns.tolist(), "| sensor_drift present:", "sensor_drift" in f.columns)
    f["sensor_drift"] = False
    for t, a, b in rules:
        f.loc[(f.turbine == t) & (f[TS] >= a) & (f[TS] < b), "sensor_drift"] = True
    f["mask_m1"] = f[flag_cols + ["sensor_drift"]].any(axis=1)
    f.to_parquet(FP(farm), index=False)
    print("  sensor_drift share:", round(f.sensor_drift.mean(), 4), "| clean:", round((~f.mask_m1).mean(), 4))

win = {"penmanshiel": [("2018-03-01","2018-08-01"), ("2020-07-01","2020-12-01")],
       "kelmarsh":    [("2018-08-01","2018-11-01"), ("2021-04-01","2021-08-01")]}
for farm, ws_ in win.items():
    f = pd.read_parquet(FP(farm))
    for a, b in ws_:
        x = f[(f[TS] >= a) & (f[TS] < b)]
        print(f"\n{farm} {a}..{b}: share flagged by each rule, per month (all turbines)")
        print(x[flag_cols + ["mask_m1"]].groupby(x[TS].dt.to_period("M")).mean().round(2).T.to_string())

for farm in ["penmanshiel", "kelmarsh"]:
    d = load_feat(farm).drop(columns="mask_m1").merge(pd.read_parquet(FP(farm)), on=[TS, "turbine"])
    v = d[d["Wind speed (m/s)"].between(5, 9) & d["Power (kW)"].notna()].copy()
    v["bin"] = np.floor(v["Wind speed (m/s)"] * 2) / 2
    cv = v[~v.mask_m1].groupby("bin")["Power (kW)"].median()
    v["on_curve"] = (v["Power (kW)"] / v["bin"].map(cv) - 1).abs() < 0.15
    rows = {c: dict(n_flagged=int(v[c].sum()), on_curve_share=round(v.loc[v[c], "on_curve"].mean(), 2))
            for c in flag_cols + ["sensor_drift"] if v[c].any()}
    print(f"\n=== {farm}: for rows removed by each rule (ws 5-9), share still within 15% of the clean curve")
    print(pd.DataFrame(rows).T.to_string())
    print("streak check, clean rows ws 8-13 with P in 550-650 kW:",
          int(((~v.mask_m1) & v["Power (kW)"].between(550, 650)).sum()),
          "of", int((~v.mask_m1).sum()), "(ws 5-9 only)")
    del d, v

In [ ]:
from huggingface_hub import hf_hub_download
#login(UserSecretsClient().get_secret("HF_TOKEN"))

api.upload_folder(folder_path="/kaggle/working/parquet/flags", path_in_repo="parquet/flags",
                  repo_id=REPO, repo_type="dataset", commit_message="Re-add sensor_drift flag")
chk = hf_hub_download(REPO, "parquet/flags/penmanshiel_flags.parquet", repo_type="dataset",
                      local_dir="/kaggle/working/verify", force_download=True)
print("HF copy has sensor_drift:", "sensor_drift" in pd.read_parquet(chk).columns)

flag_cols = ["st_stop","st_curtailment","st_communication","sp_idle","sp_binding",
             "loss_curt","core_nan","farm_cap","cap_kel_2016","commissioning"]
FP = lambda farm: f"/kaggle/working/parquet/flags/{farm}_flags.parquet"
LEVELS = [200, 600, 1000, 1200, 1680]

for farm in ["penmanshiel", "kelmarsh"]:
    d = load_feat(farm).drop(columns="mask_m1").merge(pd.read_parquet(FP(farm)), on=[TS, "turbine"])
    other = [c for c in flag_cols if c != "st_communication"] + ["sensor_drift"]
    d["comm_only"] = d["st_communication"] & ~d[other].any(axis=1)
    d["clean"] = ~d.mask_m1
    ws, P = d["Wind speed (m/s)"], d["Power (kW)"]

    v = d[ws.between(5, 9) & P.notna()].copy()
    v["bin"] = np.floor(v["Wind speed (m/s)"] * 2) / 2
    cv = v[v.clean].groupby("bin")["Power (kW)"].median()
    v["ratio"] = v["Power (kW)"] / v["bin"].map(cv)
    q = pd.DataFrame({"clean": v[v.clean].ratio.quantile([.05, .25, .5, .75, .95]),
                      "comm_only": v[v.comm_only].ratio.quantile([.05, .25, .5, .75, .95])}).round(3)
    hi = d[ws >= 13]
    tail = {k: round(((g["Power (kW)"] < 0.88 * rated).mean()), 4) for k, g in
            [("clean", hi[hi.clean]), ("comm_only", hi[hi.comm_only])]}
    print(f"\n=== {farm}: comm_only rows {int(d.comm_only.sum()):,} ({d.comm_only.mean():.2%} of all)")
    print("power / clean curve at 5-9 m/s:"); print(q.T.to_string())
    print("tail rate at ws>=13 (P<0.88*rated):", tail, "| n comm_only at ws>=13:", len(hi[hi.comm_only]))

    h = d[d.clean & (ws >= 10) & P.notna()]
    cnt = {L: int(h["Power (kW)"].between(L - 25, L + 25).sum()) for L in LEVELS}
    print(f"clean rows at ws>=10 ({len(h):,}), count within +-25 kW of level:", cnt)
    for L in LEVELS:
        s = h[h["Power (kW)"].between(L - 25, L + 25)]
        if len(s) > 200:
            print(f"  level {L}: by year", s.groupby("year").size().to_dict(), "| top turbines", s.groupby("turbine").size().nlargest(3).to_dict())
    del d, v, h

# Pull from HF the merged dataset

In [ ]:
!pip -q install huggingface_hub pyarrow
import os, glob
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from huggingface_hub import HfApi, login, snapshot_download
from kaggle_secrets import UserSecretsClient

login(UserSecretsClient().get_secret("HF_TOKEN"))
api = HfApi()
REPO = "B4BUK4/MPVI_project"
TS = "Date and time"
rated = 2050

snapshot_download(REPO, repo_type="dataset", allow_patterns=["parquet/*"], local_dir="/kaggle/working")

def load_feat(farm):
    parts = []
    for f in sorted(glob.glob(f"/kaggle/working/parquet/{farm}/*.parquet")):
        x = pd.read_parquet(f)
        x["turbine"] = int(os.path.basename(f).split("_")[1]); parts.append(x)
    d = (pd.concat(parts).sort_values(["turbine", TS])
           .drop_duplicates(["turbine", TS], keep="last").reset_index(drop=True))
    fl = pd.read_parquet(f"/kaggle/working/parquet/flags/{farm}_flags.parquet",
                         columns=[TS, "turbine", "mask_m1"]).drop_duplicates(["turbine", TS])
    d = d.merge(fl, on=[TS, "turbine"], how="left")
    d["year"] = d[TS].dt.year
    return d

for d_ in ["kelmarsh", "penmanshiel", "flags"]:
    print(d_, len(glob.glob(f"/kaggle/working/parquet/{d_}/*.parquet")), "files")
print(open("/kaggle/working/parquet/flags/MASK_SPEC.txt").read()[-400:])

In [ ]:
flag_cols = ["st_stop","st_curtailment","sp_idle","sp_binding","loss_curt","core_nan",
             "farm_cap","cap_kel_2016","commissioning"]          # st_communication deliberately excluded
FP = lambda farm: f"/kaggle/working/parquet/flags/{farm}_flags.parquet"

for farm in ["penmanshiel", "kelmarsh"]:
    f = pd.read_parquet(FP(farm))
    f["mask_m1"] = f[flag_cols + ["sensor_drift"]].any(axis=1)
    f.to_parquet(FP(farm), index=False)
    print(farm, "clean share:", round((~f.mask_m1).mean(), 4), "| columns:", len(f.columns))

with open("/kaggle/working/parquet/flags/MASK_SPEC.txt", "a") as fh:
    fh.write("st_communication removed from mask_m1 (2026-11-07): communication-only rows match clean rows "
             "(power/curve median 0.995-1.007 at 5-9 m/s). Column kept for analysis.\n")
api.upload_folder(folder_path="/kaggle/working/parquet/flags", path_in_repo="parquet/flags",
                  repo_id=REPO, repo_type="dataset", commit_message="Drop st_communication from mask_m1")
from huggingface_hub import hf_hub_download
for farm in ["penmanshiel", "kelmarsh"]:
    p = hf_hub_download(REPO, f"parquet/flags/{farm}_flags.parquet", repo_type="dataset",
                        local_dir="/kaggle/working/verify", force_download=True)
    g = pd.read_parquet(p)
    print("HF readback", farm, "| sensor_drift:", "sensor_drift" in g.columns,
          "| clean:", round((~g.mask_m1).mean(), 4))

LEVELS = [600, 1000, 1200, 1400, 1680]
for farm in ["penmanshiel", "kelmarsh"]:
    d = load_feat(farm)
    h = d[(~d.mask_m1.fillna(True).astype(bool)) & (d["Wind speed (m/s)"] >= 13) & d["Power (kW)"].notna()]
    print(f"\n=== {farm}: clean rows at ws>=13: {len(h):,} | share with P<1800: {(h['Power (kW)']<1800).mean():.4f}")
    for L in LEVELS:
        s = h[h["Power (kW)"].between(L - 25, L + 25)]
        print(f"  level {L}: {len(s)} rows", ("| by year " + str(s.groupby('year').size().to_dict())) if len(s) > 100 else "")
    del d, h

In [ ]:
import os, sys, importlib
os.makedirs("/kaggle/working/code", exist_ok=True)
os.makedirs("/kaggle/working/figs", exist_ok=True)

COMMON = r'''
import os, glob
import numpy as np, pandas as pd

W = "/kaggle/working"
REPO = "B4BUK4/MPVI_project"
TS = "Date and time"
rated = 2050
STATIC = {"penmanshiel": "Penmanshiel_WT_static.csv", "kelmarsh": "Kelmarsh_WT_static.csv"}
# mask_m1 = any of these flags EXCEPT st_communication (kept as an analysis column only)
FLAG_COLS = ["st_stop", "st_curtailment", "st_communication", "sp_idle", "sp_binding", "loss_curt",
             "core_nan", "farm_cap", "cap_kel_2016", "commissioning", "sensor_drift"]
START = {"penmanshiel": "2016-01-01", "kelmarsh": "2018-01-01"}   # model-1 train/eval start

def circ(a):
    return (a + 180) % 360 - 180

def ensure_static():
    from huggingface_hub import hf_hub_download
    for farm, fn in STATIC.items():
        if not os.path.exists(f"{W}/hf/{farm}/{fn}"):
            hf_hub_download(REPO, f"{farm}/{fn}", repo_type="dataset", local_dir=f"{W}/hf")

def load_feat(farm, all_flags=False):
    parts = []
    for f in sorted(glob.glob(f"{W}/parquet/{farm}/*.parquet")):
        x = pd.read_parquet(f)
        x["turbine"] = int(os.path.basename(f).split("_")[1])
        parts.append(x)
    d = (pd.concat(parts).sort_values(["turbine", TS])
           .drop_duplicates(["turbine", TS], keep="last").reset_index(drop=True))
    fl = pd.read_parquet(f"{W}/parquet/flags/{farm}_flags.parquet").drop_duplicates(["turbine", TS])
    if not all_flags:
        fl = fl[[TS, "turbine", "mask_m1"]]
    d = d.merge(fl, on=[TS, "turbine"], how="left")
    d["mask_m1"] = d["mask_m1"].fillna(True).astype(bool)
    d["year"] = d[TS].dt.year
    return d

def clean_rows(d, farm, start=None):
    start = start or START[farm]
    return d[(~d["mask_m1"]) & (d[TS] >= start)]

def layout(farm):
    ensure_static()
    s = pd.read_csv(f"{W}/hf/{farm}/{STATIC[farm]}").dropna(subset=["Latitude"])
    s["turbine"] = s["Alternative Title"].astype(str).str.findall(r"\d+").str[-1].astype(int)
    s = s.set_index("turbine")
    lat0, lon0 = s.Latitude.mean(), s.Longitude.mean()
    s["x"] = (s.Longitude - lon0) * 111320 * np.cos(np.deg2rad(lat0))
    s["y"] = (s.Latitude - lat0) * 110540
    s["elev"] = s["Elevation (m)"]
    return s, float(s["Rotor Diameter (m)"].iloc[0])

def feats(c):
    wd = np.deg2rad(c["Wind direction (°)"])
    X = pd.DataFrame({
        "ws": c["Wind speed (m/s)"],
        "sin_wd": np.sin(wd), "cos_wd": np.cos(wd),
        "vane": c["Vane position 1+2 (°)"].clip(-45, 45),
        "temp": c["Nacelle ambient temperature (°C)"],
        "TI": c["Wind speed, Standard deviation (m/s)"] / c["Wind speed (m/s)"].where(c["Wind speed (m/s)"] >= 1),
        "turbine": c["turbine"].astype(float)}, index=c.index)
    X["abs_vane"] = X["vane"].abs()
    return X

def wake_feats(turb, wdir, s, D):
    ang, dist, n_up = np.full(len(turb), 180.0), np.full(len(turb), 10.0), np.zeros(len(turb))
    for i in s.index:
        m = turb == i
        nb = [j for j in s.index if j != i and np.hypot(s.x[j] - s.x[i], s.y[j] - s.y[i]) / D <= 8]
        if not m.any() or not nb:
            continue
        b = np.array([np.degrees(np.arctan2(s.x[j] - s.x[i], s.y[j] - s.y[i])) % 360 for j in nb])
        dd = np.array([np.hypot(s.x[j] - s.x[i], s.y[j] - s.y[i]) / D for j in nb])
        off = np.abs(circ(wdir[m][:, None] - b[None, :]))
        k = off.argmin(axis=1)
        ang[m], dist[m], n_up[m] = off[np.arange(len(k)), k], dd[k], (off <= 20).sum(axis=1)
    return ang, dist, n_up
'''
with open("/kaggle/working/code/common.py", "w", encoding="utf-8") as fh:
    fh.write(COMMON)

sys.path.insert(0, "/kaggle/working/code")
import common; importlib.reload(common)
from common import *
import matplotlib.pyplot as plt
ensure_static()
print("helpers ready:", [n for n in ["load_feat", "clean_rows", "layout", "feats", "wake_feats"] if n in globals()])

In [ ]:
for farm in ["penmanshiel", "kelmarsh"]:
    d = load_feat(farm)
    v = d[(~d.mask_m1) & d["Wind speed (m/s)"].between(5, 9) & d["Power (kW)"].notna()].copy()
    v["bin"] = np.floor(v["Wind speed (m/s)"] * 2) / 2
    v["res"] = v["Power (kW)"] / v.groupby("bin")["Power (kW)"].transform("median")
    v["month"] = v[TS].dt.to_period("M")
    g = v.groupby(["month", "turbine"]).res.agg(["median", "size"]).reset_index()
    g.loc[g["size"] < 150, "median"] = np.nan
    t = g.pivot(index="month", columns="turbine", values="median")
    x = t.index.to_timestamp()

    fig, ax = plt.subplots(2, 1, figsize=(15, 8), sharex=True)
    for c in t.columns:
        ax[0].plot(x, t[c], lw=1, label=f"T{c}")
    ax[0].plot(x, t.median(axis=1), "k", lw=2.5, label="farm median")
    ax[0].set_ylabel("power / farm curve (5-9 m/s)"); ax[0].legend(ncol=8, fontsize=7)
    ax[0].set_title(f"{farm}: monthly power ratio per turbine (clean rows)")
    rel = t.div(t.median(axis=1), axis=0)
    for c in rel.columns:
        ax[1].plot(x, rel[c], lw=1, label=f"T{c}")
    ax[1].axhline(1, color="k", lw=1); ax[1].set_ylim(0.7, 1.3); ax[1].set_ylabel("relative to farm median")
    ax[1].legend(ncol=8, fontsize=7)
    if farm == "kelmarsh":
        for a in ax: a.axvline(pd.Timestamp("2018-01-01"), color="r", ls="--", lw=1)
    plt.tight_layout(); plt.savefig(f"/kaggle/working/figs/{farm}_v2_drift.png", dpi=110); plt.show()
    del d, v

In [ ]:
for farm in ["penmanshiel", "kelmarsh"]:
    d = load_feat(farm)
    c = clean_rows(d, farm, "2016-01-01")
    c = c[c["Wind speed (m/s)"].notna()]
    c = c.sample(min(800_000, len(c)), random_state=0)
    ws = c["Wind speed (m/s)"]
    yrs = sorted(c.year.unique()); cols = plt.cm.viridis(np.linspace(0, 1, len(yrs)))

    fig = plt.figure(figsize=(19, 10))
    ax = fig.add_subplot(2, 3, 1, projection="polar")
    wd = (c["Wind direction (°)"].dropna() % 360)
    h, _ = np.histogram(wd, bins=np.arange(0, 361, 15))
    ax.bar(np.deg2rad(np.arange(7.5, 360, 15)), h / h.sum() * 100, width=np.deg2rad(15), alpha=.85)
    ax.set_theta_zero_location("N"); ax.set_theta_direction(-1); ax.set_title("wind direction frequency (%)")

    ax = fig.add_subplot(2, 3, 2)
    for y, col in zip(yrs, cols):
        h, e = np.histogram(ws[c.year == y], bins=np.arange(0, 25.5, 0.5), density=True)
        ax.plot(e[:-1] + .25, h, color=col, label=str(y))
    ax.set_title("wind speed density by year"); ax.set_xlabel("m/s"); ax.legend(ncol=3, fontsize=7)

    ax = fig.add_subplot(2, 3, 3)
    b = c[c["Wind speed (m/s)"].between(4, 12) & c["Wind speed, Standard deviation (m/s)"].notna()]
    ti = b["Wind speed, Standard deviation (m/s)"] / b["Wind speed (m/s)"]
    for y, col in zip(yrs, cols):
        if (b.year == y).sum() > 5000:
            h, e = np.histogram(ti[b.year == y], bins=np.arange(0, 0.5, 0.01), density=True)
            ax.plot(e[:-1] + .005, h, color=col, label=str(y))
    ax.set_title("turbulence intensity (4-12 m/s) by year"); ax.set_xlabel("std/mean"); ax.legend(ncol=3, fontsize=7)

    ax = fig.add_subplot(2, 3, 4)
    tm = c["Nacelle ambient temperature (°C)"]
    ax.boxplot([tm[c[TS].dt.month == m].dropna() for m in range(1, 13)], showfliers=False)
    ax.set_xticklabels(range(1, 13)); ax.set_title("nacelle temperature by month (C)")

    ax = fig.add_subplot(2, 3, 5)
    hm = ws.groupby([c[TS].dt.month, c[TS].dt.hour]).mean().unstack()
    im = ax.imshow(hm.values, aspect="auto", cmap="viridis")
    ax.set_yticks(range(12)); ax.set_yticklabels(range(1, 13)); ax.set_xlabel("hour (UTC)")
    ax.set_title("mean wind speed: month x hour"); plt.colorbar(im, ax=ax)

    ax = fig.add_subplot(2, 3, 6)
    sec = (c["Wind direction (°)"] % 360 // 15 * 15)
    g = ws.groupby(sec).agg(["mean", lambda s: s.quantile(.9)])
    ax.plot(g.index + 7.5, g.iloc[:, 0], "o-", label="mean"); ax.plot(g.index + 7.5, g.iloc[:, 1], "o-", label="p90")
    ax.set_title("wind speed by direction sector"); ax.set_xlabel("deg"); ax.legend()

    fig.suptitle(f"{farm}: distributions (clean rows)", fontsize=14)
    plt.tight_layout(); plt.savefig(f"/kaggle/working/figs/{farm}_v3_distributions.png", dpi=105); plt.show()
    del d, c

In [ ]:
def binned(ax, x, y, nb=15, label=None):
    m = x.notna() & y.notna(); x, y = x[m], y[m]
    b = pd.qcut(x, nb, duplicates="drop")
    g = pd.DataFrame({"x": x, "y": y}).groupby(b, observed=True)
    xm, ym, lo, hi = g.x.mean(), g.y.mean(), g.y.quantile(.25), g.y.quantile(.75)
    ax.plot(xm, ym, "o-", label=label); ax.fill_between(xm, lo, hi, alpha=.2)

for farm in ["penmanshiel", "kelmarsh"]:
    d = load_feat(farm)
    c = clean_rows(d, farm, "2018-04-01")
    X = feats(c); X["P"] = c["Power (kW)"]; X["turbine"] = c["turbine"]; X["year"] = c["year"]
    X = X[X.ws.between(3, 20) & X.P.notna()]
    tr, te = X[X.year <= 2021], X[X.year >= 2022]
    cv = tr.groupby(np.floor(tr.ws * 4) / 4 + 0.125)["P"].median()
    te = te.assign(r=te.P - np.interp(te.ws, cv.index.values, cv.values))
    te = te.sample(min(400_000, len(te)), random_state=0)

    fig, ax = plt.subplots(2, 3, figsize=(19, 10)); ax = ax.ravel()
    hb = ax[0].hexbin(te.ws, te.r, gridsize=70, bins="log", extent=(3, 20, -600, 600), mincnt=1, cmap="viridis")
    q = te.groupby(np.floor(te.ws * 2) / 2 + .25).r.quantile([.1, .5, .9]).unstack()
    for col, ls in zip(q.columns, [":", "-", ":"]): ax[0].plot(q.index, q[col], "r", ls=ls)
    ax[0].set_title("residual vs wind speed (red: p10/p50/p90)"); ax[0].set_xlabel("m/s"); ax[0].set_ylabel("kW")

    mm = te[te.ws.between(4, 12) & te.TI.notna()].copy()
    mm["wbin"] = np.floor(mm.ws).astype(int)
    mm["tir"] = mm.groupby("wbin").TI.rank(pct=True)
    mm["terc"] = pd.cut(mm.tir, [0, 1/3, 2/3, 1.0], labels=["low TI", "mid TI", "high TI"], include_lowest=True)
    for k, g in mm.groupby("terc", observed=True):
        s_ = g.groupby("wbin").r.std(); ax[1].plot(s_.index + .5, s_.values, "o-", label=k)
    ax[1].set_title("residual std vs wind speed, by TI tercile (within bin)"); ax[1].legend(); ax[1].set_xlabel("m/s")

    mid = te[te.ws.between(6, 10)]
    binned(ax[2], mid.TI, mid.r); ax[2].set_title("mean residual vs TI (6-10 m/s, band p25-p75)"); ax[2].set_xlabel("TI")
    binned(ax[3], mid.temp, mid.r); ax[3].set_title("mean residual vs temperature (6-10 m/s)"); ax[3].set_xlabel("C")
    binned(ax[4], mid.vane, mid.r, nb=20); ax[4].set_title("mean residual vs vane (6-10 m/s)"); ax[4].set_xlabel("deg")
    g = mid.groupby("turbine").r; m_, se = g.mean(), g.std() / np.sqrt(g.size())
    ax[5].bar(m_.index.astype(int).astype(str), m_.values, yerr=se.values)
    ax[5].axhline(0, color="k", lw=1); ax[5].set_title("mean residual by turbine (6-10 m/s)")
    for a in ax[2:5]: a.axhline(0, color="k", lw=.8)

    fig.suptitle(f"{farm}: residual structure, test years 2022+", fontsize=14)
    plt.tight_layout(); plt.savefig(f"/kaggle/working/figs/{farm}_v4_residuals.png", dpi=105); plt.show()
    del d, c, X, tr, te

In [ ]:
for farm in ["penmanshiel", "kelmarsh"]:
    s, D = layout(farm)
    fig, ax = plt.subplots(figsize=(8, 7))
    for i in s.index:
        for j in s.index:
            if i < j and np.hypot(s.x[i] - s.x[j], s.y[i] - s.y[j]) / D <= 5:
                ax.plot([s.x[i], s.x[j]], [s.y[i], s.y[j]], color="grey", lw=.6, zorder=1)
    sc = ax.scatter(s.x, s.y, c=s.elev, s=260, cmap="cividis", edgecolor="k", zorder=3)
    for t in s.index:
        ax.annotate(f"T{t}", (s.x[t], s.y[t]), xytext=(0, 14), textcoords="offset points", ha="center", fontsize=9)
    ax.set_aspect("equal"); plt.colorbar(sc, label="elevation (m)")
    ax.set_title(f"{farm}: layout (grey = pairs within 5 D, D={D:.0f} m)"); ax.set_xlabel("x (m)"); ax.set_ylabel("y (m)")
    plt.tight_layout(); plt.savefig(f"/kaggle/working/figs/{farm}_v5_layout.png", dpi=110); plt.show()

    d = load_feat(farm); d = d[d[TS] >= START[farm]]
    d.loc[d.mask_m1, ["Power (kW)", "Wind speed (m/s)", "Wind direction (°)"]] = np.nan
    pv = lambda c_: d.pivot(index=TS, columns="turbine", values=c_)
    Pw, Ws, Wd = pv("Power (kW)"), pv("Wind speed (m/s)"), pv("Wind direction (°)")
    n = Pw.shape[1]
    wsm = Ws.median(axis=1).where(Ws.notna().sum(axis=1) >= int(0.8 * n))
    wdir = np.rad2deg(np.arctan2(np.sin(np.deg2rad(Wd)).median(axis=1),
                                 np.cos(np.deg2rad(Wd)).median(axis=1))) % 360
    ok = wsm.between(6, 9) & wdir.notna() & (Pw.notna().sum(axis=1) >= int(0.8 * n))
    rel = Pw.div(Pw.mean(axis=1), axis=0)[ok]
    sec = (wdir[ok] // 10 * 10).astype(int)
    tab = rel.groupby(sec).median(); cnt = sec.value_counts()
    tab = tab.loc[cnt[cnt >= 150].index].sort_index()

    cols_n = 5; rows_n = int(np.ceil(n / cols_n))
    fig = plt.figure(figsize=(3.6 * cols_n, 3.6 * rows_n))
    for k, t in enumerate(tab.columns):
        ax = fig.add_subplot(rows_n, cols_n, k + 1, projection="polar")
        th = np.deg2rad(tab.index.values + 5)
        ax.plot(np.r_[th, th[:1]], np.r_[tab[t].values, tab[t].values[:1]], lw=1.3)
        ax.plot(np.linspace(0, 2 * np.pi, 100), np.ones(100), "k", lw=.6)
        ax.set_theta_zero_location("N"); ax.set_theta_direction(-1); ax.set_ylim(0, 1.7)
        ax.set_yticks([.5, 1, 1.5]); ax.tick_params(labelsize=6)
        ax.set_title(f"T{t} ({s.elev[t]:.0f} m)", fontsize=9)
    fig.suptitle(f"{farm}: median P_turbine / farm-mean P by wind direction (ws 6-9, clean)", fontsize=13)
    plt.tight_layout(); plt.savefig(f"/kaggle/working/figs/{farm}_v5_polar.png", dpi=100); plt.show()
    del d, Pw, Ws, Wd

In [ ]:
import sys; sys.path.insert(0, "/kaggle/working/code")
from common import *
from sklearn.ensemble import HistGradientBoostingRegressor as HGB
from sklearn.metrics import r2_score
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)
nod_cols = [c for c in FLAG_COLS if c not in ("st_communication", "sensor_drift")]

# ---- A) drift windows with sensor_drift switched off
for farm, tl in [("penmanshiel", [2, 5, 8, 10, 12, 14]), ("kelmarsh", [1, 2, 4, 5])]:
    d = load_feat(farm, all_flags=True)
    ok = ~d[nod_cols].fillna(False).astype(bool).any(axis=1)
    v = d[ok & d["Wind speed (m/s)"].between(5, 9) & d["Power (kW)"].notna()].copy()
    v["bin"] = np.floor(v["Wind speed (m/s)"] * 2) / 2
    v["res"] = v["Power (kW)"] / v.groupby("bin")["Power (kW)"].transform("median")
    v["q"] = v[TS].dt.to_period("Q")
    g = v.groupby(["q", "turbine"]).res.agg(["median", "size"]).reset_index()
    g.loc[g["size"] < 100, "median"] = np.nan
    t = g.pivot(index="q", columns="turbine", values="median")
    rel = t.div(t.median(axis=1), axis=0)
    print(f"\n[A] {farm}: quarterly power ratio relative to farm median (sensor_drift NOT applied)")
    print(rel[tl].round(3).to_string())
    del d, v

# ---- B) variance structure after removing a feature-based mean
B = ["ws", "sin_wd", "cos_wd", "vane", "abs_vane", "temp", "TI"]
for farm in ["penmanshiel", "kelmarsh"]:
    d = load_feat(farm)
    c = clean_rows(d, farm, "2018-04-01")
    X = feats(c); X["P"] = c["Power (kW)"]; X["year"] = c["year"]
    X = X[X.ws.between(3, 20)].dropna()
    tr, te = X[X.year <= 2021], X[X.year >= 2022]
    tr = tr.sample(min(400_000, len(tr)), random_state=0).reset_index(drop=True)
    te = te.sample(min(200_000, len(te)), random_state=1).reset_index(drop=True)

    oof = np.zeros(len(tr))
    for y in [2018, 2019, 2020, 2021]:
        m_ = (tr.year == y).values
        if m_.sum() == 0: continue
        mm = HGB(max_iter=150, random_state=0).fit(tr.loc[~m_, B], tr.loc[~m_, "P"])
        oof[m_] = tr.loc[m_, "P"].values - mm.predict(tr.loc[m_, B])
    mu = HGB(max_iter=150, random_state=0).fit(tr[B], tr["P"])
    e_te = te["P"].values - mu.predict(te[B])
    print(f"\n[B] {farm}: mean model test R2(P)={r2_score(te.P, te.P - e_te):.4f} | test RMSE={np.sqrt((e_te**2).mean()):.1f} kW"
          f" | OOF RMSE={np.sqrt((oof**2).mean()):.1f} kW")

    k = np.sqrt(np.pi / 2)
    def fit_sig(cols):
        m = HGB(max_iter=200, random_state=0).fit(tr[cols], np.abs(oof))
        return np.maximum(m.predict(te[cols]) * k, 3.0)
    sig = {"const": np.full(len(te), np.abs(oof).mean() * k), "ws only": fit_sig(["ws"]), "B feats": fit_sig(B)}
    rows = {}
    for n_, s in sig.items():
        z = np.abs(e_te) / s
        rows[n_] = dict(NLL=np.mean(np.log(s) + 0.5 * (e_te / s) ** 2) + 0.5 * np.log(2 * np.pi),
                        cover_1sig=(z < 1).mean(), cover_2sig=(z < 2).mean(),
                        R2_abs=r2_score(np.abs(e_te), s / k))
    print(pd.DataFrame(rows).T.round(4).to_string())
    print("   (Gaussian reference: coverage 0.683 / 0.954; lower NLL is better)")

    te["e"] = e_te
    mm_ = te[te.ws.between(4, 12) & te.TI.notna()].copy(); mm_["wb"] = np.floor(mm_.ws).astype(int)
    mm_["terc"] = pd.cut(mm_.groupby("wb").TI.rank(pct=True), [0, 1/3, 2/3, 1.0],
                         labels=["lowTI", "midTI", "highTI"], include_lowest=True)
    print("   std of mean-corrected residual by ws bin and TI tercile (kW)")
    print(mm_.pivot_table(index="terc", columns="wb", values="e", aggfunc="std", observed=True).round(1).to_string())
    tb = pd.qcut(te.temp, 4, labels=["coldest", "cold", "warm", "warmest"])
    print("   std by temperature quartile (6-10 m/s):",
          te[te.ws.between(6, 10)].groupby(tb[te.ws.between(6, 10)], observed=True).e.std().round(1).to_dict())
    del d, c, X, tr, te

In [ ]:
import sys; sys.path.insert(0, "/kaggle/working/code")
from common import *
from huggingface_hub import HfApi, hf_hub_download
api = HfApi()
nod = ["st_stop","st_curtailment","sp_idle","sp_binding","loss_curt","core_nan","farm_cap","cap_kel_2016","commissioning"]
drift = {"penmanshiel": [(2,"2021-01-01","2021-10-01"), (8,"2021-10-01","2022-07-01"), (14,"2023-01-01","2023-07-01")],
         "kelmarsh": []}
FP = lambda farm: f"{W}/parquet/flags/{farm}_flags.parquet"
for farm, rules in drift.items():
    f = pd.read_parquet(FP(farm))
    f["sensor_drift"] = False
    for t, a, b in rules:
        f.loc[(f.turbine == t) & (f[TS] >= a) & (f[TS] < b), "sensor_drift"] = True
    f["mask_m1"] = f[nod + ["sensor_drift"]].any(axis=1)
    f.to_parquet(FP(farm), index=False)
    print(farm, "sensor_drift share:", round(f.sensor_drift.mean(), 4), "| clean:", round((~f.mask_m1).mean(), 4))

with open(f"{W}/parquet/flags/MASK_SPEC.txt", "a") as fh:
    fh.write("sensor_drift REVISED (2026-11-07): Penmanshiel T2 2021Q1-Q3, T8 2021Q4-2022Q2, T14 2023Q1-Q2 (quarterly power ratio vs farm median, "
             "step/collapse episodes). Slow drifts (T8 2016-2020 +7%, T5, T12 level change mid-2022, Kelmarsh T1/T2/T4) not flagged.\n")

for folder in ["parquet/flags", "code"]:
    api.upload_folder(folder_path=f"{W}/{folder}", path_in_repo=folder, repo_id=REPO,
                      repo_type="dataset", commit_message=f"Revise sensor_drift, state ({folder})")
for farm in ["penmanshiel", "kelmarsh"]:
    p = hf_hub_download(REPO, f"parquet/flags/{farm}_flags.parquet", repo_type="dataset",
                        local_dir=f"{W}/verify", force_download=True)
    g = pd.read_parquet(p)
    print("HF readback", farm, "| sensor_drift share:", round(g.sensor_drift.mean(), 4), "| clean:", round((~g.mask_m1).mean(), 4))

In [ ]:
import os, sys, time
import numpy as np, pandas as pd
from huggingface_hub import HfApi, login, hf_hub_download
from kaggle_secrets import UserSecretsClient

# 1. Environment & Auth
login(UserSecretsClient().get_secret("HF_TOKEN"))
api = HfApi()
W = "/kaggle/working"
REPO = "B4BUK4/MPVI_project"
TS = "Date and time"

sys.path.insert(0, f"{W}/code")
from common import load_feat, clean_rows, layout

os.makedirs(f"{W}/parquet/train", exist_ok=True)
os.makedirs(f"{W}/verify_train", exist_ok=True)

# 2. Dataset Assembly Pipeline
def build_m1_dataset(farm: str) -> pd.DataFrame:
    t0 = time.time()
    s, _ = layout(farm)
    d = load_feat(farm)
    
    # Filter using frozen mask and verified operational starts
    # Penmanshiel: 2016-09-01 (post-commissioning); Kelmarsh: 2018-01-01 (column availability)
    start_dt = "2016-09-01" if farm == "penmanshiel" else "2018-01-01"
    c = clean_rows(d, farm, start=start_dt).copy()
    
    # Trigonometry & Feature Materialization
    wd_rad = np.deg2rad(c["Wind direction (°)"])
    month = c[TS].dt.month
    month_rad = 2 * np.pi * (month - 1) / 12.0
    vane_clip = c["Vane position 1+2 (°)"].clip(-45, 45)
    
    df = pd.DataFrame({
        TS: c[TS],
        "turbine": c["turbine"].astype(np.int8),
        "year": c[TS].dt.year.astype(np.int16),
        "month": month.astype(np.int8),
        "x": c["turbine"].map(s["x"]).astype(np.float32),
        "y": c["turbine"].map(s["y"]).astype(np.float32),
        "elev": c["turbine"].map(s["elev"]).astype(np.float32),
        "ws": c["Wind speed (m/s)"].astype(np.float32),
        "sin_wd": np.sin(wd_rad).astype(np.float32),
        "cos_wd": np.cos(wd_rad).astype(np.float32),
        "vane": vane_clip.astype(np.float32),
        "abs_vane": vane_clip.abs().astype(np.float32),
        "temp": c["Nacelle ambient temperature (°C)"].astype(np.float32),
        "TI": (c["Wind speed, Standard deviation (m/s)"] / c["Wind speed (m/s)"].where(c["Wind speed (m/s)"] >= 1.0)).astype(np.float32),
        "month_sin": np.sin(month_rad).astype(np.float32),
        "month_cos": np.cos(month_rad).astype(np.float32),
        "power_kw": c["Power (kW)"].astype(np.float32),
        "is_test": (c[TS].dt.year >= 2022).astype(bool)
    })
    
    # Drop rows missing essential features or target (e.g. ws < 1 where TI is NaN)
    core_cols = ["ws", "sin_wd", "cos_wd", "vane", "temp", "TI", "power_kw"]
    n_before = len(df)
    df = df.dropna(subset=core_cols).sort_values(["turbine", TS]).reset_index(drop=True)
    
    out_path = f"{W}/parquet/train/{farm}_m1_clean.parquet"
    df.to_parquet(out_path, index=False)
    
    mb = os.path.getsize(out_path) / 1e6
    print(f"[{farm}] {n_before:,} rows -> {len(df):,} valid rows ({n_before - len(df):,} dropped NaN) | {mb:.1f} MB in {time.time()-t0:.1f}s")
    return df

# 3. Process Both Farms
pen_df = build_m1_dataset("penmanshiel")
kel_df = build_m1_dataset("kelmarsh")

# 4. Upload to Hugging Face
print("\nUploading parquet/train folder to Hugging Face")
api.upload_folder(
    folder_path=f"{W}/parquet/train",
    path_in_repo="parquet/train",
    repo_id=REPO,
    repo_type="dataset",
    commit_message="Add consolidated M1 training tables"
)

# 5. Readback Verification
print("\n HF Readback Verification")
for farm in ["penmanshiel", "kelmarsh"]:
    p = hf_hub_download(
        REPO, f"parquet/train/{farm}_m1_clean.parquet",
        repo_type="dataset", local_dir=f"{W}/verify_train", force_download=True
    )
    chk = pd.read_parquet(p)
    n_train = (~chk["is_test"]).sum()
    n_test = chk["is_test"].sum()
    null_count = chk.isna().sum().sum()
    
    print(f"{farm.upper()}:")
    print(f"  Rows total: {len(chk):,} | Train (<=2021): {n_train:,} | Test (>=2022): {n_test:,}")
    print(f"  Date range: {chk[TS].min().date()} to {chk[TS].max().date()} | Turbines: {sorted(chk['turbine'].unique())}")
    print(f"  Total nulls: {null_count} | Memory footprint: {chk.memory_usage().sum() / 1e6:.1f} MB")

In [ ]:
import time
import numpy as np, pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor as HGB
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

W = "/kaggle/working"
B_FEATS = ["ws", "sin_wd", "cos_wd", "vane", "abs_vane", "temp", "TI"]
K_NORM = np.sqrt(np.pi / 2)  # Mean Absolute Deviation -> std for zero-mean Gaussian

def evaluate_baseline(farm: str, sample_train: int = 500_000):
    print(f"\n{'='*25} {farm.upper()} BASELINE {'='*25}")
    t0 = time.time()
    
    # 1. Load consolidated table
    path = f"{W}/parquet/train/{farm}_m1_clean.parquet"
    df = pd.read_parquet(path)
    
    # Evaluate on the operational power-curve wind regime (3 <= ws <= 20 m/s)
    df = df[df["ws"].between(3.0, 20.0)].reset_index(drop=True)
    
    tr = df[~df["is_test"]].copy()
    te = df[df["is_test"]].copy()
    
    if len(tr) > sample_train:
        tr_fit = tr.sample(sample_train, random_state=42).reset_index(drop=True)
    else:
        tr_fit = tr.reset_index(drop=True)
    
    print(f"Dataset: {len(df):,} rows (3-20 m/s) | Train: {len(tr):,} (sampled {len(tr_fit):,}) | Test: {len(te):,}")
    
    # 2. Benchmark Level 1: Simple Binned Power Curve
    bins = np.floor(tr_fit["ws"] * 4) / 4 + 0.125
    binned_curve = tr_fit.groupby(bins)["power_kw"].median()
    te_pred_binned = np.interp(te["ws"], binned_curve.index.values, binned_curve.values)
    rmse_binned = np.sqrt(mean_squared_error(te["power_kw"], te_pred_binned))
    r2_binned = r2_score(te["power_kw"], te_pred_binned)
    
    # 3. Benchmark Level 2: Mean Model (HistGradientBoosting on B_FEATS)
    print("Training Mean Model (HistGradientBoosting)...")
    mu_model = HGB(max_iter=150, learning_rate=0.1, random_state=42)
    
    # Out-of-fold (OOF) cross-validation on train set by year (to train variance head without leakage)
    train_years = sorted(tr_fit["year"].unique())
    oof_residuals = np.zeros(len(tr_fit))
    
    for y in train_years:
        mask_val = (tr_fit["year"] == y).values
        if mask_val.sum() == 0:
            continue
        m_cv = HGB(max_iter=100, learning_rate=0.1, random_state=42)
        m_cv.fit(tr_fit.loc[~mask_val, B_FEATS], tr_fit.loc[~mask_val, "power_kw"])
        oof_residuals[mask_val] = tr_fit.loc[mask_val, "power_kw"].values - m_cv.predict(tr_fit.loc[mask_val, B_FEATS])
    
    mu_model.fit(tr_fit[B_FEATS], tr_fit["power_kw"])
    te_pred_mu = mu_model.predict(te[B_FEATS])
    
    rmse_mu = np.sqrt(mean_squared_error(te["power_kw"], te_pred_mu))
    mae_mu = mean_absolute_error(te["power_kw"], te_pred_mu)
    r2_mu = r2_score(te["power_kw"], te_pred_mu)
    oof_rmse = np.sqrt(np.mean(oof_residuals**2))
    
    # 4. Benchmark Level 3: Uncertainty / Variance Models
    print("Training Variance Heads...")
    e_te = te["power_kw"].values - te_pred_mu
    abs_oof = np.abs(oof_residuals)
    
    # Sigma models: constant, wind-speed only, and full B-features
    sig_models = {
        "const": np.full(len(te), abs_oof.mean() * K_NORM),
        "ws_only": np.maximum(HGB(max_iter=150, random_state=42).fit(tr_fit[["ws"]], abs_oof).predict(te[["ws"]]) * K_NORM, 3.0),
        "B_feats": np.maximum(HGB(max_iter=150, random_state=42).fit(tr_fit[B_FEATS], abs_oof).predict(te[B_FEATS]) * K_NORM, 3.0),
    }
    
    metrics = []
    for sig_name, sigma in sig_models.items():
        z = np.abs(e_te) / sigma
        nll = np.mean(np.log(sigma) + 0.5 * (e_te / sigma) ** 2) + 0.5 * np.log(2 * np.pi)
        c1 = (z < 1.0).mean()
        c2 = (z < 2.0).mean()
        metrics.append({
            "Variance Head": sig_name,
            "NLL": round(nll, 4),
            "1-sigma (target 0.683)": round(c1, 4),
            "2-sigma (target 0.954)": round(c2, 4),
            "Mean Sigma (kW)": round(sigma.mean(), 1)
        })
    
    # Print Executive Report
    print(f"\nMean Model Performance (Target: Power kW)")
    print(f"  Binned Empirical Curve : RMSE = {rmse_binned:.1f} kW | R² = {r2_binned:.4f}")
    print(f"  HGB (B-features)       : RMSE = {rmse_mu:.1f} kW | MAE = {mae_mu:.1f} kW | R² = {r2_mu:.4f}")
    print(f"  OOF CV RMSE (Train)    : {oof_rmse:.1f} kW (generalization gap = {abs(rmse_mu - oof_rmse):.1f} kW)")
    
    print(f"\n Heteroscedastic Uncertainty Performance (Test Set)")
    print(pd.DataFrame(metrics).to_string(index=False))
    print(f"Completed in {time.time()-t0:.1f}s")
    
    return {"rmse": rmse_mu, "r2": r2_mu, "nll_b": metrics[-1]["NLL"]}

# Run for both farms
pen_res = evaluate_baseline("penmanshiel")
kel_res = evaluate_baseline("kelmarsh")

DATA LOADING

In [ ]:
!pip -q install huggingface_hub pyarrow scikit-learn

import os, sys, glob, time
import numpy as np, pandas as pd
from huggingface_hub import HfApi, login, snapshot_download, hf_hub_download
from kaggle_secrets import UserSecretsClient

# login(UserSecretsClient().get_secret("HF_TOKEN"))
api = HfApi()

REPO = "B4BUK4/MPVI_project"
W = "/kaggle/working"
TS = "Date and time"
rated = 2050

print("Downloading codebase, flags, layouts, and training tables from HF...")
t0 = time.time()

snapshot_download(
    REPO,
    repo_type="dataset",
    allow_patterns=["code/*", "parquet/flags/*", "parquet/train/*"],
    local_dir=W
)

for fn in ["penmanshiel_flags.parquet", "kelmarsh_flags.parquet", "MASK_SPEC.txt"]:
    hf_hub_download(REPO, f"parquet/flags/{fn}", repo_type="dataset", local_dir=W, force_download=True)

os.makedirs(f"{W}/figs", exist_ok=True)
os.makedirs(f"{W}/hf", exist_ok=True)

sys.path.insert(0, f"{W}/code")
from common import load_feat, clean_rows, layout, feats, wake_feats, ensure_static, START, FLAG_COLS
ensure_static()

print(f"Restore finished in {time.time() - t0:.1f}s.\n")

print("=" * 60)
print("ENVIRONMENT & DATA INTEGRITY VERIFICATION")
print("=" * 60)

for farm in ["penmanshiel", "kelmarsh"]:
    fl = pd.read_parquet(f"{W}/parquet/flags/{farm}_flags.parquet", columns=["mask_m1", "sensor_drift"])
    print(f"[{farm.upper()} FLAGS] Clean share: {(~fl.mask_m1).mean():.4f} (target 0.9186/0.9489) | Drift share: {fl.sensor_drift.mean():.4f}")

print("-" * 60)

for farm in ["penmanshiel", "kelmarsh"]:
    p = f"{W}/parquet/train/{farm}_m1_clean.parquet"
    if os.path.exists(p):
        df = pd.read_parquet(p, columns=[TS, "turbine", "ws", "power_kw", "is_test"])
        n_train = (~df["is_test"]).sum()
        n_test = df["is_test"].sum()
        nulls = df.isna().sum().sum()
        print(f"[{farm.upper()} M1 DATASET] Total: {len(df):,} | Train: {n_train:,} | Test: {n_test:,} | Nulls: {nulls}")
    else:
        print(f"[{farm.upper()}] Warning: training table not found locally!")


In [ ]:
import os, time
import numpy as np, pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# 1. Environment & Device Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")

W = "/kaggle/working"
TS = "Date and time"
rated = 2050.0

# Feature columns
INPUT_COLS = ["ws", "sin_wd", "cos_wd", "vane", "abs_vane", "temp", "TI", "x", "y", "elev"]
TARGET_COL = "power_kw"

# 2. Load Consolidated Training Table
df = pd.read_parquet(f"{W}/parquet/train/penmanshiel_m1_clean.parquet")
df = df[df["ws"].between(3.0, 20.0)].reset_index(drop=True)

tr_df = df[~df["is_test"]].copy()
te_df = df[df["is_test"]].copy()

# Sample 500k train rows to match the baseline training scale
tr_sample = tr_df.sample(min(500_000, len(tr_df)), random_state=42).reset_index(drop=True)

print(f"Penmanshiel: {len(tr_sample):,} train samples, {len(te_df):,} test samples")

# 3. Normalization (Fit scaler strictly on training data)
scaler_X = StandardScaler()
X_tr = scaler_X.fit_transform(tr_sample[INPUT_COLS]).astype(np.float32)
X_te = scaler_X.transform(te_df[INPUT_COLS]).astype(np.float32)

y_tr = tr_sample[TARGET_COL].values.astype(np.float32)
y_te = te_df[TARGET_COL].values.astype(np.float32)

# Unnormalized wind speed for the analytic prior (ws is at index 0)
ws_tr = tr_sample["ws"].values.astype(np.float32)
ws_te = te_df["ws"].values.astype(np.float32)

train_dataset = TensorDataset(torch.tensor(X_tr), torch.tensor(y_tr), torch.tensor(ws_tr))
test_dataset = TensorDataset(torch.tensor(X_te), torch.tensor(y_te), torch.tensor(ws_te))

train_loader = DataLoader(train_dataset, batch_size=2048, shuffle=True, drop_last=True)
test_loader = DataLoader(test_dataset, batch_size=4096, shuffle=False)

# 4. PINN Architecture Definition
class SmoothPowerPrior(nn.Module):
    """Analytic aerodynamic base power curve (Region I -> Region II cubic -> Region III plateau)"""
    def __init__(self, rated_power=2050.0):
        super().__init__()
        self.rated = rated_power
        # Trainable aerodynamic inflection parameters initialized near physical values
        self.u_mid = nn.Parameter(torch.tensor(8.5))      # Inflection wind speed
        self.u_scale = nn.Parameter(torch.tensor(1.8))    # Transition steepness
        self.cut_in = nn.Parameter(torch.tensor(3.0))     # Cut-in wind speed

    def forward(self, ws):
        # Smooth logistic-based power curve proxy bounded in [0, rated]
        s = torch.sigmoid((ws - self.u_mid) / torch.clamp(self.u_scale, min=0.5))
        cut_in_gate = torch.sigmoid((ws - self.cut_in) * 3.0)
        return self.rated * s * cut_in_gate

class WindTurbinePINN(nn.Module):
    def __init__(self, num_inputs=10, rated_power=2050.0):
        super().__init__()
        self.rated = rated_power
        self.prior = SmoothPowerPrior(rated_power)

        # Residual network for environmental perturbations (density, turbulence, yaw, terrain)
        self.net = nn.Sequential(
            nn.Linear(num_inputs, 128),
            nn.LayerNorm(128),
            nn.SiLU(),
            nn.Linear(128, 128),
            nn.LayerNorm(128),
            nn.SiLU(),
            nn.Linear(128, 64),
            nn.LayerNorm(64),
            nn.SiLU(),
        )

        # Dual output heads: Mean residual (Δμ) and log variance (log σ²)
        self.head_mu = nn.Linear(64, 1)
        self.head_log_var = nn.Linear(64, 1)

        # Initialize residual weights small so prior dominates at start of training
        nn.init.zeros_(self.head_mu.weight)
        nn.init.zeros_(self.head_mu.bias)
        nn.init.constant_(self.head_log_var.bias, np.log(40.0**2))  # Initialized around baseline ~40 kW spread

    def forward(self, x, ws):
        p_base = self.prior(ws.unsqueeze(-1))
        h = self.net(x)
        delta_mu = self.head_mu(h)
        log_var = self.head_log_var(h)

        # Physical clamping: Power must sit in [0, 1.05 * rated]
        mu = torch.clamp(p_base + delta_mu, min=0.0, max=self.rated * 1.05)
        
        # Floor uncertainty at 3.0 kW
        sigma = torch.clamp(torch.exp(0.5 * log_var), min=3.0)
        return mu.squeeze(-1), sigma.squeeze(-1)

# 5. Training Setup
model = WindTurbinePINN(num_inputs=len(INPUT_COLS), rated_power=rated).to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)

# Physics weights
LAMBDA_MONO = 10.0   # Penalty weight for negative power-wind slope in Region II

print("\nStarting PINN Training (8 Epochs)...")
for epoch in range(1, 9):
    model.train()
    total_loss, total_nll, total_mono = 0.0, 0.0, 0.0
    t_start = time.time()

    for bx, by, bws in train_loader:
        bx, by, bws = bx.to(device), by.to(device), bws.to(device)
        bx.requires_grad_(True)

        optimizer.zero_grad()
        mu, sigma = model(bx, bws)

        # 1. Heteroscedastic Gaussian NLL loss
        nll_loss = torch.mean(torch.log(sigma) + 0.5 * ((by - mu) / sigma) ** 2)

        # 2. Physics Loss: Monotonicity d(mu)/d(ws) >= 0 in Region II (ws in [3, 12])
        # Compute gradient of mu with respect to normalized ws (index 0)
        grad_ws = torch.autograd.grad(
            outputs=mu.sum(),
            inputs=bx,
            create_graph=True,
            retain_graph=True
        )[0][:, 0]

        # Penalize negative slope where ws <= 12 m/s
        reg2_mask = (bws <= 12.0)
        mono_violations = torch.relu(-grad_ws) * reg2_mask.float()
        mono_loss = LAMBDA_MONO * torch.mean(mono_violations ** 2)

        loss = nll_loss + mono_loss
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
        optimizer.step()

        total_loss += loss.item() * len(by)
        total_nll += nll_loss.item() * len(by)
        total_mono += mono_loss.item() * len(by)

    n_samples = len(train_loader.dataset)
    train_nll = total_nll / n_samples
    train_mono = total_mono / n_samples

    # Quick Validation Check on subset
    model.eval()
    val_nll = 0.0
    with torch.no_grad():
        for bx, by, bws in test_loader:
            bx, by, bws = bx.to(device), by.to(device), bws.to(device)
            mu, sigma = model(bx, bws)
            nll = torch.mean(torch.log(sigma) + 0.5 * ((by - mu) / sigma) ** 2)
            val_nll += nll.item() * len(by)
            break  # evaluate first batch for speed during training loop
    val_nll /= len(bx)
    scheduler.step(val_nll)

    print(f"Epoch {epoch:02d} | Train NLL: {train_nll:.4f} | Mono Pen: {train_mono:.5f} | Val NLL: {val_nll:.4f} | {time.time()-t_start:.1f}s")

# 6. Full Evaluation on Held-Out Test Set (>= 2022)
print("\n" + "="*50)
print("EVALUATING PINN ON TEST SET (Years >= 2022)")
print("="*50)
model.eval()

preds_mu, preds_sigma = [], []
with torch.no_grad():
    for bx, by, bws in test_loader:
        bx, bws = bx.to(device), bws.to(device)
        mu, sigma = model(bx, bws)
        preds_mu.append(mu.cpu().numpy())
        preds_sigma.append(sigma.cpu().numpy())

preds_mu = np.concatenate(preds_mu)
preds_sigma = np.concatenate(preds_sigma)

# Metrics Calculation
e_te = y_te - preds_mu
rmse_pinn = np.sqrt(mean_squared_error(y_te, preds_mu))
mae_pinn = mean_absolute_error(y_te, preds_mu)
r2_pinn = r2_score(y_te, preds_mu)

z = np.abs(e_te) / preds_sigma
nll_pinn = np.mean(np.log(preds_sigma) + 0.5 * (e_te / preds_sigma) ** 2) + 0.5 * np.log(2 * np.pi)
c1 = (z < 1.0).mean()
c2 = (z < 2.0).mean()

print(f"Mean Model Performance:")
print(f"  Test RMSE : {rmse_pinn:.2f} kW  (Baseline HGB: 49.80 kW)")
print(f"  Test MAE  : {mae_pinn:.2f} kW  (Baseline HGB: 33.70 kW)")
print(f"  Test R²   : {r2_pinn:.4f}     (Baseline HGB: 0.9944)")

print(f"\nUncertainty & Calibration Performance:")
print(f"  Test NLL  : {nll_pinn:.4f}   (Baseline HGB: 5.0070)")
print(f"  1-sigma   : {c1:.4f}       (Gaussian target: 0.6830)")
print(f"  2-sigma   : {c2:.4f}       (Gaussian target: 0.9540)")
print(f"  Mean σ    : {preds_sigma.mean():.1f} kW")

In [ ]:
# ============================================================
# GMM SYNTHETIC DATA MODEL (fit, one per farm)
# Purpose: a GaussianMixture over the *stochastic environment state*
#   + power, so we can (case 1) complete rows that contain NaNs and
#   (case 2) generate fully synthetic rows with the same columns.
#
# GMM_COLS (sampled):      ws, sin_wd, cos_wd, vane, temp, TI,
#                          month_sin, month_cos, power_kw
# NOT sampled (rebuilt deterministically after any draw):
#   abs_vane  = |clipped vane|           (degenerate duplicate)
#   x/y/elev  = static per turbine       (mapped from layout())
#   u_iec/rho = closed form of (ws, temp, elev)
#   turbine/year/month = metadata / conditioning, not environment state
# ============================================================
import time
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from scipy.special import logsumexp

W = "/kaggle/working"
TS = "Date and time"

GMM_COLS = ["ws", "sin_wd", "cos_wd", "vane", "temp", "TI", "month_sin", "month_cos", "power_kw"]
GMM_K_GRID = [8, 16, 24, 32, 48, 64]     # BIC scan over component count
GMM_BIC_SAMPLE = 200_000                 # rows for the BIC scan
GMM_FIT_SAMPLE = 600_000                 # rows for the final refit
GMM_SEED = 42

gmm_models = {}      # farm -> {gmm, scaler, cols, bounds, k, n_fit, seconds}

for farm in ["penmanshiel", "kelmarsh"]:
    t0 = time.time()
    d = pd.read_parquet(f"{W}/parquet/train/{farm}_m1_clean.parquet")
    d = d[~d["is_test"]].dropna(subset=GMM_COLS).reset_index(drop=True)  # train rows only, fully observed
    print(f"\n=== {farm}: {len(d):,} fully-observed train rows available for the GMM fit")

    # -- component count by BIC on a subsample
    fit_sub = d.sample(min(GMM_BIC_SAMPLE, len(d)), random_state=GMM_SEED)
    scaler = StandardScaler().fit(fit_sub[GMM_COLS])
    Z_sub = scaler.transform(fit_sub[GMM_COLS]).astype(np.float64)
    bic = {}
    for k in GMM_K_GRID:
        g = GaussianMixture(n_components=k, covariance_type="full", reg_covar=1e-5,
                            n_init=1, random_state=GMM_SEED).fit(Z_sub)   # n_init=1 keeps the BIC scan fast
        bic[k] = g.bic(Z_sub)
        print(f"  k={k:>2}  BIC={bic[k]:>14,.0f}")
    k_best = min(bic, key=bic.get)

    # -- final refit on a larger sample
    fit_big = d.sample(min(GMM_FIT_SAMPLE, len(d)), random_state=GMM_SEED)
    Z_big = scaler.transform(fit_big[GMM_COLS]).astype(np.float64)
    gmm = GaussianMixture(n_components=k_best, covariance_type="full", reg_covar=1e-5,
                          n_init=2, random_state=GMM_SEED).fit(Z_big)
    print(f"  selected k={k_best} | refit on {len(fit_big):,} rows | "
          f"components with >1% weight: {(gmm.weights_ > 0.01).sum()}")

    # -- physical clip bounds from train quantiles, applied to every draw
    bounds = {
        "ws":        (0.0,  float(np.floor(d["ws"].quantile(0.999)))),
        "sin_wd":    (-1.0, 1.0),   "cos_wd": (-1.0, 1.0),
        "vane":      (-45.0, 45.0),
        "temp":      (float(d["temp"].quantile(0.001)), float(d["temp"].quantile(0.999))),
        "TI":        (0.0, 0.6),
        "month_sin": (-1.0, 1.0),   "month_cos": (-1.0, 1.0),
        "power_kw":  (float(d["power_kw"].quantile(0.001)), float(d["power_kw"].quantile(0.999)) + 60.0),
    }
    gmm_models[farm] = dict(gmm=gmm, scaler=scaler, cols=GMM_COLS, bounds=bounds,
                            k=k_best, n_fit=len(fit_big), seconds=round(time.time() - t0, 1))
    print(f"  fit done in {gmm_models[farm]['seconds']}s")


In [ ]:
# ============================================================
# CASE 1: NaN completion via conditional GMM
# A row with NaNs in GMM_COLS is completed by drawing the missing
# dims from the mixture *conditioned on the observed dims*
# (responsibilities use the observed dims only). Known metadata
# (turbine, x/y/elev, month when a timestamp exists) is never sampled.
# Also rebuilds the rows the cell-44 build dropped for core NaN
# and completes them -> {farm}_m1_gmm_completed.parquet
# ============================================================
import time as _t
import sys
sys.path.insert(0, f"{W}/code")
from common import load_feat, clean_rows, layout

def _comp_logpdf(Z, mu, Cov):
    # (n, K) log N(z | mu_k, Cov_k) for a fixed set of dims
    out = np.empty((Z.shape[0], mu.shape[0]))
    for j in range(mu.shape[0]):
        Lc = np.linalg.cholesky(Cov[j])
        sol = np.linalg.solve(Lc, (Z - mu[j]).T)
        out[:, j] = -0.5 * (Z.shape[1] * np.log(2 * np.pi)
                            + 2 * np.log(np.diag(Lc)).sum() + (sol ** 2).sum(axis=0))
    return out

def _cond_draw(m, obs_idx, Z_obs, rng, n):
    """Standardized rows: dims in obs_idx fixed at Z_obs, the rest drawn from the
    conditional mixture. Returns (n, d) standardized rows."""
    gmm, cols = m["gmm"], m["cols"]
    d, K = len(cols), gmm.weights_.shape[0]
    mu, Cov = gmm.means_, gmm.covariances_
    O = np.asarray(obs_idx, dtype=int)
    M = np.array([j for j in range(d) if j not in set(obs_idx)], dtype=int)
    Z = np.empty((n, d))
    if len(O):
        Z[:, O] = Z_obs
    if len(M) == 0:
        return Z

    if len(O) == 0:                                   # nothing observed: plain mixture draw
        log_r = np.log(gmm.weights_)[None, :].repeat(n, axis=0)
    else:                                             # responsibilities from observed dims
        lp = _comp_logpdf(Z[:, O], mu[:, O], Cov[:, O][:, :, O])
        log_r = lp + np.log(gmm.weights_)[None, :]
        log_r = log_r - logsumexp(log_r, axis=1, keepdims=True)

    # per-component conditional Gaussian over the missing dims
    A = np.empty((K, len(M), len(O)))                 # Cov_MO @ inv(Cov_OO)
    cmu = np.empty((K, len(M)))
    cL = np.empty((K, len(M), len(M)))
    for j in range(K):
        Coo, Cmo, Cmm = Cov[j][np.ix_(O, O)], Cov[j][np.ix_(M, O)], Cov[j][np.ix_(M, M)]
        if len(O):
            A[j] = np.linalg.solve(Coo, Cmo.T).T      # Cmo @ inv(Coo), stable
        cmu[j] = mu[j, M]
        cL[j] = np.linalg.cholesky(Cmm)

    k_draw = np.argmax(log_r + rng.gumbel(size=(n, K)), axis=1)
    if len(O):
        diff = Z_obs - mu[k_draw][:, O]
        base = cmu[k_draw] + np.einsum("no,nmo->nm", diff, A[k_draw])
    else:
        base = cmu[k_draw]
    eps = rng.standard_normal((n, len(M)))
    Z[:, M] = base + np.einsum("nm,nlm->nl", eps, cL[k_draw])
    return Z

def gmm_impute_rows(df, farm, seed=GMM_SEED, verbose=True):
    """Fill NaNs in GMM_COLS. Only imputed cells are clipped to physical bounds;
    observed values are left untouched."""
    m = gmm_models[farm]
    cols, scaler, bounds = m["cols"], m["scaler"], m["bounds"]
    rng = np.random.default_rng(seed)
    out = df.copy()

    # month is known whenever a timestamp exists -> fill from TS, never sampled
    if TS in out.columns:
        rad = 2 * np.pi * (out[TS].dt.month - 1) / 12.0
        out["month_sin"] = out["month_sin"].fillna(np.sin(rad))
        out["month_cos"] = out["month_cos"].fillna(np.cos(rad))

    todo = out.index[out[cols].isna().any(axis=1)]
    if verbose:
        print(f"[{farm}] rows with NaN in GMM_COLS: {len(todo):,}")
    if len(todo) == 0:
        return out
    filled = {c: 0 for c in cols}

    na_bool = out.loc[todo, cols].isna().values
    key = pd.Series([tuple(c for c, b in zip(cols, row) if b) for row in na_bool], index=todo)
    for pat, pat_idx in key.groupby(key):
        pat = list(pat)
        grp = out.loc[pat_idx.index]                 # rows sharing the same NaN pattern
        obs_cols = [c for c in cols if c not in pat]
        obs_pos = [cols.index(c) for c in obs_cols]
        if obs_cols:                                 # standardize observed dims (subset of the fitted cols)
            Z_obs = ((grp[obs_cols].values.astype(np.float64) - scaler.mean_[obs_pos])
                     / scaler.scale_[obs_pos])
        else:
            Z_obs = np.zeros((len(grp), 0))
        Z = _cond_draw(m, obs_pos, Z_obs, rng, len(grp))
        X = scaler.inverse_transform(Z)
        for j, c in enumerate(cols):
            if c in pat:
                vals = np.clip(X[:, j], *bounds[c]).astype(np.float32)
                out.loc[grp.index, c] = vals
                filled[c] += len(grp)
    out["abs_vane"] = out["vane"].clip(-45, 45).abs().astype(np.float32)
    if verbose:
        print(f"[{farm}] filled: " + ", ".join(f"{c}={v:,}" for c, v in filled.items() if v))
    return out

def build_impute_candidates(farm):
    """Rebuild the rows cell 44 dropped for NaN core features (clean per mask_m1,
    after the farm start date). ws/power are always present there (core_nan flag),
    so the typical NaNs are vane / temp / TI / wind-direction."""
    s, _ = layout(farm)
    d = load_feat(farm)
    start = "2016-09-01" if farm == "penmanshiel" else "2018-01-01"
    c = clean_rows(d, farm, start=start).copy()
    wd = np.deg2rad(c["Wind direction (°)"])
    month = c[TS].dt.month
    rad = 2 * np.pi * (month - 1) / 12.0
    vane_clip = c["Vane position 1+2 (°)"].clip(-45, 45)
    df = pd.DataFrame({
        TS: c[TS], "turbine": c["turbine"].astype(np.int8), "year": c[TS].dt.year.astype(np.int16),
        "month": month.astype(np.int8),
        "x": c["turbine"].map(s["x"]).astype(np.float32), "y": c["turbine"].map(s["y"]).astype(np.float32),
        "elev": c["turbine"].map(s["elev"]).astype(np.float32),
        "ws": c["Wind speed (m/s)"].astype(np.float32),
        "sin_wd": np.sin(wd).astype(np.float32), "cos_wd": np.cos(wd).astype(np.float32),
        "vane": vane_clip.astype(np.float32), "abs_vane": vane_clip.abs().astype(np.float32),
        "temp": c["Nacelle ambient temperature (°C)"].astype(np.float32),
        "TI": (c["Wind speed, Standard deviation (m/s)"] / c["Wind speed (m/s)"].where(c["Wind speed (m/s)"] >= 1.0)).astype(np.float32),
        "month_sin": np.sin(rad).astype(np.float32), "month_cos": np.cos(rad).astype(np.float32),
        "power_kw": c["Power (kW)"].astype(np.float32),
        "is_test": (c[TS].dt.year >= 2022).astype(bool),
    })
    df = df.dropna(subset=["ws", "power_kw", TS])     # clean rows always have these
    core = ["ws", "sin_wd", "cos_wd", "vane", "temp", "TI", "power_kw"]
    return df[df[core].isna().any(axis=1)].copy()

imputed_frames = {}
for farm in ["penmanshiel", "kelmarsh"]:
    _t0 = _t.time()
    cand = build_impute_candidates(farm)
    print(f"\n=== {farm}: {len(cand):,} recoverable rows with NaN core features "
          f"({cand[~cand['is_test']].shape[0]:,} train / {cand['is_test'].sum():,} test)")
    done = gmm_impute_rows(cand, farm)
    done["gmm_imputed"] = True
    done["synthetic"] = False
    imputed_frames[farm] = done.reset_index(drop=True)
    p = f"{W}/parquet/train/{farm}_m1_gmm_completed.parquet"
    done.to_parquet(p, index=False)
    print(f"[{farm}] saved {len(done):,} completed rows -> {p}  ({_t.time() - _t0:.1f}s)")
    del cand, done


In [ ]:
# ============================================================
# CASE 2: full-row generation from the fitted GMM + validation
# generate_rows(farm, n, priors=..., turbine=...) -> rows with the
# m1_clean schema; statics (turbine, x/y/elev) attached, derived
# cols (abs_vane) recomputed, is_test=False, synthetic=True.
# Priors:  fixed value  e.g. {"temp": 11.5}     -> conditional sampling
#          range        e.g. {"ws": (6.0, 9.0)} -> resampling within range
# ============================================================
import matplotlib.pyplot as plt

def _mixture_draw(m, rng, n):
    gmm, d = m["gmm"], len(m["cols"])
    L = np.stack([np.linalg.cholesky(C) for C in gmm.covariances_])
    k = rng.choice(gmm.weights_.shape[0], size=n, p=gmm.weights_)
    eps = rng.standard_normal((n, d))
    return gmm.means_[k] + np.einsum("nd,nde->ne", eps, L[k])

def _clip_bounds(m, X):
    for c, (lo, hi) in m["bounds"].items():
        X[c] = X[c].clip(lo, hi)
    return X

def generate_rows(farm, n, priors=None, turbine=None, seed=GMM_SEED, max_tries=25):
    m = gmm_models[farm]
    cols, scaler = m["cols"], m["scaler"]
    rng = np.random.default_rng(seed)
    priors = dict(priors or {})
    fixed = {c: v for c, v in priors.items() if np.isscalar(v)}                 # conditioning
    ranges = {c: v for c, v in priors.items() if isinstance(v, (tuple, list))}  # rejection

    if fixed:
        obs_cols = list(fixed)
        obs_pos = [cols.index(c) for c in obs_cols]
        Xfix = np.array([[fixed[c] for c in obs_cols]], dtype=np.float64)
        Z_obs = np.repeat((Xfix - scaler.mean_[obs_pos]) / scaler.scale_[obs_pos], n, axis=0)
        Z = _cond_draw(m, obs_pos, Z_obs, rng, n)
    else:
        Z = _mixture_draw(m, rng, n)

    X = _clip_bounds(m, pd.DataFrame(scaler.inverse_transform(Z), columns=cols))
    for c, (lo, hi) in ranges.items():                    # range priors: resample outliers
        bad = ~X[c].between(lo, hi)
        tries = 0
        while bad.any() and tries < max_tries:
            Xr = _clip_bounds(m, pd.DataFrame(scaler.inverse_transform(
                _mixture_draw(m, rng, int(bad.sum()))), columns=cols))
            X.loc[bad, cols] = Xr.values
            bad = ~X[c].between(lo, hi)
            tries += 1
        X[c] = X[c].clip(lo, hi)

    # ---- assemble the m1_clean schema ----
    s, _ = layout(farm)
    turb = rng.choice(np.array(sorted(s.index)), size=n) if turbine is None else np.full(n, turbine)
    month = (np.round(np.arctan2(X["month_sin"], X["month_cos"]) / (2 * np.pi / 12)).astype(int) % 12) + 1
    out = pd.DataFrame({
        TS: pd.NaT,                                        # synthetic rows carry no timestamp
        "turbine": turb.astype(np.int8),
        "year": np.int16(-1),                              # -1 = synthetic sentinel
        "month": month.astype(np.int8),
        "x": s.loc[turb, "x"].values.astype(np.float32),
        "y": s.loc[turb, "y"].values.astype(np.float32),
        "elev": s.loc[turb, "elev"].values.astype(np.float32),
        "ws": X["ws"].values.astype(np.float32),
        "sin_wd": X["sin_wd"].values.astype(np.float32),
        "cos_wd": X["cos_wd"].values.astype(np.float32),
        "vane": X["vane"].values.astype(np.float32),
        "abs_vane": X["vane"].abs().values.astype(np.float32),
        "temp": X["temp"].values.astype(np.float32),
        "TI": X["TI"].values.astype(np.float32),
        "month_sin": X["month_sin"].values.astype(np.float32),
        "month_cos": X["month_cos"].values.astype(np.float32),
        "power_kw": X["power_kw"].values.astype(np.float32),
        "is_test": False,
        "synthetic": True,
    })
    return out

# ---------------- validation: real vs synthetic ----------------
def validate_gmm(farm, n=100_000, seed=7):
    real = pd.read_parquet(f"{W}/parquet/train/{farm}_m1_clean.parquet")
    real = real[~real["is_test"]].dropna(subset=GMM_COLS)
    real = real.sample(min(n, len(real)), random_state=seed).reset_index(drop=True)
    syn = generate_rows(farm, len(real), seed=seed + 1)

    show = ["ws", "vane", "temp", "TI", "power_kw"]
    fig, axes = plt.subplots(len(show), 1, figsize=(9, 2.6 * len(show)))
    for ax, c in zip(axes, show):
        lo, hi = real[c].quantile([0.001, 0.999])
        bins = np.linspace(lo, hi, 60)
        ax.hist(real[c], bins=bins, alpha=0.55, density=True, label="real")
        ax.hist(syn[c], bins=bins, alpha=0.55, density=True, label="GMM")
        ax.set_title(c); ax.legend()
    plt.tight_layout(); plt.suptitle(f"{farm}: marginals real vs GMM", y=1.002); plt.show()

    fig, ax = plt.subplots(1, 2, figsize=(14, 6))
    for a, X, t in [(ax[0], real, "real"), (ax[1], syn, "GMM")]:
        im = a.imshow(X[GMM_COLS].corr(), vmin=-1, vmax=1, cmap="coolwarm")
        a.set_xticks(range(len(GMM_COLS))); a.set_xticklabels(GMM_COLS, rotation=90, fontsize=7)
        a.set_yticks(range(len(GMM_COLS))); a.set_yticklabels(GMM_COLS, fontsize=7)
        a.set_title(f"{farm}: corr ({t})")
    plt.colorbar(im, ax=ax, shrink=0.8); plt.show()

    b = np.arange(3, 20.5, 0.5)
    rb = real.groupby(np.floor(real["ws"] * 2) / 2)["power_kw"].median()
    sb = syn.groupby(np.floor(syn["ws"] * 2) / 2)["power_kw"].median()
    plt.figure(figsize=(9, 4))
    plt.plot(rb.index, rb.values, ".", label="real")
    plt.plot(sb.index, sb.values, ".", label="GMM")
    plt.xlabel("ws bin"); plt.ylabel("median power_kw"); plt.title(f"{farm}: binned power curve")
    plt.legend(); plt.grid(alpha=0.3); plt.show()

for farm in ["penmanshiel", "kelmarsh"]:
    validate_gmm(farm)


In [ ]:
# ============================================================
# INFERENCE: generate N synthetic rows (optionally with priors)
# and concat them onto the training data used downstream.
# Result: train_aug  (also saved to parquet, m1_clean is NOT overwritten)
# ============================================================
# ---------- USER CONFIG ----------
FARM          = "penmanshiel"   # "penmanshiel" | "kelmarsh"
N_SAMPLES     = 200_000         # how many synthetic rows
PRIORS        = {}              # e.g. {"ws": (6.0, 9.0)} range, {"temp": 11.5} fixed value
TURBINE       = None            # e.g. 7 to fix a turbine, None = uniform over the farm
WS_RANGE      = (3.0, 20.0)     # keep rows inside the model-1 operating band (None = no filter)
MERGE_IMPUTED = True            # also append the case-1 GMM-completed rows
SEED          = 1234
# ---------------------------------

syn = generate_rows(FARM, N_SAMPLES, priors=PRIORS, turbine=TURBINE, seed=SEED)
if WS_RANGE:
    syn = syn[syn["ws"].between(*WS_RANGE)].reset_index(drop=True)

# base training frame: reuse the active one from cell 48 when it matches, else reload
if FARM == "penmanshiel" and "tr_df" in globals():
    base = tr_df.copy()
else:
    _d = pd.read_parquet(f"{W}/parquet/train/{FARM}_m1_clean.parquet")
    base = _d[~_d["is_test"] & _d["ws"].between(3.0, 20.0)].reset_index(drop=True)
base["synthetic"] = False
base["gmm_imputed"] = False

parts = [base]
if MERGE_IMPUTED and "imputed_frames" in globals() and FARM in imputed_frames:
    imp = imputed_frames[FARM]
    imp = imp[imp["ws"].between(*WS_RANGE)] if WS_RANGE else imp
    parts.append(imp)
parts.append(syn.assign(gmm_imputed=False))

train_aug = pd.concat(parts, ignore_index=True)
out_path = f"{W}/parquet/train/{FARM}_gmm_augmented.parquet"
train_aug.to_parquet(out_path, index=False)

print(f"{FARM}: base {len(base):,} + imputed {int((train_aug.gmm_imputed).sum()):,} "
      f"+ synthetic {len(syn):,} -> train_aug {len(train_aug):,} rows "
      f"({(train_aug.synthetic | train_aug.gmm_imputed).mean():.1%} augmented)")
print(f"saved -> {out_path}")
print(train_aug.groupby(["synthetic"], dropna=False)["power_kw"].describe().round(1).to_string())


In [ ]:
import os, time
import numpy as np, pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from huggingface_hub import HfApi, login, hf_hub_download
from kaggle_secrets import UserSecretsClient

# 1. Environment & Auth
login(UserSecretsClient().get_secret("HF_TOKEN"))
api = HfApi()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")

W = "/kaggle/working"
REPO = "B4BUK4/MPVI_project"
TS = "Date and time"
rated = 2050.0

# Upgraded input feature set (12 inputs: aerodynamic, atmospheric, spatial, seasonal)
INPUT_COLS = [
    "ws", "sin_wd", "cos_wd", "vane", "abs_vane", "temp", "TI",
    "x", "y", "elev", "month_sin", "month_cos"
]
TARGET_COL = "power_kw"

# 2. Load Consolidated Training Table
df = pd.read_parquet(f"{W}/parquet/train/penmanshiel_m1_clean.parquet")
df = df[df["ws"].between(3.0, 20.0)].reset_index(drop=True)

tr_df = df[~df["is_test"]].copy()
te_df = df[df["is_test"]].copy()

# Sample 1,200,000 rows (out of 3.14M available)
SAMPLE_SIZE = 1_200_000
tr_sample = tr_df.sample(min(SAMPLE_SIZE, len(tr_df)), random_state=42).reset_index(drop=True)

print(f"Dataset scale: {len(tr_sample):,} train samples | {len(te_df):,} test samples")

# 3. Normalization (Fit scaler strictly on training set)
scaler_X = StandardScaler()
X_tr = scaler_X.fit_transform(tr_sample[INPUT_COLS]).astype(np.float32)
X_te = scaler_X.transform(te_df[INPUT_COLS]).astype(np.float32)

y_tr = tr_sample[TARGET_COL].values.astype(np.float32)
y_te = te_df[TARGET_COL].values.astype(np.float32)

ws_tr = tr_sample["ws"].values.astype(np.float32)
ws_te = te_df["ws"].values.astype(np.float32)

train_dataset = TensorDataset(torch.tensor(X_tr), torch.tensor(y_tr), torch.tensor(ws_tr))
test_dataset = TensorDataset(torch.tensor(X_te), torch.tensor(y_te), torch.tensor(ws_te))

train_loader = DataLoader(train_dataset, batch_size=2048, shuffle=True, drop_last=True)
test_loader = DataLoader(test_dataset, batch_size=4096, shuffle=False)

# 4. Deep Residual PINN Architecture
class ResBlock(nn.Module):
    """Residual block with LayerNorm for tabular neural representations"""
    def __init__(self, dim):
        super().__init__()
        self.fc1 = nn.Linear(dim, dim)
        self.ln1 = nn.LayerNorm(dim)
        self.act = nn.SiLU()
        self.fc2 = nn.Linear(dim, dim)
        self.ln2 = nn.LayerNorm(dim)

    def forward(self, x):
        residual = x
        out = self.act(self.ln1(self.fc1(x)))
        out = self.ln2(self.fc2(out))
        return self.act(out + residual)

class SmoothPowerPrior(nn.Module):
    """Analytic aerodynamic base power curve (Region I -> Region II cubic -> Region III plateau)"""
    def __init__(self, rated_power=2050.0):
        super().__init__()
        self.rated = rated_power
        self.u_mid = nn.Parameter(torch.tensor(8.5))
        self.u_scale = nn.Parameter(torch.tensor(1.8))
        self.cut_in = nn.Parameter(torch.tensor(3.0))

    def forward(self, ws):
        s = torch.sigmoid((ws - self.u_mid) / torch.clamp(self.u_scale, min=0.5))
        cut_in_gate = torch.sigmoid((ws - self.cut_in) * 3.0)
        return self.rated * s * cut_in_gate

class OptimizedWindPINN(nn.Module):
    def __init__(self, num_inputs=12, hidden_dim=128, rated_power=2050.0):
        super().__init__()
        self.rated = rated_power
        self.prior = SmoothPowerPrior(rated_power)

        # Feature projection
        self.in_proj = nn.Sequential(
            nn.Linear(num_inputs, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.SiLU()
        )

        # 2 Residual blocks
        self.res1 = ResBlock(hidden_dim)
        self.res2 = ResBlock(hidden_dim)

        self.out_proj = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.LayerNorm(64),
            nn.SiLU()
        )

        # Dual output heads: Mean residual (Δμ) and log variance (log σ²)
        self.head_mu = nn.Linear(64, 1)
        self.head_log_var = nn.Linear(64, 1)

        nn.init.zeros_(self.head_mu.weight)
        nn.init.zeros_(self.head_mu.bias)
        nn.init.constant_(self.head_log_var.bias, np.log(40.0**2))

    def forward(self, x, ws):
        p_base = self.prior(ws.unsqueeze(-1))
        h = self.in_proj(x)
        h = self.res1(h)
        h = self.res2(h)
        feat = self.out_proj(h)

        delta_mu = self.head_mu(feat)
        log_var = self.head_log_var(feat)

        mu = torch.clamp(p_base + delta_mu, min=0.0, max=self.rated * 1.05)
        sigma = torch.clamp(torch.exp(0.5 * log_var), min=3.0)
        return mu.squeeze(-1), sigma.squeeze(-1)

# 5. Training Setup
model = OptimizedWindPINN(num_inputs=len(INPUT_COLS), hidden_dim=128, rated_power=rated).to(device)
optimizer = optim.AdamW(model.parameters(), lr=1.5e-3, weight_decay=1e-4)

EPOCHS = 14
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)
LAMBDA_MONO = 10.0

print(f"\nStarting Optimized PINN Training ({EPOCHS} Epochs on {len(train_loader.dataset):,} rows)...")
t_all = time.time()

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss, total_nll, total_mono = 0.0, 0.0, 0.0
    t_start = time.time()

    for bx, by, bws in train_loader:
        bx, by, bws = bx.to(device), by.to(device), bws.to(device)
        bx.requires_grad_(True)

        optimizer.zero_grad()
        mu, sigma = model(bx, bws)

        # Heteroscedastic NLL Loss
        nll_loss = torch.mean(torch.log(sigma) + 0.5 * ((by - mu) / sigma) ** 2)

        # Monotonicity Physics Constraint via autograd (ws is at index 0)
        grad_ws = torch.autograd.grad(
            outputs=mu.sum(),
            inputs=bx,
            create_graph=True,
            retain_graph=True
        )[0][:, 0]

        reg2_mask = (bws <= 12.0)
        mono_violations = torch.relu(-grad_ws) * reg2_mask.float()
        mono_loss = LAMBDA_MONO * torch.mean(mono_violations ** 2)

        loss = nll_loss + mono_loss
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
        optimizer.step()

        total_loss += loss.item() * len(by)
        total_nll += nll_loss.item() * len(by)
        total_mono += mono_loss.item() * len(by)

    scheduler.step()
    n_samples = len(train_loader.dataset)
    current_lr = scheduler.get_last_lr()[0]
    print(f"Epoch {epoch:02d}/{EPOCHS} | Train NLL: {total_nll/n_samples:.4f} | Mono Pen: {total_mono/n_samples:.5f} | lr: {current_lr:.6f} | {time.time()-t_start:.1f}s")

print(f"\nTraining completed in {time.time()-t_all:.1f}s.")

# 6. Full Evaluation on Held-Out Test Set (Years >= 2022)
print("\n" + "="*55)
print("EVALUATING OPTIMIZED PINN ON TEST SET (Years >= 2022)")
print("="*55)
model.eval()

preds_mu, preds_sigma = [], []
with torch.no_grad():
    for bx, by, bws in test_loader:
        bx, bws = bx.to(device), bws.to(device)
        mu, sigma = model(bx, bws)
        preds_mu.append(mu.cpu().numpy())
        preds_sigma.append(sigma.cpu().numpy())

preds_mu = np.concatenate(preds_mu)
preds_sigma = np.concatenate(preds_sigma)

# Metrics
e_te = y_te - preds_mu
rmse_opt = np.sqrt(mean_squared_error(y_te, preds_mu))
mae_opt = mean_absolute_error(y_te, preds_mu)
r2_opt = r2_score(y_te, preds_mu)

z = np.abs(e_te) / preds_sigma
nll_opt = np.mean(np.log(preds_sigma) + 0.5 * (e_te / preds_sigma) ** 2) + 0.5 * np.log(2 * np.pi)
c1 = (z < 1.0).mean()
c2 = (z < 2.0).mean()

print(f"\n--- Comparative Performance Report (Penmanshiel Test Set) ---")
report = pd.DataFrame({
    "Model": ["Baseline HGB (Trees)", "PINN v1 (500k samples)", "PINN Optimized (1.2M samples)"],
    "Test RMSE (kW)": [49.80, 51.82, round(rmse_opt, 2)],
    "Test MAE (kW)": [33.70, 34.84, round(mae_opt, 2)],
    "Test R²": [0.9944, 0.9939, round(r2_opt, 4)],
    "Test NLL": [5.0070, 5.0279, round(nll_opt, 4)],
    "1-sigma Cov": [0.6846, 0.6631, round(c1, 4)],
    "2-sigma Cov": [0.9442, 0.9316, round(c2, 4)]
})
print(report.to_string(index=False))

# 7. Save Checkpoint and Upload to Hugging Face
os.makedirs(f"{W}/models", exist_ok=True)
ckpt_path = f"{W}/models/pinn_m1_penmanshiel.pt"

torch.save({
    "model_state_dict": model.state_dict(),
    "scaler_mean": scaler_X.mean_,
    "scaler_scale": scaler_X.scale_,
    "input_cols": INPUT_COLS,
    "metrics": {"rmse": rmse_opt, "nll": nll_opt, "r2": r2_opt}
}, ckpt_path)

print(f"\nSaved checkpoint: {ckpt_path} ({os.path.getsize(ckpt_path) / 1e3:.1f} KB)")
print("Uploading model checkpoint to Hugging Face (repo: B4BUK4/MPVI_project)...")

api.upload_file(
    path_or_fileobj=ckpt_path,
    path_in_repo="models/pinn_m1_penmanshiel.pt",
    repo_id=REPO,
    repo_type="dataset",
    commit_message=f"Add optimized Penmanshiel PINN M1 checkpoint (RMSE {rmse_opt:.2f} kW)"
)

# Readback check
chk_p = hf_hub_download(REPO, "models/pinn_m1_penmanshiel.pt", repo_type="dataset", local_dir=f"{W}/verify_model", force_download=True)
print(f"HF readback verified: {chk_p} exists and ready for Model 3 controller integration.")

Kelmarsh

In [ ]:
import os, time
import numpy as np, pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from huggingface_hub import HfApi, login, hf_hub_download
from kaggle_secrets import UserSecretsClient

# 1. Environment & Auth
login(UserSecretsClient().get_secret("HF_TOKEN"))
api = HfApi()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")

W = "/kaggle/working"
REPO = "B4BUK4/MPVI_project"
TS = "Date and time"
rated = 2050.0

INPUT_COLS = [
    "ws", "sin_wd", "cos_wd", "vane", "abs_vane", "temp", "TI",
    "x", "y", "elev", "month_sin", "month_cos"
]
TARGET_COL = "power_kw"

# 2. Load Consolidated Kelmarsh Training Table
df = pd.read_parquet(f"{W}/parquet/train/kelmarsh_m1_clean.parquet")
df = df[df["ws"].between(3.0, 20.0)].reset_index(drop=True)

tr_df = df[~df["is_test"]].copy()
te_df = df[df["is_test"]].copy()

# Sample 800,000 rows (out of ~1.05M available train rows)
SAMPLE_SIZE = 800_000
tr_sample = tr_df.sample(min(SAMPLE_SIZE, len(tr_df)), random_state=42).reset_index(drop=True)

print(f"Kelmarsh scale: {len(tr_sample):,} train samples (<=2021) | {len(te_df):,} test samples (>=2022)")

# 3. Normalization (Fit scaler strictly on training set)
scaler_X = StandardScaler()
X_tr = scaler_X.fit_transform(tr_sample[INPUT_COLS]).astype(np.float32)
X_te = scaler_X.transform(te_df[INPUT_COLS]).astype(np.float32)

y_tr = tr_sample[TARGET_COL].values.astype(np.float32)
y_te = te_df[TARGET_COL].values.astype(np.float32)

ws_tr = tr_sample["ws"].values.astype(np.float32)
ws_te = te_df["ws"].values.astype(np.float32)

train_dataset = TensorDataset(torch.tensor(X_tr), torch.tensor(y_tr), torch.tensor(ws_tr))
test_dataset = TensorDataset(torch.tensor(X_te), torch.tensor(y_te), torch.tensor(ws_te))

train_loader = DataLoader(train_dataset, batch_size=2048, shuffle=True, drop_last=True)
test_loader = DataLoader(test_dataset, batch_size=4096, shuffle=False)

# 4. Deep Residual PINN Architecture
class ResBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.fc1 = nn.Linear(dim, dim)
        self.ln1 = nn.LayerNorm(dim)
        self.act = nn.SiLU()
        self.fc2 = nn.Linear(dim, dim)
        self.ln2 = nn.LayerNorm(dim)

    def forward(self, x):
        residual = x
        out = self.act(self.ln1(self.fc1(x)))
        out = self.ln2(self.fc2(out))
        return self.act(out + residual)

class SmoothPowerPrior(nn.Module):
    def __init__(self, rated_power=2050.0):
        super().__init__()
        self.rated = rated_power
        # Initialized slightly lower for MM92 (reaches rated power earlier than MM82)
        self.u_mid = nn.Parameter(torch.tensor(8.0))
        self.u_scale = nn.Parameter(torch.tensor(1.8))
        self.cut_in = nn.Parameter(torch.tensor(3.0))

    def forward(self, ws):
        s = torch.sigmoid((ws - self.u_mid) / torch.clamp(self.u_scale, min=0.5))
        cut_in_gate = torch.sigmoid((ws - self.cut_in) * 3.0)
        return self.rated * s * cut_in_gate

class OptimizedWindPINN(nn.Module):
    def __init__(self, num_inputs=12, hidden_dim=128, rated_power=2050.0):
        super().__init__()
        self.rated = rated_power
        self.prior = SmoothPowerPrior(rated_power)

        self.in_proj = nn.Sequential(
            nn.Linear(num_inputs, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.SiLU()
        )

        self.res1 = ResBlock(hidden_dim)
        self.res2 = ResBlock(hidden_dim)

        self.out_proj = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.LayerNorm(64),
            nn.SiLU()
        )

        self.head_mu = nn.Linear(64, 1)
        self.head_log_var = nn.Linear(64, 1)

        nn.init.zeros_(self.head_mu.weight)
        nn.init.zeros_(self.head_mu.bias)
        nn.init.constant_(self.head_log_var.bias, np.log(49.0**2))  # Initialized near Kelmarsh's baseline ~49 kW spread

    def forward(self, x, ws):
        p_base = self.prior(ws.unsqueeze(-1))
        h = self.in_proj(x)
        h = self.res1(h)
        h = self.res2(h)
        feat = self.out_proj(h)

        delta_mu = self.head_mu(feat)
        log_var = self.head_log_var(feat)

        mu = torch.clamp(p_base + delta_mu, min=0.0, max=self.rated * 1.05)
        sigma = torch.clamp(torch.exp(0.5 * log_var), min=3.0)
        return mu.squeeze(-1), sigma.squeeze(-1)

# 5. Training Setup
model = OptimizedWindPINN(num_inputs=len(INPUT_COLS), hidden_dim=128, rated_power=rated).to(device)
optimizer = optim.AdamW(model.parameters(), lr=1.5e-3, weight_decay=1e-4)

EPOCHS = 14
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)
LAMBDA_MONO = 10.0

print(f"\nStarting Kelmarsh PINN Training ({EPOCHS} Epochs on {len(train_loader.dataset):,} rows)...")
t_all = time.time()

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss, total_nll, total_mono = 0.0, 0.0, 0.0
    t_start = time.time()

    for bx, by, bws in train_loader:
        bx, by, bws = bx.to(device), by.to(device), bws.to(device)
        bx.requires_grad_(True)

        optimizer.zero_grad()
        mu, sigma = model(bx, bws)

        nll_loss = torch.mean(torch.log(sigma) + 0.5 * ((by - mu) / sigma) ** 2)

        # Monotonicity Physics Loss in Region II (ws <= 12 m/s)
        grad_ws = torch.autograd.grad(
            outputs=mu.sum(),
            inputs=bx,
            create_graph=True,
            retain_graph=True
        )[0][:, 0]

        reg2_mask = (bws <= 12.0)
        mono_violations = torch.relu(-grad_ws) * reg2_mask.float()
        mono_loss = LAMBDA_MONO * torch.mean(mono_violations ** 2)

        loss = nll_loss + mono_loss
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
        optimizer.step()

        total_loss += loss.item() * len(by)
        total_nll += nll_loss.item() * len(by)
        total_mono += mono_loss.item() * len(by)

    scheduler.step()
    n_samples = len(train_loader.dataset)
    current_lr = scheduler.get_last_lr()[0]
    print(f"Epoch {epoch:02d}/{EPOCHS} | Train NLL: {total_nll/n_samples:.4f} | Mono Pen: {total_mono/n_samples:.5f} | lr: {current_lr:.6f} | {time.time()-t_start:.1f}s")

print(f"\nTraining completed in {time.time()-t_all:.1f}s.")

# 6. Evaluation on Held-Out Kelmarsh Test Set (Years >= 2022)
print("\n" + "="*55)
print("EVALUATING KELMARSH PINN ON TEST SET (Years >= 2022)")
print("="*55)
model.eval()

preds_mu, preds_sigma = [], []
with torch.no_grad():
    for bx, by, bws in test_loader:
        bx, bws = bx.to(device), bws.to(device)
        mu, sigma = model(bx, bws)
        preds_mu.append(mu.cpu().numpy())
        preds_sigma.append(sigma.cpu().numpy())

preds_mu = np.concatenate(preds_mu)
preds_sigma = np.concatenate(preds_sigma)

# Metrics
e_te = y_te - preds_mu
rmse_kel = np.sqrt(mean_squared_error(y_te, preds_mu))
mae_kel = mean_absolute_error(y_te, preds_mu)
r2_kel = r2_score(y_te, preds_mu)

z = np.abs(e_te) / preds_sigma
nll_kel = np.mean(np.log(preds_sigma) + 0.5 * (e_te / preds_sigma) ** 2) + 0.5 * np.log(2 * np.pi)
c1 = (z < 1.0).mean()
c2 = (z < 2.0).mean()

print(f"\n--- Comparative Performance Report (Kelmarsh Test Set) ---")
report = pd.DataFrame({
    "Model": ["Binned Curve (Lookup)", "Baseline HGB (Trees)", "PINN Optimized (Kelmarsh)"],
    "Test RMSE (kW)": [74.40, 60.10, round(rmse_kel, 2)],
    "Test MAE (kW)": ["-", 41.10, round(mae_kel, 2)],
    "Test R²": [0.9845, 0.9899, round(r2_kel, 4)],
    "Test NLL": ["-", 5.2996, round(nll_kel, 4)],
    "1-sigma Cov": ["-", 0.6874, round(c1, 4)],
    "2-sigma Cov": ["-", 0.9425, round(c2, 4)]
})
print(report.to_string(index=False))

# 7. Save and Upload Checkpoint
os.makedirs(f"{W}/models", exist_ok=True)
ckpt_path = f"{W}/models/pinn_m1_kelmarsh.pt"

torch.save({
    "model_state_dict": model.state_dict(),
    "scaler_mean": scaler_X.mean_,
    "scaler_scale": scaler_X.scale_,
    "input_cols": INPUT_COLS,
    "metrics": {"rmse": rmse_kel, "nll": nll_kel, "r2": r2_kel}
}, ckpt_path)

print(f"\nSaved checkpoint: {ckpt_path} ({os.path.getsize(ckpt_path) / 1e3:.1f} KB)")
api.upload_file(
    path_or_fileobj=ckpt_path,
    path_in_repo="models/pinn_m1_kelmarsh.pt",
    repo_id=REPO,
    repo_type="dataset",
    commit_message=f"Add optimized Kelmarsh PINN M1 checkpoint (RMSE {rmse_kel:.2f} kW)"
)
print("Uploaded Kelmarsh model checkpoint to Hugging Face successfully.")

In [ ]:
import os
import numpy as np, pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

W = "/kaggle/working"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
rated = 2050.0

# 1. Load Checkpoints (weights_only=False to unpickle numpy arrays)
pen_ckpt = torch.load(f"{W}/models/pinn_m1_penmanshiel.pt", map_location=device, weights_only=False)
kel_ckpt = torch.load(f"{W}/models/pinn_m1_kelmarsh.pt", map_location=device, weights_only=False)

# 2. Architecture Definitions
class ResBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.fc1 = nn.Linear(dim, dim); self.ln1 = nn.LayerNorm(dim); self.act = nn.SiLU()
        self.fc2 = nn.Linear(dim, dim); self.ln2 = nn.LayerNorm(dim)
    def forward(self, x): return self.act(x + self.ln2(self.fc2(self.act(self.ln1(self.fc1(x))))))

class SmoothPowerPrior(nn.Module):
    def __init__(self, rated_power=2050.0):
        super().__init__()
        self.rated = rated_power
        self.u_mid = nn.Parameter(torch.tensor(8.5)); self.u_scale = nn.Parameter(torch.tensor(1.8)); self.cut_in = nn.Parameter(torch.tensor(3.0))
    def forward(self, ws):
        s = torch.sigmoid((ws - self.u_mid) / torch.clamp(self.u_scale, min=0.5))
        return self.rated * s * torch.sigmoid((ws - self.cut_in) * 3.0)

class OptimizedWindPINN(nn.Module):
    def __init__(self, num_inputs=12, hidden_dim=128, rated_power=2050.0):
        super().__init__()
        self.rated = rated_power; self.prior = SmoothPowerPrior(rated_power)
        self.in_proj = nn.Sequential(nn.Linear(num_inputs, hidden_dim), nn.LayerNorm(hidden_dim), nn.SiLU())
        self.res1 = ResBlock(hidden_dim); self.res2 = ResBlock(hidden_dim)
        self.out_proj = nn.Sequential(nn.Linear(hidden_dim, 64), nn.LayerNorm(64), nn.SiLU())
        self.head_mu = nn.Linear(64, 1); self.head_log_var = nn.Linear(64, 1)
    def forward(self, x, ws):
        p_base = self.prior(ws.unsqueeze(-1))
        feat = self.out_proj(self.res2(self.res1(self.in_proj(x))))
        mu = torch.clamp(p_base + self.head_mu(feat), min=0.0, max=self.rated * 1.05)
        sigma = torch.clamp(torch.exp(0.5 * self.head_log_var(feat)), min=3.0)
        return mu.squeeze(-1), sigma.squeeze(-1)

def load_pinn(ckpt):
    m = OptimizedWindPINN(num_inputs=len(ckpt["input_cols"]), rated_power=rated).to(device)
    m.load_state_dict(ckpt["model_state_dict"]); m.eval()
    mean_f32 = ckpt["scaler_mean"].astype(np.float32)
    scale_f32 = ckpt["scaler_scale"].astype(np.float32)
    return m, mean_f32, scale_f32, ckpt["input_cols"]

pen_model, pen_mean, pen_scale, cols = load_pinn(pen_ckpt)
kel_model, kel_mean, kel_scale, _ = load_pinn(kel_ckpt)

# 3. Diagnostic 1: Power Curves vs Wind Speed
ws_grid = np.linspace(2.0, 22.0, 200, dtype=np.float32)

def sweep_ws(model, mean, scale):
    x_synth = np.zeros((len(ws_grid), len(cols)), dtype=np.float32)
    x_synth[:, 0] = ws_grid                          # ws
    x_synth[:, 1] = 0.0; x_synth[:, 2] = 1.0         # south wind (cos=1)
    x_synth[:, 3] = 0.0; x_synth[:, 4] = 0.0         # 0 vane error
    x_synth[:, 5] = 10.0                             # 10 deg C
    x_synth[:, 6] = 0.10                             # 10% TI
    x_norm = ((x_synth - mean) / scale).astype(np.float32)
    
    with torch.no_grad():
        x_tensor = torch.tensor(x_norm, dtype=torch.float32).to(device)
        ws_tensor = torch.tensor(ws_grid, dtype=torch.float32).to(device)
        mu, sig = model(x_tensor, ws_tensor)
    return mu.cpu().numpy(), sig.cpu().numpy()

pen_mu, pen_sig = sweep_ws(pen_model, pen_mean, pen_scale)
kel_mu, kel_sig = sweep_ws(kel_model, kel_mean, kel_scale)

# 4. Diagnostic 2: Yaw Sensitivity (At 8 m/s, sweep vane -30 to +30 deg)
vane_grid = np.linspace(-30.0, 30.0, 100, dtype=np.float32)

def sweep_vane(model, mean, scale, ws_fixed=8.0):
    x_synth = np.zeros((len(vane_grid), len(cols)), dtype=np.float32)
    x_synth[:, 0] = ws_fixed
    x_synth[:, 1] = 0.0; x_synth[:, 2] = 1.0
    x_synth[:, 3] = vane_grid                        # signed vane
    x_synth[:, 4] = np.abs(vane_grid)                # abs vane
    x_synth[:, 5] = 10.0
    x_synth[:, 6] = 0.10
    x_norm = ((x_synth - mean) / scale).astype(np.float32)
    
    with torch.no_grad():
        x_tensor = torch.tensor(x_norm, dtype=torch.float32).to(device)
        ws_tensor = torch.full((len(vane_grid),), ws_fixed, dtype=torch.float32).to(device)
        mu, _ = model(x_tensor, ws_tensor)
    return mu.cpu().numpy()

pen_yaw = sweep_vane(pen_model, pen_mean, pen_scale, ws_fixed=8.0)
kel_yaw = sweep_vane(kel_model, kel_mean, kel_scale, ws_fixed=8.0)

# 5. Generate Diagnostic Plot
fig, axes = plt.subplots(1, 3, figsize=(20, 5.5))

# Plot A: Learned Power Curve & 2-sigma band
axes[0].plot(ws_grid, pen_mu, "b-", lw=2.5, label="Penmanshiel PINN (MM82, D=82m)")
axes[0].fill_between(ws_grid, pen_mu - 2*pen_sig, np.minimum(pen_mu + 2*pen_sig, rated*1.05), color="b", alpha=0.15, label="Penmanshiel ±2σ Band")
axes[0].plot(ws_grid, kel_mu, "r-", lw=2.5, label="Kelmarsh PINN (MM92, D=92m)")
axes[0].fill_between(ws_grid, kel_mu - 2*kel_sig, np.minimum(kel_mu + 2*kel_sig, rated*1.05), color="r", alpha=0.15, label="Kelmarsh ±2σ Band")
axes[0].axhline(rated, color="k", ls="--", alpha=0.6, label="Rated Power (2,050 kW)")
axes[0].set_xlabel("Wind Speed (m/s)"); axes[0].set_ylabel("Power (kW)")
axes[0].set_title("A: Learned Aerodynamic Curves & ±2σ Spread")
axes[0].legend(fontsize=8, loc="lower right"); axes[0].grid(True, alpha=0.3)

# Plot B: Swept Area Ratio (Kelmarsh / Penmanshiel)
ratio = kel_mu / np.maximum(pen_mu, 10.0)
mask_mid = (ws_grid >= 4.0) & (ws_grid <= 14.0)
axes[1].plot(ws_grid[mask_mid], ratio[mask_mid], "m-", lw=2.5)
axes[1].axhline((92/82)**2, color="k", ls="--", label="Theoretical Swept Area Ratio (1.26)")
axes[1].set_xlabel("Wind Speed (m/s)"); axes[1].set_ylabel("Power Ratio (Kelmarsh / Penmanshiel)")
axes[1].set_title("B: Kelmarsh MM92 vs Penmanshiel MM82 Power Ratio")
axes[1].legend(fontsize=9); axes[1].grid(True, alpha=0.3)

# Plot C: Learned Yaw Error Sensitivity
axes[2].plot(vane_grid, pen_yaw, "b-", lw=2.5, label="Penmanshiel (ws = 8 m/s)")
axes[2].plot(vane_grid, kel_yaw, "r-", lw=2.5, label="Kelmarsh (ws = 8 m/s)")
axes[2].set_xlabel("Vane Yaw Misalignment (°)"); axes[2].set_ylabel("Predicted Power (kW)")
axes[2].set_title("C: Learned Yaw Sensitivity ∂P/∂γ (Model 3 Cost Profile)")
axes[2].legend(fontsize=9); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
os.makedirs(f"{W}/figs", exist_ok=True)
out_fig = f"{W}/figs/pinn_physical_diagnostics.png"
plt.savefig(out_fig, dpi=120)
plt.show()
print(f"Diagnostics figure saved to: {out_fig}")

IMprovement 1 Kelmarsh

In [ ]:
import os, time
import numpy as np, pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from huggingface_hub import HfApi, login
from kaggle_secrets import UserSecretsClient

# 1. Environment & Auth
login(UserSecretsClient().get_secret("HF_TOKEN"))
api = HfApi()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")

W = "/kaggle/working"
REPO = "B4BUK4/MPVI_project"
TS = "Date and time"
rated = 2050.0

# Choose farm to test: 'kelmarsh' (proves the fix) or 'penmanshiel'
FARM = "kelmarsh"

# 2. Physics Helper: IEC 61400-12 Air Density Normalization
def compute_u_iec(ws, temp, elev):
    """
    Computes IEC 61400-12 standard equivalent wind speed using
    barometric lapse rate and ideal gas air density.
    """
    # Pressure at elevation z (Pa)
    p_z = 101325.0 * (1.0 - 0.0065 * elev / 288.15) ** 5.25588
    # Air density (kg/m^3)
    rho = p_z / (287.05 * (temp + 273.15))
    # Equivalent normalized wind speed (reference rho_0 = 1.225 kg/m^3)
    return ws * (rho / 1.225) ** (1.0 / 3.0)

# 3. Load Dataset & Precompute IEC Normalization
df = pd.read_parquet(f"{W}/parquet/train/{FARM}_m1_clean.parquet")
df = df[df["ws"].between(3.0, 20.0)].reset_index(drop=True)

# Compute IEC wind speed column
df["u_iec"] = compute_u_iec(df["ws"].values, df["temp"].values, df["elev"].values).astype(np.float32)

tr_df = df[~df["is_test"]].copy()
te_df = df[df["is_test"]].copy()

SAMPLE_SIZE = 800_000 if FARM == "kelmarsh" else 1_200_000
tr_sample = tr_df.sample(min(SAMPLE_SIZE, len(tr_df)), random_state=42).reset_index(drop=True)

print(f"[{FARM.upper()} TIER 1] Train samples: {len(tr_sample):,} | Test samples: {len(te_df):,}")

# Feature columns
INPUT_COLS = [
    "ws", "sin_wd", "cos_wd", "vane", "abs_vane", "temp", "TI",
    "x", "y", "elev", "month_sin", "month_cos"
]
TARGET_COL = "power_kw"

# Normalization
scaler_X = StandardScaler()
X_tr = scaler_X.fit_transform(tr_sample[INPUT_COLS]).astype(np.float32)
X_te = scaler_X.transform(te_df[INPUT_COLS]).astype(np.float32)

y_tr = tr_sample[TARGET_COL].values.astype(np.float32)
y_te = te_df[TARGET_COL].values.astype(np.float32)

# Unnormalized physical tensors for prior
u_iec_tr = tr_sample["u_iec"].values.astype(np.float32)
u_iec_te = te_df["u_iec"].values.astype(np.float32)

vane_tr = tr_sample["vane"].values.astype(np.float32)
vane_te = te_df["vane"].values.astype(np.float32)

train_dataset = TensorDataset(torch.tensor(X_tr), torch.tensor(y_tr), torch.tensor(u_iec_tr), torch.tensor(vane_tr))
test_dataset = TensorDataset(torch.tensor(X_te), torch.tensor(y_te), torch.tensor(u_iec_te), torch.tensor(vane_te))

train_loader = DataLoader(train_dataset, batch_size=2048, shuffle=True, drop_last=True)
test_loader = DataLoader(test_dataset, batch_size=4096, shuffle=False)

# 4. Tier 1 PINN Architecture
class ResBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.fc1 = nn.Linear(dim, dim); self.ln1 = nn.LayerNorm(dim); self.act = nn.SiLU()
        self.fc2 = nn.Linear(dim, dim); self.ln2 = nn.LayerNorm(dim)
    def forward(self, x): return self.act(x + self.ln2(self.fc2(self.act(self.ln1(self.fc1(x))))))

class AerodynamicPriorWithYaw(nn.Module):
    """Aerodynamic prior operating on u_iec with analytical cos^p(gamma) yaw loss"""
    def __init__(self, rated_power=2050.0):
        super().__init__()
        self.rated = rated_power
        self.u_mid = nn.Parameter(torch.tensor(8.2 if FARM == "kelmarsh" else 8.5))
        self.u_scale = nn.Parameter(torch.tensor(1.8))
        self.cut_in = nn.Parameter(torch.tensor(3.0))
        # Learnable yaw-loss exponent p in [1.8, 2.8], initialized at ~2.2
        self.p_raw = nn.Parameter(torch.tensor(0.0))

    def forward(self, u_iec, vane_deg):
        # Aerodynamic power curve on density-normalized wind speed
        s = torch.sigmoid((u_iec - self.u_mid) / torch.clamp(self.u_scale, min=0.5))
        cut_in_gate = torch.sigmoid((u_iec - self.cut_in) * 3.0)
        p_curve = self.rated * s * cut_in_gate
        
        # Physical cosine yaw loss
        vane_rad = torch.deg2rad(vane_deg)
        p_yaw = 1.8 + 1.0 * torch.sigmoid(self.p_raw)
        yaw_factor = torch.clamp(torch.cos(vane_rad), min=0.5) ** p_yaw
        return p_curve * yaw_factor

class Tier1PINN(nn.Module):
    def __init__(self, num_inputs=12, hidden_dim=128, rated_power=2050.0):
        super().__init__()
        self.rated = rated_power
        self.prior = AerodynamicPriorWithYaw(rated_power)

        self.in_proj = nn.Sequential(nn.Linear(num_inputs, hidden_dim), nn.LayerNorm(hidden_dim), nn.SiLU())
        self.res1 = ResBlock(hidden_dim)
        self.res2 = ResBlock(hidden_dim)
        self.out_proj = nn.Sequential(nn.Linear(hidden_dim, 64), nn.LayerNorm(64), nn.SiLU())

        self.head_mu = nn.Linear(64, 1)
        self.head_log_var = nn.Linear(64, 1)

        nn.init.zeros_(self.head_mu.weight)
        nn.init.zeros_(self.head_mu.bias)
        initial_sigma = 49.0 if FARM == "kelmarsh" else 40.0
        nn.init.constant_(self.head_log_var.bias, np.log(initial_sigma**2))

    def forward(self, x, u_iec, vane_deg):
        p_base = self.prior(u_iec.unsqueeze(-1), vane_deg.unsqueeze(-1))
        feat = self.out_proj(self.res2(self.res1(self.in_proj(x))))

        delta_mu = self.head_mu(feat)
        log_var = self.head_log_var(feat)

        mu = torch.clamp(p_base + delta_mu, min=0.0, max=self.rated * 1.05)
        sigma = torch.clamp(torch.exp(0.5 * log_var), min=3.0)
        return p_base.squeeze(-1), mu.squeeze(-1), sigma.squeeze(-1)

# 5. Training Setup
model = Tier1PINN(num_inputs=len(INPUT_COLS), hidden_dim=128, rated_power=rated).to(device)
optimizer = optim.AdamW(model.parameters(), lr=1.5e-3, weight_decay=1e-4)

EPOCHS = 14
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)
LAMBDA_MONO = 10.0

print(f"\nStarting Tier 1 PINN Training on {FARM.upper()} ({EPOCHS} Epochs)...")
t_all = time.time()

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss, total_nll, total_mono = 0.0, 0.0, 0.0
    t_start = time.time()

    for bx, by, bu_iec, bvane in train_loader:
        bx, by = bx.to(device), by.to(device)
        bu_iec, bvane = bu_iec.to(device), bvane.to(device)
        bx.requires_grad_(True)

        optimizer.zero_grad()
        p_base, mu, sigma = model(bx, bu_iec, bvane)

        # 1. NLL Loss
        nll_loss = torch.mean(torch.log(sigma) + 0.5 * ((by - mu) / sigma) ** 2)

        # 2. ADAPTIVE Monotonicity Loss: Only penalize when p_base < 92% of rated power
        grad_ws = torch.autograd.grad(
            outputs=mu.sum(),
            inputs=bx,
            create_graph=True,
            retain_graph=True
        )[0][:, 0]

        # Automatically shuts off when the turbine enters rated plateau!
        reg2_mask = (p_base < 0.92 * rated)
        mono_violations = torch.relu(-grad_ws) * reg2_mask.float()
        mono_loss = LAMBDA_MONO * torch.mean(mono_violations ** 2)

        loss = nll_loss + mono_loss
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
        optimizer.step()

        total_loss += loss.item() * len(by)
        total_nll += nll_loss.item() * len(by)
        total_mono += mono_loss.item() * len(by)

    scheduler.step()
    n_samples = len(train_loader.dataset)
    current_lr = scheduler.get_last_lr()[0]
    print(f"Epoch {epoch:02d}/{EPOCHS} | Train NLL: {total_nll/n_samples:.4f} | Mono Pen: {total_mono/n_samples:.5f} | lr: {current_lr:.6f} | {time.time()-t_start:.1f}s")

print(f"\nTraining completed in {time.time()-t_all:.1f}s.")

# 6. Evaluation on Held-Out Test Set
print("\n" + "="*55)
print(f"EVALUATING TIER 1 PINN ON {FARM.upper()} TEST SET (Years >= 2022)")
print("="*55)
model.eval()

preds_mu, preds_sigma = [], []
with torch.no_grad():
    for bx, by, bu_iec, bvane in test_loader:
        bx = bx.to(device)
        bu_iec, bvane = bu_iec.to(device), bvane.to(device)
        _, mu, sigma = model(bx, bu_iec, bvane)
        preds_mu.append(mu.cpu().numpy())
        preds_sigma.append(sigma.cpu().numpy())

preds_mu = np.concatenate(preds_mu)
preds_sigma = np.concatenate(preds_sigma)

# Metrics
e_te = y_te - preds_mu
rmse_t1 = np.sqrt(mean_squared_error(y_te, preds_mu))
mae_t1 = mean_absolute_error(y_te, preds_mu)
r2_t1 = r2_score(y_te, preds_mu)

z = np.abs(e_te) / preds_sigma
nll_t1 = np.mean(np.log(preds_sigma) + 0.5 * (e_te / preds_sigma) ** 2) + 0.5 * np.log(2 * np.pi)
c1 = (z < 1.0).mean()
c2 = (z < 2.0).mean()

# Reference values for Kelmarsh
prev_pinn = 62.29 if FARM == "kelmarsh" else 47.12
prev_hgb = 60.10 if FARM == "kelmarsh" else 49.80

print(f"\n--- Comparative Performance Report ({FARM.upper()}) ---")
report = pd.DataFrame({
    "Model": ["Baseline HGB (Trees)", "PINN Previous", "PINN Tier 1 (Upgraded)"],
    "Test RMSE (kW)": [prev_hgb, prev_pinn, round(rmse_t1, 2)],
    "Test MAE (kW)": [41.10 if FARM == "kelmarsh" else 33.70, 42.81 if FARM == "kelmarsh" else 31.92, round(mae_t1, 2)],
    "Test R²": [0.9899 if FARM == "kelmarsh" else 0.9944, 0.9891 if FARM == "kelmarsh" else 0.9950, round(r2_t1, 4)],
    "Test NLL": [5.2996 if FARM == "kelmarsh" else 5.0070, 5.3051 if FARM == "kelmarsh" else 4.9652, round(nll_t1, 4)],
    "1-sigma Cov": ["-", "-", round(c1, 4)],
    "2-sigma Cov": ["-", "-", round(c2, 4)]
})
print(report.to_string(index=False))

# 7. Save Checkpoint
os.makedirs(f"{W}/models", exist_ok=True)
ckpt_path = f"{W}/models/pinn_tier1_{FARM}.pt"

torch.save({
    "model_state_dict": model.state_dict(),
    "scaler_mean": scaler_X.mean_,
    "scaler_scale": scaler_X.scale_,
    "input_cols": INPUT_COLS,
    "metrics": {"rmse": rmse_t1, "nll": nll_t1, "r2": r2_t1}
}, ckpt_path)

print(f"\nSaved checkpoint: {ckpt_path}")

In [ ]:
import os, time
import numpy as np, pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from huggingface_hub import HfApi, login
from kaggle_secrets import UserSecretsClient

# 1. Environment & Auth
login(UserSecretsClient().get_secret("HF_TOKEN"))
api = HfApi()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")

W = "/kaggle/working"
REPO = "B4BUK4/MPVI_project"
TS = "Date and time"
rated = 2050.0

# Choose farm to test: 'kelmarsh' (proves the fix) or 'penmanshiel'
FARM = "penmanshiel"

# 2. Physics Helper: IEC 61400-12 Air Density Normalization
def compute_u_iec(ws, temp, elev):
    """
    Computes IEC 61400-12 standard equivalent wind speed using
    barometric lapse rate and ideal gas air density.
    """
    # Pressure at elevation z (Pa)
    p_z = 101325.0 * (1.0 - 0.0065 * elev / 288.15) ** 5.25588
    # Air density (kg/m^3)
    rho = p_z / (287.05 * (temp + 273.15))
    # Equivalent normalized wind speed (reference rho_0 = 1.225 kg/m^3)
    return ws * (rho / 1.225) ** (1.0 / 3.0)

# 3. Load Dataset & Precompute IEC Normalization
df = pd.read_parquet(f"{W}/parquet/train/{FARM}_m1_clean.parquet")
df = df[df["ws"].between(3.0, 20.0)].reset_index(drop=True)

# Compute IEC wind speed column
df["u_iec"] = compute_u_iec(df["ws"].values, df["temp"].values, df["elev"].values).astype(np.float32)

tr_df = df[~df["is_test"]].copy()
te_df = df[df["is_test"]].copy()

SAMPLE_SIZE = 800_000 if FARM == "kelmarsh" else 1_200_000
tr_sample = tr_df.sample(min(SAMPLE_SIZE, len(tr_df)), random_state=42).reset_index(drop=True)

print(f"[{FARM.upper()} TIER 1] Train samples: {len(tr_sample):,} | Test samples: {len(te_df):,}")

# Feature columns
INPUT_COLS = [
    "ws", "sin_wd", "cos_wd", "vane", "abs_vane", "temp", "TI",
    "x", "y", "elev", "month_sin", "month_cos"
]
TARGET_COL = "power_kw"

# Normalization
scaler_X = StandardScaler()
X_tr = scaler_X.fit_transform(tr_sample[INPUT_COLS]).astype(np.float32)
X_te = scaler_X.transform(te_df[INPUT_COLS]).astype(np.float32)

y_tr = tr_sample[TARGET_COL].values.astype(np.float32)
y_te = te_df[TARGET_COL].values.astype(np.float32)

# Unnormalized physical tensors for prior
u_iec_tr = tr_sample["u_iec"].values.astype(np.float32)
u_iec_te = te_df["u_iec"].values.astype(np.float32)

vane_tr = tr_sample["vane"].values.astype(np.float32)
vane_te = te_df["vane"].values.astype(np.float32)

train_dataset = TensorDataset(torch.tensor(X_tr), torch.tensor(y_tr), torch.tensor(u_iec_tr), torch.tensor(vane_tr))
test_dataset = TensorDataset(torch.tensor(X_te), torch.tensor(y_te), torch.tensor(u_iec_te), torch.tensor(vane_te))

train_loader = DataLoader(train_dataset, batch_size=2048, shuffle=True, drop_last=True)
test_loader = DataLoader(test_dataset, batch_size=4096, shuffle=False)

# 4. Tier 1 PINN Architecture
class ResBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.fc1 = nn.Linear(dim, dim); self.ln1 = nn.LayerNorm(dim); self.act = nn.SiLU()
        self.fc2 = nn.Linear(dim, dim); self.ln2 = nn.LayerNorm(dim)
    def forward(self, x): return self.act(x + self.ln2(self.fc2(self.act(self.ln1(self.fc1(x))))))

class AerodynamicPriorWithYaw(nn.Module):
    """Aerodynamic prior operating on u_iec with analytical cos^p(gamma) yaw loss"""
    def __init__(self, rated_power=2050.0):
        super().__init__()
        self.rated = rated_power
        self.u_mid = nn.Parameter(torch.tensor(8.2 if FARM == "kelmarsh" else 8.5))
        self.u_scale = nn.Parameter(torch.tensor(1.8))
        self.cut_in = nn.Parameter(torch.tensor(3.0))
        # Learnable yaw-loss exponent p in [1.8, 2.8], initialized at ~2.2
        self.p_raw = nn.Parameter(torch.tensor(0.0))

    def forward(self, u_iec, vane_deg):
        # Aerodynamic power curve on density-normalized wind speed
        s = torch.sigmoid((u_iec - self.u_mid) / torch.clamp(self.u_scale, min=0.5))
        cut_in_gate = torch.sigmoid((u_iec - self.cut_in) * 3.0)
        p_curve = self.rated * s * cut_in_gate
        
        # Physical cosine yaw loss
        vane_rad = torch.deg2rad(vane_deg)
        p_yaw = 1.8 + 1.0 * torch.sigmoid(self.p_raw)
        yaw_factor = torch.clamp(torch.cos(vane_rad), min=0.5) ** p_yaw
        return p_curve * yaw_factor

class Tier1PINN(nn.Module):
    def __init__(self, num_inputs=12, hidden_dim=128, rated_power=2050.0):
        super().__init__()
        self.rated = rated_power
        self.prior = AerodynamicPriorWithYaw(rated_power)

        self.in_proj = nn.Sequential(nn.Linear(num_inputs, hidden_dim), nn.LayerNorm(hidden_dim), nn.SiLU())
        self.res1 = ResBlock(hidden_dim)
        self.res2 = ResBlock(hidden_dim)
        self.out_proj = nn.Sequential(nn.Linear(hidden_dim, 64), nn.LayerNorm(64), nn.SiLU())

        self.head_mu = nn.Linear(64, 1)
        self.head_log_var = nn.Linear(64, 1)

        nn.init.zeros_(self.head_mu.weight)
        nn.init.zeros_(self.head_mu.bias)
        initial_sigma = 49.0 if FARM == "kelmarsh" else 40.0
        nn.init.constant_(self.head_log_var.bias, np.log(initial_sigma**2))

    def forward(self, x, u_iec, vane_deg):
        p_base = self.prior(u_iec.unsqueeze(-1), vane_deg.unsqueeze(-1))
        feat = self.out_proj(self.res2(self.res1(self.in_proj(x))))

        delta_mu = self.head_mu(feat)
        log_var = self.head_log_var(feat)

        mu = torch.clamp(p_base + delta_mu, min=0.0, max=self.rated * 1.05)
        sigma = torch.clamp(torch.exp(0.5 * log_var), min=3.0)
        return p_base.squeeze(-1), mu.squeeze(-1), sigma.squeeze(-1)

# 5. Training Setup
model = Tier1PINN(num_inputs=len(INPUT_COLS), hidden_dim=128, rated_power=rated).to(device)
optimizer = optim.AdamW(model.parameters(), lr=1.5e-3, weight_decay=1e-4)

EPOCHS = 25
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)
LAMBDA_MONO = 10.0

print(f"\nStarting Tier 1 PINN Training on {FARM.upper()} ({EPOCHS} Epochs)...")
t_all = time.time()

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss, total_nll, total_mono = 0.0, 0.0, 0.0
    t_start = time.time()

    for bx, by, bu_iec, bvane in train_loader:
        bx, by = bx.to(device), by.to(device)
        bu_iec, bvane = bu_iec.to(device), bvane.to(device)
        bx.requires_grad_(True)

        optimizer.zero_grad()
        p_base, mu, sigma = model(bx, bu_iec, bvane)

        # 1. NLL Loss
        nll_loss = torch.mean(torch.log(sigma) + 0.5 * ((by - mu) / sigma) ** 2)

        # 2. ADAPTIVE Monotonicity Loss: Only penalize when p_base < 92% of rated power
        grad_ws = torch.autograd.grad(
            outputs=mu.sum(),
            inputs=bx,
            create_graph=True,
            retain_graph=True
        )[0][:, 0]

        # Automatically shuts off when the turbine enters rated plateau!
        reg2_mask = (p_base < 0.92 * rated)
        mono_violations = torch.relu(-grad_ws) * reg2_mask.float()
        mono_loss = LAMBDA_MONO * torch.mean(mono_violations ** 2)

        loss = nll_loss + mono_loss
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
        optimizer.step()

        total_loss += loss.item() * len(by)
        total_nll += nll_loss.item() * len(by)
        total_mono += mono_loss.item() * len(by)

    scheduler.step()
    n_samples = len(train_loader.dataset)
    current_lr = scheduler.get_last_lr()[0]
    print(f"Epoch {epoch:02d}/{EPOCHS} | Train NLL: {total_nll/n_samples:.4f} | Mono Pen: {total_mono/n_samples:.5f} | lr: {current_lr:.6f} | {time.time()-t_start:.1f}s")

print(f"\nTraining completed in {time.time()-t_all:.1f}s.")

# 6. Evaluation on Held-Out Test Set
print("\n" + "="*55)
print(f"EVALUATING TIER 1 PINN ON {FARM.upper()} TEST SET (Years >= 2022)")
print("="*55)
model.eval()

preds_mu, preds_sigma = [], []
with torch.no_grad():
    for bx, by, bu_iec, bvane in test_loader:
        bx = bx.to(device)
        bu_iec, bvane = bu_iec.to(device), bvane.to(device)
        _, mu, sigma = model(bx, bu_iec, bvane)
        preds_mu.append(mu.cpu().numpy())
        preds_sigma.append(sigma.cpu().numpy())

preds_mu = np.concatenate(preds_mu)
preds_sigma = np.concatenate(preds_sigma)

# Metrics
e_te = y_te - preds_mu
rmse_t1 = np.sqrt(mean_squared_error(y_te, preds_mu))
mae_t1 = mean_absolute_error(y_te, preds_mu)
r2_t1 = r2_score(y_te, preds_mu)

z = np.abs(e_te) / preds_sigma
nll_t1 = np.mean(np.log(preds_sigma) + 0.5 * (e_te / preds_sigma) ** 2) + 0.5 * np.log(2 * np.pi)
c1 = (z < 1.0).mean()
c2 = (z < 2.0).mean()

# Reference values for Kelmarsh
prev_pinn = 62.29 if FARM == "kelmarsh" else 47.12
prev_hgb = 60.10 if FARM == "kelmarsh" else 49.80

print(f"\n--- Comparative Performance Report ({FARM.upper()}) ---")
report = pd.DataFrame({
    "Model": ["Baseline HGB (Trees)", "PINN Previous", "PINN Tier 1 (Upgraded)"],
    "Test RMSE (kW)": [prev_hgb, prev_pinn, round(rmse_t1, 2)],
    "Test MAE (kW)": [41.10 if FARM == "kelmarsh" else 33.70, 42.81 if FARM == "kelmarsh" else 31.92, round(mae_t1, 2)],
    "Test R²": [0.9899 if FARM == "kelmarsh" else 0.9944, 0.9891 if FARM == "kelmarsh" else 0.9950, round(r2_t1, 4)],
    "Test NLL": [5.2996 if FARM == "kelmarsh" else 5.0070, 5.3051 if FARM == "kelmarsh" else 4.9652, round(nll_t1, 4)],
    "1-sigma Cov": ["-", "-", round(c1, 4)],
    "2-sigma Cov": ["-", "-", round(c2, 4)]
})
print(report.to_string(index=False))

# 7. Save Checkpoint
os.makedirs(f"{W}/models", exist_ok=True)
ckpt_path = f"{W}/models/pinn_tier1_{FARM}.pt"

torch.save({
    "model_state_dict": model.state_dict(),
    "scaler_mean": scaler_X.mean_,
    "scaler_scale": scaler_X.scale_,
    "input_cols": INPUT_COLS,
    "metrics": {"rmse": rmse_t1, "nll": nll_t1, "r2": r2_t1}
}, ckpt_path)

print(f"\nSaved checkpoint: {ckpt_path}")

New architecture

In [ ]:
import os, time
import numpy as np, pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from huggingface_hub import HfApi, login
from kaggle_secrets import UserSecretsClient

# 1. Environment & Auth
login(UserSecretsClient().get_secret("HF_TOKEN"))
api = HfApi()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")

W = "/kaggle/working"
REPO = "B4BUK4/MPVI_project"
TS = "Date and time"
rated = 2050.0

# Choose farm: 'penmanshiel' (benchmark 46.79 kW) or 'kelmarsh' (benchmark 61.55 kW)
FARM = "penmanshiel"

# 2. Physics Helper: IEC 61400-12 Air Density Normalization
def compute_u_iec(ws, temp, elev):
    p_z = 101325.0 * (1.0 - 0.0065 * elev / 288.15) ** 5.25588
    rho = p_z / (287.05 * (temp + 273.15))
    return ws * (rho / 1.225) ** (1.0 / 3.0)

# 3. Load Clean Parquet Dataset
df = pd.read_parquet(f"{W}/parquet/train/{FARM}_m1_clean.parquet")
df = df[df["ws"].between(3.0, 20.0)].reset_index(drop=True)
df["u_iec"] = compute_u_iec(df["ws"].values, df["temp"].values, df["elev"].values).astype(np.float32)

tr_df = df[~df["is_test"]].copy()
te_df = df[df["is_test"]].copy()

SAMPLE_SIZE = 1_200_000 if FARM == "penmanshiel" else 800_000
tr_sample = tr_df.sample(min(SAMPLE_SIZE, len(tr_df)), random_state=42).reset_index(drop=True)

print(f"[{FARM.upper()} WIDE & DEEP PINN] Train: {len(tr_sample):,} | Test: {len(te_df):,}")

# Feature Set
INPUT_COLS = [
    "ws", "sin_wd", "cos_wd", "vane", "abs_vane", "temp", "TI",
    "x", "y", "elev", "month_sin", "month_cos"
]
TARGET_COL = "power_kw"

# Normalization
scaler_X = StandardScaler()
X_tr = scaler_X.fit_transform(tr_sample[INPUT_COLS]).astype(np.float32)
X_te = scaler_X.transform(te_df[INPUT_COLS]).astype(np.float32)

y_tr = tr_sample[TARGET_COL].values.astype(np.float32)
y_te = te_df[TARGET_COL].values.astype(np.float32)

u_iec_tr = tr_sample["u_iec"].values.astype(np.float32)
u_iec_te = te_df["u_iec"].values.astype(np.float32)

vane_tr = tr_sample["vane"].values.astype(np.float32)
vane_te = te_df["vane"].values.astype(np.float32)

train_dataset = TensorDataset(torch.tensor(X_tr), torch.tensor(y_tr), torch.tensor(u_iec_tr), torch.tensor(vane_tr))
test_dataset = TensorDataset(torch.tensor(X_te), torch.tensor(y_te), torch.tensor(u_iec_te), torch.tensor(vane_te))

train_loader = DataLoader(train_dataset, batch_size=2048, shuffle=True, drop_last=True)
test_loader = DataLoader(test_dataset, batch_size=4096, shuffle=False)

# 4. Wide & Deep PINN Architecture
class ResBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.fc1 = nn.Linear(dim, dim); self.ln1 = nn.LayerNorm(dim); self.act = nn.SiLU()
        self.fc2 = nn.Linear(dim, dim); self.ln2 = nn.LayerNorm(dim)
    def forward(self, x): return self.act(x + self.ln2(self.fc2(self.act(self.ln1(self.fc1(x))))))

class AerodynamicPriorWithYaw(nn.Module):
    def __init__(self, rated_power=2050.0):
        super().__init__()
        self.rated = rated_power
        self.u_mid = nn.Parameter(torch.tensor(8.5 if FARM == "penmanshiel" else 8.2))
        self.u_scale = nn.Parameter(torch.tensor(1.8))
        self.cut_in = nn.Parameter(torch.tensor(3.0))
        self.p_raw = nn.Parameter(torch.tensor(0.0))  # learnable p in [1.8, 2.8]

    def forward(self, u_iec, vane_deg):
        s = torch.sigmoid((u_iec - self.u_mid) / torch.clamp(self.u_scale, min=0.5))
        cut_in_gate = torch.sigmoid((u_iec - self.cut_in) * 3.0)
        p_curve = self.rated * s * cut_in_gate
        
        vane_rad = torch.deg2rad(vane_deg)
        p_yaw = 1.8 + 1.0 * torch.sigmoid(self.p_raw)
        yaw_factor = torch.clamp(torch.cos(vane_rad), min=0.5) ** p_yaw
        return p_curve * yaw_factor

class WideAndDeepPINN(nn.Module):
    def __init__(self, num_inputs=12, hidden_dim=128, rated_power=2050.0):
        super().__init__()
        self.rated = rated_power
        self.prior = AerodynamicPriorWithYaw(rated_power)

        # Path A: Wide Linear Bypass directly from inputs to residual
        self.wide_linear = nn.Linear(num_inputs, 1)

        # Path B: Deep ResNet Latent Pathway
        self.in_proj = nn.Sequential(nn.Linear(num_inputs, hidden_dim), nn.LayerNorm(hidden_dim), nn.SiLU())
        self.res1 = ResBlock(hidden_dim)
        self.res2 = ResBlock(hidden_dim)
        self.out_proj = nn.Sequential(nn.Linear(hidden_dim, 64), nn.LayerNorm(64), nn.SiLU())

        # Dual Heads
        self.deep_head_mu = nn.Linear(64, 1)
        self.head_log_var = nn.Linear(64, 1)

        # Initialize residual projections to zero (so prior dominates at step 0)
        nn.init.zeros_(self.wide_linear.weight); nn.init.zeros_(self.wide_linear.bias)
        nn.init.zeros_(self.deep_head_mu.weight); nn.init.zeros_(self.deep_head_mu.bias)
        initial_sigma = 40.0 if FARM == "penmanshiel" else 49.0
        nn.init.constant_(self.head_log_var.bias, np.log(initial_sigma**2))

    def forward(self, x, u_iec, vane_deg):
        # 1. Aerodynamic Prior Backbone
        p_base = self.prior(u_iec.unsqueeze(-1), vane_deg.unsqueeze(-1))

        # 2. Wide Linear Bypass
        delta_wide = self.wide_linear(x)

        # 3. Deep Latent ResNet Representation
        feat = self.out_proj(self.res2(self.res1(self.in_proj(x))))
        delta_deep = self.deep_head_mu(feat)
        log_var = self.head_log_var(feat)

        # 4. Total Residual = Linear Bypass + Deep Non-linear Representation
        delta_mu = delta_wide + delta_deep

        # 5. Output Clamping & Uncertainty
        mu = torch.clamp(p_base + delta_mu, min=0.0, max=self.rated * 1.05)
        sigma = torch.clamp(torch.exp(0.5 * log_var), min=3.0)
        return p_base.squeeze(-1), mu.squeeze(-1), sigma.squeeze(-1)

# 5. Training Setup
model = WideAndDeepPINN(num_inputs=len(INPUT_COLS), hidden_dim=128, rated_power=rated).to(device)
optimizer = optim.AdamW(model.parameters(), lr=1.5e-3, weight_decay=1e-4)

EPOCHS = 20
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)
LAMBDA_MONO = 10.0

print(f"\nStarting Wide & Deep PINN Training on {FARM.upper()} ({EPOCHS} Epochs)...")
t_all = time.time()

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss, total_nll, total_mono = 0.0, 0.0, 0.0
    t_start = time.time()

    for bx, by, bu_iec, bvane in train_loader:
        bx, by = bx.to(device), by.to(device)
        bu_iec, bvane = bu_iec.to(device), bvane.to(device)
        bx.requires_grad_(True)

        optimizer.zero_grad()
        p_base, mu, sigma = model(bx, bu_iec, bvane)

        # NLL Likelihood Loss
        nll_loss = torch.mean(torch.log(sigma) + 0.5 * ((by - mu) / sigma) ** 2)

        # Adaptive Monotonicity Loss: Only below 92% rated power
        grad_ws = torch.autograd.grad(
            outputs=mu.sum(),
            inputs=bx,
            create_graph=True,
            retain_graph=True
        )[0][:, 0]

        reg2_mask = (p_base < 0.92 * rated)
        mono_violations = torch.relu(-grad_ws) * reg2_mask.float()
        mono_loss = LAMBDA_MONO * torch.mean(mono_violations ** 2)

        loss = nll_loss + mono_loss
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
        optimizer.step()

        total_loss += loss.item() * len(by)
        total_nll += nll_loss.item() * len(by)
        total_mono += mono_loss.item() * len(by)

    scheduler.step()
    n_samples = len(train_loader.dataset)
    current_lr = scheduler.get_last_lr()[0]
    print(f"Epoch {epoch:02d}/{EPOCHS} | Train NLL: {total_nll/n_samples:.4f} | Mono Pen: {total_mono/n_samples:.5f} | lr: {current_lr:.6f} | {time.time()-t_start:.1f}s")

print(f"\nTraining completed in {time.time()-t_all:.1f}s.")

# 6. Evaluation on Held-Out Test Set (Years >= 2022)
print("\n" + "="*55)
print(f"EVALUATING WIDE & DEEP PINN ON {FARM.upper()} TEST SET (Years >= 2022)")
print("="*55)
model.eval()

preds_mu, preds_sigma = [], []
with torch.no_grad():
    for bx, by, bu_iec, bvane in test_loader:
        bx = bx.to(device)
        bu_iec, bvane = bu_iec.to(device), bvane.to(device)
        _, mu, sigma = model(bx, bu_iec, bvane)
        preds_mu.append(mu.cpu().numpy())
        preds_sigma.append(sigma.cpu().numpy())

preds_mu = np.concatenate(preds_mu)
preds_sigma = np.concatenate(preds_sigma)

# Metrics Calculation
e_te = y_te - preds_mu
rmse_wd = np.sqrt(mean_squared_error(y_te, preds_mu))
mae_wd = mean_absolute_error(y_te, preds_mu)
r2_wd = r2_score(y_te, preds_mu)

z = np.abs(e_te) / preds_sigma
nll_wd = np.mean(np.log(preds_sigma) + 0.5 * (e_te / preds_sigma) ** 2) + 0.5 * np.log(2 * np.pi)
c1 = (z < 1.0).mean()
c2 = (z < 2.0).mean()

# Reference values for Penmanshiel
prev_t1 = 46.79 if FARM == "penmanshiel" else 61.55
prev_hgb = 49.80 if FARM == "penmanshiel" else 60.10

print(f"\n--- Comparative Performance Report ({FARM.upper()}) ---")
report = pd.DataFrame({
    "Model": ["Baseline HGB (Trees)", "PINN Tier 1 (Previous Best)", "PINN Wide & Deep (Upgraded)"],
    "Test RMSE (kW)": [prev_hgb, prev_t1, round(rmse_wd, 2)],
    "Test MAE (kW)": [33.70 if FARM == "penmanshiel" else 41.10, 31.82 if FARM == "penmanshiel" else 42.28, round(mae_wd, 2)],
    "Test R²": [0.9944 if FARM == "penmanshiel" else 0.9899, 0.9950 if FARM == "penmanshiel" else 0.9894, round(r2_wd, 4)],
    "Test NLL": [5.0070 if FARM == "penmanshiel" else 5.2996, 4.9653 if FARM == "penmanshiel" else 5.2960, round(nll_wd, 4)],
    "1-sigma Cov": ["-", 0.6550 if FARM == "penmanshiel" else 0.6661, round(c1, 4)],
    "2-sigma Cov": ["-", 0.9279 if FARM == "penmanshiel" else 0.9356, round(c2, 4)]
})
print(report.to_string(index=False))

# 7. Save and Upload Checkpoint
os.makedirs(f"{W}/models", exist_ok=True)
ckpt_path = f"{W}/models/pinn_wide_deep_{FARM}.pt"

torch.save({
    "model_state_dict": model.state_dict(),
    "scaler_mean": scaler_X.mean_,
    "scaler_scale": scaler_X.scale_,
    "input_cols": INPUT_COLS,
    "metrics": {"rmse": rmse_wd, "nll": nll_wd, "r2": r2_wd}
}, ckpt_path)

print(f"\nSaved checkpoint: {ckpt_path}")
api.upload_file(
    path_or_fileobj=ckpt_path,
    path_in_repo=f"models/pinn_wide_deep_{FARM}.pt",
    repo_id=REPO,
    repo_type="dataset",
    commit_message=f"Save Wide & Deep PINN {FARM} checkpoint (RMSE {rmse_wd:.2f} kW)"
)
print("Uploaded to Hugging Face successfully.")

Final push

In [ ]:
import os, time
import numpy as np, pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from huggingface_hub import HfApi, login
from kaggle_secrets import UserSecretsClient

# 1. Environment & Auth
login(UserSecretsClient().get_secret("HF_TOKEN"))
api = HfApi()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")

W = "/kaggle/working"
REPO = "B4BUK4/MPVI_project"
TS = "Date and time"
rated = 2050.0

# Choose farm: 'penmanshiel' (best model: 46.79 kW) or 'kelmarsh' (best model: 61.55 kW)
FARM = "penmanshiel"

# 2. Physics Helper: IEC 61400-12 Air Density Normalization
def compute_u_iec(ws, temp, elev):
    p_z = 101325.0 * (1.0 - 0.0065 * elev / 288.15) ** 5.25588
    rho = p_z / (287.05 * (temp + 273.15))
    return ws * (rho / 1.225) ** (1.0 / 3.0)

# 3. Load Clean Parquet Dataset & Compute Engineered Features
df = pd.read_parquet(f"{W}/parquet/train/{FARM}_m1_clean.parquet")
df = df[df["ws"].between(3.0, 20.0)].reset_index(drop=True)

# A. IEC Density-normalized wind speed
df["u_iec"] = compute_u_iec(df["ws"].values, df["temp"].values, df["elev"].values).astype(np.float32)

# B. Diurnal Solar Harmonics (24-hour cycle)
hour = df[TS].dt.hour.values + df[TS].dt.minute.values / 60.0
hour_rad = 2.0 * np.pi * hour / 24.0
df["hour_sin"] = np.sin(hour_rad).astype(np.float32)
df["hour_cos"] = np.cos(hour_rad).astype(np.float32)

# C. Integer mapping for Turbine Entity Embeddings
unique_turbines = sorted(df["turbine"].unique())
turb_to_idx = {t: i for i, t in enumerate(unique_turbines)}
df["turb_idx"] = df["turbine"].map(turb_to_idx).astype(np.int64)
NUM_TURBINES = len(unique_turbines)

tr_df = df[~df["is_test"]].copy()
te_df = df[df["is_test"]].copy()

SAMPLE_SIZE = 1_200_000 if FARM == "penmanshiel" else 800_000
tr_sample = tr_df.sample(min(SAMPLE_SIZE, len(tr_df)), random_state=42).reset_index(drop=True)

print(f"[{FARM.upper()} ULTIMATE PINN] Train: {len(tr_sample):,} | Test: {len(te_df):,} | Turbines: {NUM_TURBINES}")

# Continuous Features (14 inputs: atmospheric, aeromechanics, spatial, annual & diurnal cycles)
INPUT_COLS = [
    "ws", "sin_wd", "cos_wd", "vane", "abs_vane", "temp", "TI",
    "x", "y", "elev", "month_sin", "month_cos", "hour_sin", "hour_cos"
]
TARGET_COL = "power_kw"

# Normalization of continuous features
scaler_X = StandardScaler()
X_tr = scaler_X.fit_transform(tr_sample[INPUT_COLS]).astype(np.float32)
X_te = scaler_X.transform(te_df[INPUT_COLS]).astype(np.float32)

y_tr = tr_sample[TARGET_COL].values.astype(np.float32)
y_te = te_df[TARGET_COL].values.astype(np.float32)

u_iec_tr = tr_sample["u_iec"].values.astype(np.float32)
u_iec_te = te_df["u_iec"].values.astype(np.float32)

vane_tr = tr_sample["vane"].values.astype(np.float32)
vane_te = te_df["vane"].values.astype(np.float32)

t_idx_tr = tr_sample["turb_idx"].values.astype(np.int64)
t_idx_te = te_df["turb_idx"].values.astype(np.int64)

train_dataset = TensorDataset(
    torch.tensor(X_tr), torch.tensor(y_tr), torch.tensor(u_iec_tr),
    torch.tensor(vane_tr), torch.tensor(t_idx_tr)
)
test_dataset = TensorDataset(
    torch.tensor(X_te), torch.tensor(y_te), torch.tensor(u_iec_te),
    torch.tensor(vane_te), torch.tensor(t_idx_te)
)

train_loader = DataLoader(train_dataset, batch_size=2048, shuffle=True, drop_last=True)
test_loader = DataLoader(test_dataset, batch_size=4096, shuffle=False)

# 4. Ultimate PINN Architecture Definition
class ResBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.fc1 = nn.Linear(dim, dim); self.ln1 = nn.LayerNorm(dim); self.act = nn.SiLU()
        self.fc2 = nn.Linear(dim, dim); self.ln2 = nn.LayerNorm(dim)
    def forward(self, x): return self.act(x + self.ln2(self.fc2(self.act(self.ln1(self.fc1(x))))))

class AerodynamicPriorWithYaw(nn.Module):
    def __init__(self, rated_power=2050.0):
        super().__init__()
        self.rated = rated_power
        self.u_mid = nn.Parameter(torch.tensor(8.5 if FARM == "penmanshiel" else 8.2))
        self.u_scale = nn.Parameter(torch.tensor(1.8))
        self.cut_in = nn.Parameter(torch.tensor(3.0))
        self.p_raw = nn.Parameter(torch.tensor(0.0))  # learnable p in [1.8, 2.8]

    def forward(self, u_iec, vane_deg):
        s = torch.sigmoid((u_iec - self.u_mid) / torch.clamp(self.u_scale, min=0.5))
        cut_in_gate = torch.sigmoid((u_iec - self.cut_in) * 3.0)
        p_curve = self.rated * s * cut_in_gate
        
        vane_rad = torch.deg2rad(vane_deg)
        p_yaw = 1.8 + 1.0 * torch.sigmoid(self.p_raw)
        yaw_factor = torch.clamp(torch.cos(vane_rad), min=0.5) ** p_yaw
        return p_curve * yaw_factor

class UltimateWindPINN(nn.Module):
    def __init__(self, num_inputs=14, num_turbines=14, emb_dim=8, hidden_dim=128, rated_power=2050.0):
        super().__init__()
        self.rated = rated_power
        self.prior = AerodynamicPriorWithYaw(rated_power)

        # Micro-siting entity embedding for turbine identity
        self.turb_embed = nn.Embedding(num_turbines, emb_dim)

        total_in = num_inputs + emb_dim  # 14 continuous + 8 embedding = 22 dims

        # Latent ResNet pathway
        self.in_proj = nn.Sequential(nn.Linear(total_in, hidden_dim), nn.LayerNorm(hidden_dim), nn.SiLU())
        self.res1 = ResBlock(hidden_dim)
        self.res2 = ResBlock(hidden_dim)
        self.out_proj = nn.Sequential(nn.Linear(hidden_dim, 64), nn.LayerNorm(64), nn.SiLU())

        # Dual heads for residual mean and log variance
        self.head_mu = nn.Linear(64, 1)
        self.head_log_var = nn.Linear(64, 1)

        nn.init.zeros_(self.head_mu.weight); nn.init.zeros_(self.head_mu.bias)
        initial_sigma = 40.0 if FARM == "penmanshiel" else 49.0
        nn.init.constant_(self.head_log_var.bias, np.log(initial_sigma**2))

    def forward(self, x_cont, turb_idx, u_iec, vane_deg):
        # Aerodynamic baseline
        p_base = self.prior(u_iec.unsqueeze(-1), vane_deg.unsqueeze(-1))

        # Fuse continuous atmospheric inputs with learned turbine micro-siting vector
        emb = self.turb_embed(turb_idx)
        x_fused = torch.cat([x_cont, emb], dim=-1)

        # Latent representation
        feat = self.out_proj(self.res2(self.res1(self.in_proj(x_fused))))

        delta_mu = self.head_mu(feat)
        log_var = self.head_log_var(feat)

        mu = torch.clamp(p_base + delta_mu, min=0.0, max=self.rated * 1.05)
        sigma = torch.clamp(torch.exp(0.5 * log_var), min=3.0)
        return p_base.squeeze(-1), mu.squeeze(-1), sigma.squeeze(-1)

# 5. Robust Huberized Gaussian NLL Loss Function
def robust_huber_nll(y, mu, sigma, k=2.5):
    """
    Gaussian NLL when error <= k * sigma; smooth linear loss when error > k * sigma.
    Protects against extreme turbulence gusts and pitch transient outliers.
    """
    z = (y - mu) / sigma
    abs_z = torch.abs(z)
    quad = 0.5 * (z ** 2)
    lin = k * abs_z - 0.5 * (k ** 2)
    huber_penalty = torch.where(abs_z <= k, quad, lin)
    return torch.mean(torch.log(sigma) + huber_penalty)

# 6. Training Pipeline
model = UltimateWindPINN(
    num_inputs=len(INPUT_COLS), num_turbines=NUM_TURBINES, emb_dim=8,
    hidden_dim=128, rated_power=rated
).to(device)

optimizer = optim.AdamW(model.parameters(), lr=1.5e-3, weight_decay=1e-4)
EPOCHS = 40
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)
LAMBDA_MONO = 10.0

print(f"\nStarting Ultimate PINN Training on {FARM.upper()} ({EPOCHS} Epochs)...")
t_all = time.time()

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss, total_huber, total_mono = 0.0, 0.0, 0.0
    t_start = time.time()

    for bx, by, bu_iec, bvane, bt_idx in train_loader:
        bx, by = bx.to(device), by.to(device)
        bu_iec, bvane, bt_idx = bu_iec.to(device), bvane.to(device), bt_idx.to(device)
        bx.requires_grad_(True)

        optimizer.zero_grad()
        p_base, mu, sigma = model(bx, bt_idx, bu_iec, bvane)

        # 1. Robust Huberized NLL Loss (k = 2.5)
        huber_loss = robust_huber_nll(by, mu, sigma, k=2.5)

        # 2. Adaptive Monotonicity Loss: Only below 92% rated power
        grad_ws = torch.autograd.grad(
            outputs=mu.sum(),
            inputs=bx,
            create_graph=True,
            retain_graph=True
        )[0][:, 0]

        reg2_mask = (p_base < 0.92 * rated)
        mono_violations = torch.relu(-grad_ws) * reg2_mask.float()
        mono_loss = LAMBDA_MONO * torch.mean(mono_violations ** 2)

        loss = huber_loss + mono_loss
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
        optimizer.step()

        total_loss += loss.item() * len(by)
        total_huber += huber_loss.item() * len(by)
        total_mono += mono_loss.item() * len(by)

    scheduler.step()
    n_samples = len(train_loader.dataset)
    current_lr = scheduler.get_last_lr()[0]
    print(f"Epoch {epoch:02d}/{EPOCHS} | Huber NLL: {total_huber/n_samples:.4f} | Mono Pen: {total_mono/n_samples:.5f} | lr: {current_lr:.6f} | {time.time()-t_start:.1f}s")

print(f"\nTraining completed in {time.time()-t_all:.1f}s.")

# 7. Full Evaluation on Held-Out Test Set (Years >= 2022)
print("\n" + "="*55)
print(f"EVALUATING ULTIMATE PINN ON {FARM.upper()} TEST SET (Years >= 2022)")
print("="*55)
model.eval()

preds_mu, preds_sigma = [], []
with torch.no_grad():
    for bx, by, bu_iec, bvane, bt_idx in test_loader:
        bx = bx.to(device)
        bu_iec, bvane, bt_idx = bu_iec.to(device), bvane.to(device), bt_idx.to(device)
        _, mu, sigma = model(bx, bt_idx, bu_iec, bvane)
        preds_mu.append(mu.cpu().numpy())
        preds_sigma.append(sigma.cpu().numpy())

preds_mu = np.concatenate(preds_mu)
preds_sigma = np.concatenate(preds_sigma)

# Metrics
e_te = y_te - preds_mu
rmse_ult = np.sqrt(mean_squared_error(y_te, preds_mu))
mae_ult = mean_absolute_error(y_te, preds_mu)
r2_ult = r2_score(y_te, preds_mu)

z = np.abs(e_te) / preds_sigma
nll_ult = np.mean(np.log(preds_sigma) + 0.5 * (e_te / preds_sigma) ** 2) + 0.5 * np.log(2 * np.pi)
c1 = (z < 1.0).mean()
c2 = (z < 2.0).mean()

# Reference values
prev_best = 46.79 if FARM == "penmanshiel" else 61.55
prev_hgb = 49.80 if FARM == "penmanshiel" else 60.10

print(f"\n--- Comparative Performance Report ({FARM.upper()}) ---")
report = pd.DataFrame({
    "Model": ["Baseline HGB (Trees)", "PINN Tier 1 (Previous Best)", "PINN Ultimate (Final Push)"],
    "Test RMSE (kW)": [prev_hgb, prev_best, round(rmse_ult, 2)],
    "Test MAE (kW)": [33.70 if FARM == "penmanshiel" else 41.10, 31.82 if FARM == "penmanshiel" else 42.28, round(mae_ult, 2)],
    "Test R²": [0.9944 if FARM == "penmanshiel" else 0.9899, 0.9950 if FARM == "penmanshiel" else 0.9894, round(r2_ult, 4)],
    "Test NLL": [5.0070 if FARM == "penmanshiel" else 5.2996, 4.9653 if FARM == "penmanshiel" else 5.2960, round(nll_ult, 4)],
    "1-sigma Cov": ["-", 0.6550 if FARM == "penmanshiel" else 0.6661, round(c1, 4)],
    "2-sigma Cov": ["-", 0.9279 if FARM == "penmanshiel" else 0.9356, round(c2, 4)]
})
print(report.to_string(index=False))

# 8. Save and Upload Checkpoint
os.makedirs(f"{W}/models", exist_ok=True)
ckpt_path = f"{W}/models/pinn_ultimate_{FARM}.pt"

torch.save({
    "model_state_dict": model.state_dict(),
    "scaler_mean": scaler_X.mean_,
    "scaler_scale": scaler_X.scale_,
    "input_cols": INPUT_COLS,
    "turb_to_idx": turb_to_idx,
    "metrics": {"rmse": rmse_ult, "nll": nll_ult, "r2": r2_ult}
}, ckpt_path)

print(f"\nSaved checkpoint: {ckpt_path}")
api.upload_file(
    path_or_fileobj=ckpt_path,
    path_in_repo=f"models/pinn_ultimate_{FARM}.pt",
    repo_id=REPO,
    repo_type="dataset",
    commit_message=f"Save Ultimate PINN {FARM} checkpoint (RMSE {rmse_ult:.2f} kW)"
)
print("Uploaded to Hugging Face successfully.")

hyperparameter optimization

In [ ]:
!pip -q install optuna

import os, time
import numpy as np, pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
import optuna

# 1. Environment & Setup
optuna.logging.set_verbosity(optuna.logging.WARNING)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device for HPO: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")

W = "/kaggle/working"
TS = "Date and time"
rated = 2050.0
FARM = "penmanshiel"

# 2. Physics Helper & Data Prep
def compute_u_iec(ws, temp, elev):
    p_z = 101325.0 * (1.0 - 0.0065 * elev / 288.15) ** 5.25588
    rho = p_z / (287.05 * (temp + 273.15))
    return ws * (rho / 1.225) ** (1.0 / 3.0)

df = pd.read_parquet(f"{W}/parquet/train/{FARM}_m1_clean.parquet")
df = df[df["ws"].between(3.0, 20.0)].reset_index(drop=True)

# Compute features
df["u_iec"] = compute_u_iec(df["ws"].values, df["temp"].values, df["elev"].values).astype(np.float32)
hour = df[TS].dt.hour.values + df[TS].dt.minute.values / 60.0
hour_rad = 2.0 * np.pi * hour / 24.0
df["hour_sin"] = np.sin(hour_rad).astype(np.float32)
df["hour_cos"] = np.cos(hour_rad).astype(np.float32)

unique_turbines = sorted(df["turbine"].unique())
turb_to_idx = {t: i for i, t in enumerate(unique_turbines)}
df["turb_idx"] = df["turbine"].map(turb_to_idx).astype(np.int64)
NUM_TURBINES = len(unique_turbines)

INPUT_COLS = [
    "ws", "sin_wd", "cos_wd", "vane", "abs_vane", "temp", "TI",
    "x", "y", "elev", "month_sin", "month_cos", "hour_sin", "hour_cos"
]
TARGET_COL = "power_kw"

# Subsample for fast HPO: 200k train, 50k validation (from training period <= 2021)
tr_pool = df[~df["is_test"]].copy()
tr_hpo = tr_pool.sample(200_000, random_state=42).reset_index(drop=True)
val_hpo = tr_pool.drop(tr_hpo.index).sample(50_000, random_state=99).reset_index(drop=True)

scaler = StandardScaler()
X_tr = scaler.fit_transform(tr_hpo[INPUT_COLS]).astype(np.float32)
X_val = scaler.transform(val_hpo[INPUT_COLS]).astype(np.float32)

y_tr = tr_hpo[TARGET_COL].values.astype(np.float32)
y_val = val_hpo[TARGET_COL].values.astype(np.float32)

train_loader = DataLoader(
    TensorDataset(
        torch.tensor(X_tr), torch.tensor(y_tr),
        torch.tensor(tr_hpo["u_iec"].values.astype(np.float32)),
        torch.tensor(tr_hpo["vane"].values.astype(np.float32)),
        torch.tensor(tr_hpo["turb_idx"].values.astype(np.int64))
    ),
    batch_size=2048, shuffle=True, drop_last=True
)

val_loader = DataLoader(
    TensorDataset(
        torch.tensor(X_val), torch.tensor(y_val),
        torch.tensor(val_hpo["u_iec"].values.astype(np.float32)),
        torch.tensor(val_hpo["vane"].values.astype(np.float32)),
        torch.tensor(val_hpo["turb_idx"].values.astype(np.int64))
    ),
    batch_size=4096, shuffle=False
)

# 3. Model Architecture
class ResBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.fc1 = nn.Linear(dim, dim); self.ln1 = nn.LayerNorm(dim); self.act = nn.SiLU()
        self.fc2 = nn.Linear(dim, dim); self.ln2 = nn.LayerNorm(dim)
    def forward(self, x): return self.act(x + self.ln2(self.fc2(self.act(self.ln1(self.fc1(x))))))

class AerodynamicPriorWithYaw(nn.Module):
    def __init__(self, rated_power=2050.0):
        super().__init__()
        self.rated = rated_power
        self.u_mid = nn.Parameter(torch.tensor(8.5))
        self.u_scale = nn.Parameter(torch.tensor(1.8))
        self.cut_in = nn.Parameter(torch.tensor(3.0))
        self.p_raw = nn.Parameter(torch.tensor(0.0))

    def forward(self, u_iec, vane_deg):
        s = torch.sigmoid((u_iec - self.u_mid) / torch.clamp(self.u_scale, min=0.5))
        cut_in_gate = torch.sigmoid((u_iec - self.cut_in) * 3.0)
        p_curve = self.rated * s * cut_in_gate
        vane_rad = torch.deg2rad(vane_deg)
        yaw_factor = torch.clamp(torch.cos(vane_rad), min=0.5) ** (1.8 + torch.sigmoid(self.p_raw))
        return p_curve * yaw_factor

class HPOWindPINN(nn.Module):
    def __init__(self, num_inputs=14, num_turbines=14, emb_dim=8, hidden_dim=128, rated_power=2050.0):
        super().__init__()
        self.rated = rated_power
        self.prior = AerodynamicPriorWithYaw(rated_power)
        self.turb_embed = nn.Embedding(num_turbines, emb_dim)
        
        self.in_proj = nn.Sequential(nn.Linear(num_inputs + emb_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.SiLU())
        self.res1 = ResBlock(hidden_dim)
        self.res2 = ResBlock(hidden_dim)
        self.out_proj = nn.Sequential(nn.Linear(hidden_dim, 64), nn.LayerNorm(64), nn.SiLU())
        self.head_mu = nn.Linear(64, 1)
        self.head_log_var = nn.Linear(64, 1)
        
        nn.init.zeros_(self.head_mu.weight); nn.init.zeros_(self.head_mu.bias)
        nn.init.constant_(self.head_log_var.bias, np.log(40.0**2))

    def forward(self, x_cont, turb_idx, u_iec, vane_deg):
        p_base = self.prior(u_iec.unsqueeze(-1), vane_deg.unsqueeze(-1))
        emb = self.turb_embed(turb_idx)
        feat = self.out_proj(self.res2(self.res1(self.in_proj(torch.cat([x_cont, emb], dim=-1)))))
        mu = torch.clamp(p_base + self.head_mu(feat), min=0.0, max=self.rated * 1.05)
        sigma = torch.clamp(torch.exp(0.5 * self.head_log_var(feat)), min=3.0)
        return p_base.squeeze(-1), mu.squeeze(-1), sigma.squeeze(-1)

def robust_huber_nll(y, mu, sigma, k=2.5):
    z = (y - mu) / sigma
    abs_z = torch.abs(z)
    huber = torch.where(abs_z <= k, 0.5 * (z ** 2), k * abs_z - 0.5 * (k ** 2))
    return torch.mean(torch.log(sigma) + huber)

# 4. Optuna Objective Function
def objective(trial):
    # Sample Hyperparameters
    lr = trial.suggest_float("lr", 5e-4, 3e-3, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-3, log=True)
    lambda_mono = trial.suggest_float("lambda_mono", 1.0, 25.0)
    huber_k = trial.suggest_float("huber_k", 1.8, 3.2)
    hidden_dim = trial.suggest_categorical("hidden_dim", [64, 128, 192])
    emb_dim = trial.suggest_categorical("emb_dim", [4, 8, 12])
    
    model = HPOWindPINN(
        num_inputs=len(INPUT_COLS), num_turbines=NUM_TURBINES,
        emb_dim=emb_dim, hidden_dim=hidden_dim, rated_power=rated
    ).to(device)
    
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=8, eta_min=1e-5)
    
    for epoch in range(1, 9):
        model.train()
        for bx, by, bu_iec, bvane, bt_idx in train_loader:
            bx, by = bx.to(device), by.to(device)
            bu_iec, bvane, bt_idx = bu_iec.to(device), bvane.to(device), bt_idx.to(device)
            bx.requires_grad_(True)
            
            optimizer.zero_grad()
            p_base, mu, sigma = model(bx, bt_idx, bu_iec, bvane)
            huber_loss = robust_huber_nll(by, mu, sigma, k=huber_k)
            
            grad_ws = torch.autograd.grad(outputs=mu.sum(), inputs=bx, create_graph=True, retain_graph=True)[0][:, 0]
            reg2_mask = (p_base < 0.92 * rated)
            mono_loss = lambda_mono * torch.mean((torch.relu(-grad_ws) * reg2_mask.float()) ** 2)
            
            (huber_loss + mono_loss).backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
            optimizer.step()
        scheduler.step()
        
    # Validation Evaluation
    model.eval()
    preds = []
    with torch.no_grad():
        for bx, by, bu_iec, bvane, bt_idx in val_loader:
            bx = bx.to(device); bu_iec, bvane, bt_idx = bu_iec.to(device), bvane.to(device), bt_idx.to(device)
            _, mu, _ = model(bx, bt_idx, bu_iec, bvane)
            preds.append(mu.cpu().numpy())
    
    val_rmse = np.sqrt(mean_squared_error(y_val, np.concatenate(preds)))
    return val_rmse

# 5. Run Fast 12-Trial Optimization Study
N_TRIALS = 12
print(f"Starting {N_TRIALS}-Trial Multi-Fidelity HPO")
t0 = time.time()

study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))

def print_trial_callback(study, trial):
    print(f"Trial {trial.number:02d}/{N_TRIALS-1} | Val RMSE: {trial.value:.2f} kW | "
          f"lr: {trial.params['lr']:.5f} | lambda: {trial.params['lambda_mono']:.1f} | "
          f"dim: {trial.params['hidden_dim']} | k: {trial.params['huber_k']:.2f}")

study.optimize(objective, n_trials=N_TRIALS, callbacks=[print_trial_callback])

print(f"\nHPO Search completed in {time.time()-t0:.1f}s.")
print("="*60)
print(f"BEST TRIAL: #{study.best_trial.number} with Validation RMSE = {study.best_value:.2f} kW")
print("="*60)
for k, v in study.best_params.items():
    if isinstance(v, float):
        print(f"  {k:15s}: {v:.6f}" if v < 0.01 else f"  {k:15s}: {v:.2f}")
    else:
        print(f"  {k:15s}: {v}")

FINAL VERSION PUSH

In [ ]:
import os, time
import numpy as np, pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from huggingface_hub import HfApi, login, hf_hub_download
from kaggle_secrets import UserSecretsClient

# 1. Environment & Auth
login(UserSecretsClient().get_secret("HF_TOKEN"))
api = HfApi()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")

W = "/kaggle/working"
REPO = "B4BUK4/MPVI_project"
TS = "Date and time"
rated = 2050.0
FARM = "penmanshiel"

# 2. Physics Helper: IEC 61400-12 Air Density Normalization
def compute_u_iec(ws, temp, elev):
    p_z = 101325.0 * (1.0 - 0.0065 * elev / 288.15) ** 5.25588
    rho = p_z / (287.05 * (temp + 273.15))
    return ws * (rho / 1.225) ** (1.0 / 3.0)

# 3. Load Dataset & Feature Engineering
df = pd.read_parquet(f"{W}/parquet/train/{FARM}_m1_clean.parquet")
df = df[df["ws"].between(3.0, 20.0)].reset_index(drop=True)

df["u_iec"] = compute_u_iec(df["ws"].values, df["temp"].values, df["elev"].values).astype(np.float32)

# Diurnal 24h Solar Harmonics
hour = df[TS].dt.hour.values + df[TS].dt.minute.values / 60.0
hour_rad = 2.0 * np.pi * hour / 24.0
df["hour_sin"] = np.sin(hour_rad).astype(np.float32)
df["hour_cos"] = np.cos(hour_rad).astype(np.float32)

# Integer Turbine IDs for Embeddings
unique_turbines = sorted(df["turbine"].unique())
turb_to_idx = {t: i for i, t in enumerate(unique_turbines)}
df["turb_idx"] = df["turbine"].map(turb_to_idx).astype(np.int64)
NUM_TURBINES = len(unique_turbines)

tr_df = df[~df["is_test"]].copy()
te_df = df[df["is_test"]].copy()

SAMPLE_SIZE = 1_200_000
tr_sample = tr_df.sample(min(SAMPLE_SIZE, len(tr_df)), random_state=42).reset_index(drop=True)

print(f"[{FARM.upper()} GOLDEN PINN] Train: {len(tr_sample):,} | Test: {len(te_df):,} | Turbines: {NUM_TURBINES}")

INPUT_COLS = [
    "ws", "sin_wd", "cos_wd", "vane", "abs_vane", "temp", "TI",
    "x", "y", "elev", "month_sin", "month_cos", "hour_sin", "hour_cos"
]
TARGET_COL = "power_kw"

# Normalization of continuous features
scaler_X = StandardScaler()
X_tr = scaler_X.fit_transform(tr_sample[INPUT_COLS]).astype(np.float32)
X_te = scaler_X.transform(te_df[INPUT_COLS]).astype(np.float32)

y_tr = tr_sample[TARGET_COL].values.astype(np.float32)
y_te = te_df[TARGET_COL].values.astype(np.float32)

u_iec_tr = tr_sample["u_iec"].values.astype(np.float32)
u_iec_te = te_df["u_iec"].values.astype(np.float32)

vane_tr = tr_sample["vane"].values.astype(np.float32)
vane_te = te_df["vane"].values.astype(np.float32)

t_idx_tr = tr_sample["turb_idx"].values.astype(np.int64)
t_idx_te = te_df["turb_idx"].values.astype(np.int64)

train_dataset = TensorDataset(
    torch.tensor(X_tr), torch.tensor(y_tr), torch.tensor(u_iec_tr),
    torch.tensor(vane_tr), torch.tensor(t_idx_tr)
)
test_dataset = TensorDataset(
    torch.tensor(X_te), torch.tensor(y_te), torch.tensor(u_iec_te),
    torch.tensor(vane_te), torch.tensor(t_idx_te)
)

train_loader = DataLoader(train_dataset, batch_size=2048, shuffle=True, drop_last=True)
test_loader = DataLoader(test_dataset, batch_size=4096, shuffle=False)

# 4. Golden Architecture (Optuna Trial #5 Specs: hidden_dim=128, emb_dim=12)
class ResBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.fc1 = nn.Linear(dim, dim); self.ln1 = nn.LayerNorm(dim); self.act = nn.SiLU()
        self.fc2 = nn.Linear(dim, dim); self.ln2 = nn.LayerNorm(dim)
    def forward(self, x): return self.act(x + self.ln2(self.fc2(self.act(self.ln1(self.fc1(x))))))

class AerodynamicPriorWithYaw(nn.Module):
    def __init__(self, rated_power=2050.0):
        super().__init__()
        self.rated = rated_power
        self.u_mid = nn.Parameter(torch.tensor(8.5))
        self.u_scale = nn.Parameter(torch.tensor(1.8))
        self.cut_in = nn.Parameter(torch.tensor(3.0))
        self.p_raw = nn.Parameter(torch.tensor(0.0))

    def forward(self, u_iec, vane_deg):
        s = torch.sigmoid((u_iec - self.u_mid) / torch.clamp(self.u_scale, min=0.5))
        cut_in_gate = torch.sigmoid((u_iec - self.cut_in) * 3.0)
        p_curve = self.rated * s * cut_in_gate
        vane_rad = torch.deg2rad(vane_deg)
        yaw_factor = torch.clamp(torch.cos(vane_rad), min=0.5) ** (1.8 + torch.sigmoid(self.p_raw))
        return p_curve * yaw_factor

class GoldenWindPINN(nn.Module):
    def __init__(self, num_inputs=14, num_turbines=14, emb_dim=12, hidden_dim=128, rated_power=2050.0):
        super().__init__()
        self.rated = rated_power
        self.prior = AerodynamicPriorWithYaw(rated_power)

        # Optuna optimal 12-dimensional turbine micro-siting embedding
        self.turb_embed = nn.Embedding(num_turbines, emb_dim)
        total_in = num_inputs + emb_dim  # 14 continuous + 12 embedding = 26 dims

        self.in_proj = nn.Sequential(nn.Linear(total_in, hidden_dim), nn.LayerNorm(hidden_dim), nn.SiLU())
        self.res1 = ResBlock(hidden_dim)
        self.res2 = ResBlock(hidden_dim)
        self.out_proj = nn.Sequential(nn.Linear(hidden_dim, 64), nn.LayerNorm(64), nn.SiLU())

        self.head_mu = nn.Linear(64, 1)
        self.head_log_var = nn.Linear(64, 1)

        nn.init.zeros_(self.head_mu.weight); nn.init.zeros_(self.head_mu.bias)
        nn.init.constant_(self.head_log_var.bias, np.log(40.0**2))

    def forward(self, x_cont, turb_idx, u_iec, vane_deg):
        p_base = self.prior(u_iec.unsqueeze(-1), vane_deg.unsqueeze(-1))
        emb = self.turb_embed(turb_idx)
        feat = self.out_proj(self.res2(self.res1(self.in_proj(torch.cat([x_cont, emb], dim=-1)))))

        delta_mu = self.head_mu(feat)
        log_var = self.head_log_var(feat)

        mu = torch.clamp(p_base + delta_mu, min=0.0, max=self.rated * 1.05)
        sigma = torch.clamp(torch.exp(0.5 * log_var), min=3.0)
        return p_base.squeeze(-1), mu.squeeze(-1), sigma.squeeze(-1)

# Robust Huber Loss Function
def robust_huber_nll(y, mu, sigma, k=3.05):
    z = (y - mu) / sigma
    abs_z = torch.abs(z)
    huber = torch.where(abs_z <= k, 0.5 * (z ** 2), k * abs_z - 0.5 * (k ** 2))
    return torch.mean(torch.log(sigma) + huber)

# 5. Training Setup (Optuna Best Hyperparameters)
BEST_LR = 0.002841
BEST_WD = 0.000355
BEST_LAMBDA_MONO = 23.55
BEST_HUBER_K = 3.05
BEST_EMB_DIM = 12
BEST_HIDDEN_DIM = 128
EPOCHS = 50

model = GoldenWindPINN(
    num_inputs=len(INPUT_COLS), num_turbines=NUM_TURBINES,
    emb_dim=BEST_EMB_DIM, hidden_dim=BEST_HIDDEN_DIM, rated_power=rated
).to(device)

optimizer = optim.AdamW(model.parameters(), lr=BEST_LR, weight_decay=BEST_WD)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)

print(f"\nStarting Golden PINN Training ({EPOCHS} Epochs on 1,200,000 samples)...")
print(f"Hyperparameters: lr={BEST_LR:.6f} | wd={BEST_WD:.6f} | lambda={BEST_LAMBDA_MONO:.2f} | k={BEST_HUBER_K:.2f} | emb_dim={BEST_EMB_DIM}")
t_all = time.time()

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss, total_huber, total_mono = 0.0, 0.0, 0.0
    t_start = time.time()

    for bx, by, bu_iec, bvane, bt_idx in train_loader:
        bx, by = bx.to(device), by.to(device)
        bu_iec, bvane, bt_idx = bu_iec.to(device), bvane.to(device), bt_idx.to(device)
        bx.requires_grad_(True)

        optimizer.zero_grad()
        p_base, mu, sigma = model(bx, bt_idx, bu_iec, bvane)

        # Robust Huber NLL Loss with Optuna's k=3.05
        huber_loss = robust_huber_nll(by, mu, sigma, k=BEST_HUBER_K)

        # Adaptive Monotonicity Loss with Optuna's lambda=23.55
        grad_ws = torch.autograd.grad(outputs=mu.sum(), inputs=bx, create_graph=True, retain_graph=True)[0][:, 0]
        reg2_mask = (p_base < 0.92 * rated)
        mono_violations = torch.relu(-grad_ws) * reg2_mask.float()
        mono_loss = BEST_LAMBDA_MONO * torch.mean(mono_violations ** 2)

        loss = huber_loss + mono_loss
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
        optimizer.step()

        total_loss += loss.item() * len(by)
        total_huber += huber_loss.item() * len(by)
        total_mono += mono_loss.item() * len(by)

    scheduler.step()
    n_samples = len(train_loader.dataset)
    current_lr = scheduler.get_last_lr()[0]
    print(f"Epoch {epoch:02d}/{EPOCHS} | Huber NLL: {total_huber/n_samples:.4f} | Mono Pen: {total_mono/n_samples:.5f} | lr: {current_lr:.6f} | {time.time()-t_start:.1f}s")

print(f"\nTraining completed in {time.time()-t_all:.1f}s.")

# 6. Final Evaluation on Held-Out Test Set (Years >= 2022)
print("\n" + "="*55)
print(f"EVALUATING GOLDEN PINN ON {FARM.upper()} TEST SET (Years >= 2022)")
print("="*55)
model.eval()

preds_mu, preds_sigma = [], []
with torch.no_grad():
    for bx, by, bu_iec, bvane, bt_idx in test_loader:
        bx = bx.to(device); bu_iec, bvane, bt_idx = bu_iec.to(device), bvane.to(device), bt_idx.to(device)
        _, mu, sigma = model(bx, bt_idx, bu_iec, bvane)
        preds_mu.append(mu.cpu().numpy())
        preds_sigma.append(sigma.cpu().numpy())

preds_mu = np.concatenate(preds_mu)
preds_sigma = np.concatenate(preds_sigma)

# Metrics Calculation
e_te = y_te - preds_mu
rmse_gold = np.sqrt(mean_squared_error(y_te, preds_mu))
mae_gold = mean_absolute_error(y_te, preds_mu)
r2_gold = r2_score(y_te, preds_mu)

z = np.abs(e_te) / preds_sigma
nll_gold = np.mean(np.log(preds_sigma) + 0.5 * (e_te / preds_sigma) ** 2) + 0.5 * np.log(2 * np.pi)
c1 = (z < 1.0).mean()
c2 = (z < 2.0).mean()

# Performance Report
print(f"\n--- Comparative Performance Report ({FARM.upper()}) ---")
report = pd.DataFrame({
    "Model": [
        "Baseline HGB (Trees)",
        "PINN Tier 1 (25 ep)",
        "PINN Ultimate (Pre-HPO, 40 ep)",
        "PINN Golden (Post-HPO, 30 ep)"
    ],
    "Test RMSE (kW)": [49.80, 46.79, 46.12, round(rmse_gold, 2)],
    "Test MAE (kW)": [33.70, 31.82, 31.30, round(mae_gold, 2)],
    "Test R²": [0.9944, 0.9950, 0.9952, round(r2_gold, 4)],
    "Test NLL": [5.0070, 4.9653, 4.9617, round(nll_gold, 4)],
    "1-sigma Cov": ["-", 0.6550, 0.6342, round(c1, 4)],
    "2-sigma Cov": ["-", 0.9279, 0.9170, round(c2, 4)]
})
print(report.to_string(index=False))

# 7. Save and Upload Golden Checkpoint
os.makedirs(f"{W}/models", exist_ok=True)
ckpt_path = f"{W}/models/pinn_golden_{FARM}.pt"

torch.save({
    "model_state_dict": model.state_dict(),
    "scaler_mean": scaler_X.mean_,
    "scaler_scale": scaler_X.scale_,
    "input_cols": INPUT_COLS,
    "turb_to_idx": turb_to_idx,
    "best_params": {
        "lr": BEST_LR, "weight_decay": BEST_WD, "lambda_mono": BEST_LAMBDA_MONO,
        "huber_k": BEST_HUBER_K, "emb_dim": BEST_EMB_DIM, "hidden_dim": BEST_HIDDEN_DIM
    },
    "metrics": {"rmse": rmse_gold, "nll": nll_gold, "r2": r2_gold}
}, ckpt_path)

print(f"\nSaved checkpoint: {ckpt_path} ({os.path.getsize(ckpt_path)/1e3:.1f} KB)")
api.upload_file(
    path_or_fileobj=ckpt_path,
    path_in_repo=f"models/pinn_golden_{FARM}.pt",
    repo_id=REPO,
    repo_type="dataset",
    commit_message=f"Save Golden PINN {FARM} checkpoint (RMSE {rmse_gold:.2f} kW)"
)
print("Uploaded Golden Model checkpoint to Hugging Face successfully.")

PINN Physical Cp

In [ ]:
import os, time
import numpy as np, pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from huggingface_hub import HfApi, login
from kaggle_secrets import UserSecretsClient

# 1. Environment & Auth
login(UserSecretsClient().get_secret("HF_TOKEN"))
api = HfApi()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")

W = "/kaggle/working"
REPO = "B4BUK4/MPVI_project"
TS = "Date and time"
rated = 2050.0

# Choose farm: 'penmanshiel' (D=82m, best 46.12 kW) or 'kelmarsh' (D=92m, best 61.55 kW)
FARM = "penmanshiel"
ROTOR_D = 82.0 if FARM == "penmanshiel" else 92.0
ROTOR_AREA = np.pi * (ROTOR_D / 2.0) ** 2  # 5,281 m² (MM82) or 6,648 m² (MM92)

# 2. Physics Helper: Exact Air Density
def compute_air_density(elev, temp):
    # Barometric pressure at elevation z (Pa)
    p_z = 101325.0 * (1.0 - 0.0065 * elev / 288.15) ** 5.25588
    # Air density rho (kg/m³)
    return (p_z / (287.05 * (temp + 273.15))).astype(np.float32)

# 3. Load Dataset & Precompute Physics
df = pd.read_parquet(f"{W}/parquet/train/{FARM}_m1_clean.parquet")
df = df[df["ws"].between(3.0, 20.0)].reset_index(drop=True)

# Kinetic Wind Power in the tube (kW): P_wind = 0.5 * rho * A * u³ / 1000
df["rho"] = compute_air_density(df["elev"].values, df["temp"].values)
df["p_wind_kw"] = (0.5 * df["rho"] * ROTOR_AREA * (df["ws"] ** 3) / 1000.0).astype(np.float32)

# Diurnal Harmonics
hour = df[TS].dt.hour.values + df[TS].dt.minute.values / 60.0
hour_rad = 2.0 * np.pi * hour / 24.0
df["hour_sin"] = np.sin(hour_rad).astype(np.float32)
df["hour_cos"] = np.cos(hour_rad).astype(np.float32)

# Turbine Indices
unique_turbines = sorted(df["turbine"].unique())
turb_to_idx = {t: i for i, t in enumerate(unique_turbines)}
df["turb_idx"] = df["turbine"].map(turb_to_idx).astype(np.int64)
NUM_TURBINES = len(unique_turbines)

tr_df = df[~df["is_test"]].copy()
te_df = df[df["is_test"]].copy()

SAMPLE_SIZE = 1_200_000 if FARM == "penmanshiel" else 800_000
tr_sample = tr_df.sample(min(SAMPLE_SIZE, len(tr_df)), random_state=42).reset_index(drop=True)

print(f"[{FARM.upper()} PHYSICAL Cp PINN] Train: {len(tr_sample):,} | Test: {len(te_df):,} | Rotor Area: {ROTOR_AREA:.0f} m²")

INPUT_COLS = [
    "ws", "sin_wd", "cos_wd", "vane", "abs_vane", "temp", "TI",
    "x", "y", "elev", "month_sin", "month_cos", "hour_sin", "hour_cos"
]
TARGET_COL = "power_kw"

scaler_X = StandardScaler()
X_tr = scaler_X.fit_transform(tr_sample[INPUT_COLS]).astype(np.float32)
X_te = scaler_X.transform(te_df[INPUT_COLS]).astype(np.float32)

y_tr = tr_sample[TARGET_COL].values.astype(np.float32)
y_te = te_df[TARGET_COL].values.astype(np.float32)

p_wind_tr = tr_sample["p_wind_kw"].values.astype(np.float32)
p_wind_te = te_df["p_wind_kw"].values.astype(np.float32)

ws_tr = tr_sample["ws"].values.astype(np.float32)
ws_te = te_df["ws"].values.astype(np.float32)

vane_tr = tr_sample["vane"].values.astype(np.float32)
vane_te = te_df["vane"].values.astype(np.float32)

t_idx_tr = tr_sample["turb_idx"].values.astype(np.int64)
t_idx_te = te_df["turb_idx"].values.astype(np.int64)

train_dataset = TensorDataset(
    torch.tensor(X_tr), torch.tensor(y_tr), torch.tensor(p_wind_tr),
    torch.tensor(ws_tr), torch.tensor(vane_tr), torch.tensor(t_idx_tr)
)
test_dataset = TensorDataset(
    torch.tensor(X_te), torch.tensor(y_te), torch.tensor(p_wind_te),
    torch.tensor(ws_te), torch.tensor(vane_te), torch.tensor(t_idx_te)
)

train_loader = DataLoader(train_dataset, batch_size=2048, shuffle=True, drop_last=True)
test_loader = DataLoader(test_dataset, batch_size=4096, shuffle=False)

# 4. Physical Cp PINN Architecture
class ResBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.fc1 = nn.Linear(dim, dim); self.ln1 = nn.LayerNorm(dim); self.act = nn.SiLU()
        self.fc2 = nn.Linear(dim, dim); self.ln2 = nn.LayerNorm(dim)
    def forward(self, x): return self.act(x + self.ln2(self.fc2(self.act(self.ln1(self.fc1(x))))))

class PhysicalCpPINN(nn.Module):
    def __init__(self, num_inputs=14, num_turbines=14, emb_dim=12, hidden_dim=128, rated_power=2050.0):
        super().__init__()
        self.rated = rated_power
        self.cp_max = 0.485  # Betz limit ceiling (physically cannot exceed 0.593)
        self.smooth_beta = 4.5  # Algebraic smooth-min transition parameter

        # Baseline analytical Cp curve (peaks at ~8 m/s, drops at cut-in and high wind)
        self.u_peak = nn.Parameter(torch.tensor(8.2))
        self.u_width = nn.Parameter(torch.tensor(3.5))
        self.cut_in = nn.Parameter(torch.tensor(3.0))
        self.p_raw = nn.Parameter(torch.tensor(0.0))  # learnable yaw exponent

        # Latent network predicting aerodynamic efficiency modulation Δlogit(Cp)
        self.turb_embed = nn.Embedding(num_turbines, emb_dim)
        total_in = num_inputs + emb_dim

        self.in_proj = nn.Sequential(nn.Linear(total_in, hidden_dim), nn.LayerNorm(hidden_dim), nn.SiLU())
        self.res1 = ResBlock(hidden_dim)
        self.res2 = ResBlock(hidden_dim)
        self.out_proj = nn.Sequential(nn.Linear(hidden_dim, 64), nn.LayerNorm(64), nn.SiLU())

        # Heads for Cp modulation and heteroscedastic uncertainty
        self.head_cp = nn.Linear(64, 1)
        self.head_log_var = nn.Linear(64, 1)

        nn.init.zeros_(self.head_cp.weight); nn.init.zeros_(self.head_cp.bias)
        initial_sigma = 40.0 if FARM == "penmanshiel" else 49.0
        nn.init.constant_(self.head_log_var.bias, np.log(initial_sigma**2))

    def forward(self, x_cont, turb_idx, p_wind, ws, vane_deg):
        # 1. Base physical Cp curve (Gaussian bell modulated by cut-in gate)
        cp_base_logit = -0.5 * ((ws.unsqueeze(-1) - self.u_peak) / torch.clamp(self.u_width, min=1.0)) ** 2
        cut_in_gate = torch.sigmoid((ws.unsqueeze(-1) - self.cut_in) * 3.0)

        # 2. Neural residual modulation on Cp
        emb = self.turb_embed(turb_idx)
        feat = self.out_proj(self.res2(self.res1(self.in_proj(torch.cat([x_cont, emb], dim=-1)))))
        delta_cp = self.head_cp(feat)
        log_var = self.head_log_var(feat)

        # 3. Final Aerodynamic Efficiency Cp (Strictly bounded in [0, 0.485])
        cp_eff = self.cp_max * torch.sigmoid(cp_base_logit + delta_cp) * cut_in_gate

        # 4. Momentum-Theory Yaw Loss Factor cos^p(gamma)
        vane_rad = torch.deg2rad(vane_deg.unsqueeze(-1))
        yaw_factor = torch.clamp(torch.cos(vane_rad), min=0.5) ** (1.8 + torch.sigmoid(self.p_raw))
        cp_final = cp_eff * yaw_factor

        # 5. Aerodynamic Power = P_wind * Cp
        p_aero = torch.clamp(p_wind.unsqueeze(-1) * cp_final, min=0.0)

        # 6. Differentiable Smooth Transition to Rated Generator Limit
        # P_pred = P_aero / (1 + (P_aero / P_rated)^beta)^(1/beta)
        ratio = p_aero / self.rated
        p_pred = p_aero / ((1.0 + ratio ** self.smooth_beta) ** (1.0 / self.smooth_beta))

        sigma = torch.clamp(torch.exp(0.5 * log_var), min=3.0)
        return cp_final.squeeze(-1), p_pred.squeeze(-1), sigma.squeeze(-1)

# Robust Huber Loss Function (k = 3.05 from Optuna)
def robust_huber_nll(y, mu, sigma, k=3.05):
    z = (y - mu) / sigma
    abs_z = torch.abs(z)
    huber = torch.where(abs_z <= k, 0.5 * (z ** 2), k * abs_z - 0.5 * (k ** 2))
    return torch.mean(torch.log(sigma) + huber)

# 5. Training Pipeline (Optuna Trial #5 Hyperparameters)
BEST_LR = 0.002841
BEST_WD = 0.000355
BEST_LAMBDA_MONO = 15.0  # Reduced: the kinetic u³ tube naturally enforces positive slope!
BEST_HUBER_K = 3.05
EPOCHS = 50

model = PhysicalCpPINN(
    num_inputs=len(INPUT_COLS), num_turbines=NUM_TURBINES, emb_dim=12,
    hidden_dim=128, rated_power=rated
).to(device)

optimizer = optim.AdamW(model.parameters(), lr=BEST_LR, weight_decay=BEST_WD)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)

print(f"\nStarting Physical Cp PINN Training on {FARM.upper()} ({EPOCHS} Epochs)...")
t_all = time.time()

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss, total_huber, total_mono = 0.0, 0.0, 0.0
    t_start = time.time()

    for bx, by, bp_wind, bws, bvane, bt_idx in train_loader:
        bx, by = bx.to(device), by.to(device)
        bp_wind, bws, bvane, bt_idx = bp_wind.to(device), bws.to(device), bvane.to(device), bt_idx.to(device)
        bx.requires_grad_(True)

        optimizer.zero_grad()
        cp, mu, sigma = model(bx, bt_idx, bp_wind, bws, bvane)

        # 1. Huber NLL Loss
        huber_loss = robust_huber_nll(by, mu, sigma, k=BEST_HUBER_K)

        # 2. Monotonicity constraint (Only active below 90% rated power)
        grad_ws = torch.autograd.grad(outputs=mu.sum(), inputs=bx, create_graph=True, retain_graph=True)[0][:, 0]
        reg2_mask = (mu < 0.90 * rated)
        mono_violations = torch.relu(-grad_ws) * reg2_mask.float()
        mono_loss = BEST_LAMBDA_MONO * torch.mean(mono_violations ** 2)

        loss = huber_loss + mono_loss
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
        optimizer.step()

        total_loss += loss.item() * len(by)
        total_huber += huber_loss.item() * len(by)
        total_mono += mono_loss.item() * len(by)

    scheduler.step()
    n_samples = len(train_loader.dataset)
    current_lr = scheduler.get_last_lr()[0]
    print(f"Epoch {epoch:02d}/{EPOCHS} | Huber NLL: {total_huber/n_samples:.4f} | Mono Pen: {total_mono/n_samples:.5f} | lr: {current_lr:.6f} | {time.time()-t_start:.1f}s")

print(f"\nTraining completed in {time.time()-t_all:.1f}s.")

# 6. Evaluation on Held-Out Test Set (Years >= 2022)
print("\n" + "="*55)
print(f"EVALUATING PHYSICAL Cp PINN ON {FARM.upper()} TEST SET (Years >= 2022)")
print("="*55)
model.eval()

# Fixed unpacking typo (3 lists on right)
preds_mu, preds_sigma, preds_cp = [], [], []

with torch.no_grad():
    for bx, by, bp_wind, bws, bvane, bt_idx in test_loader:
        bx = bx.to(device)
        bp_wind, bws, bvane, bt_idx = bp_wind.to(device), bws.to(device), bvane.to(device), bt_idx.to(device)
        cp, mu, sigma = model(bx, bt_idx, bp_wind, bws, bvane)
        preds_mu.append(mu.cpu().numpy())
        preds_sigma.append(sigma.cpu().numpy())
        preds_cp.append(cp.cpu().numpy())

preds_mu = np.concatenate(preds_mu)
preds_sigma = np.concatenate(preds_sigma)
preds_cp = np.concatenate(preds_cp)

# Metrics Calculation
e_te = y_te - preds_mu
rmse_cp = np.sqrt(mean_squared_error(y_te, preds_mu))
mae_cp = mean_absolute_error(y_te, preds_mu)
r2_cp = r2_score(y_te, preds_mu)

z = np.abs(e_te) / preds_sigma
nll_cp = np.mean(np.log(preds_sigma) + 0.5 * (e_te / preds_sigma) ** 2) + 0.5 * np.log(2 * np.pi)
c1 = (z < 1.0).mean()
c2 = (z < 2.0).mean()

# Performance Comparison Table
print(f"\n--- Comparative Performance Report ({FARM.upper()}) ---")
report = pd.DataFrame({
    "Model Architecture": [
        "Baseline HGB (Trees)",
        "PINN Tier 1 (ResNet Baseline)",
        "PINN Ultimate (Pre-HPO, 40 ep)",
        "PINN Golden (Post-HPO, 50 ep)",
        "PINN Physical Cp (First-Principles)"
    ],
    "Test RMSE (kW)": [49.80, 46.79, 46.12, 46.11, round(rmse_cp, 2)],
    "Test MAE (kW)": [33.70, 31.82, 31.30, 31.31, round(mae_cp, 2)],
    "Test R²": [0.9944, 0.9950, 0.9952, 0.9952, round(r2_cp, 4)],
    "Test NLL": [5.0070, 4.9653, 4.9617, 4.9574, round(nll_cp, 4)],
    "1-sigma Cov": ["-", 0.6550, 0.6342, 0.6421, round(c1, 4)],
    "2-sigma Cov": ["-", 0.9279, 0.9170, 0.9212, round(c2, 4)]
})
print(report.to_string(index=False))

print(f"\nLearned Rotor Efficiency: Mean Cp = {preds_cp.mean():.3f} | Max Cp = {preds_cp.max():.3f} (Betz Limit = 0.593)")

# Save Checkpoint
os.makedirs(f"{W}/models", exist_ok=True)
ckpt_path = f"{W}/models/pinn_physical_cp_{FARM}.pt"

torch.save({
    "model_state_dict": model.state_dict(),
    "scaler_mean": scaler_X.mean_,
    "scaler_scale": scaler_X.scale_,
    "input_cols": INPUT_COLS,
    "turb_to_idx": turb_to_idx,
    "metrics": {"rmse": rmse_cp, "nll": nll_cp, "r2": r2_cp}
}, ckpt_path)

print(f"\nSaved checkpoint: {ckpt_path}")
api.upload_file(
    path_or_fileobj=ckpt_path,
    path_in_repo=f"models/pinn_physical_cp_{FARM}.pt",
    repo_id=REPO,
    repo_type="dataset",
    commit_message=f"Save Physical Cp PINN {FARM} checkpoint (RMSE {rmse_cp:.2f} kW)"
)
print("Uploaded to Hugging Face successfully.")